# 21 · 검출기 + 정상 음성 = 1단계 후보 (STEP 52, 캐글 T4)

**Add Input**: `dogskin-detect-full-0/1/2` · `dogskin-detect-normals-0/1/2` · `dogskin-detect-full-labels` (+ 재개 시 이전 출력).
**Accelerator: GPU T4 ×2 도 되지만 하나만 씁니다** · Internet ON → **Save & Run All (Commit)**.
사전등록: [`STEP52`](../docs/results/STEP52_검출기_1단계후보_정상음성_사전등록.md) — 문턱은 여기서 안 바꿉니다.

- 병변 창 149,800 + **정상 창 ~150k(정답 네모 0개)** · fold 0 = 검증
- D-FINE-M · 512 · fp16 · 증강 `dfine` · EMA 워밍업 · `HOURS` 예산 안에서 epoch 경계 정지 → `/kaggle/working/dfine_stage1_resume.zip`
- 매 epoch: 검증 표본(병변 3,000 + 정상 3,000)에 **최고 쿼리 점수 = 1단계 점수** → AUROC · recall@헛알림34% · 헛알림@recall73% · 병변 밴드
- 재개: 출력 ZIP 을 Dataset 으로 붙이면 자동 (풀린 폴더도 찾음). **판정은 로컬** `tools/detect_coverage.py --detector-stage1`

⚠️ holdout 은 안 엽니다.

In [ ]:
from pathlib import Path
import json, os, sys, time, shutil, subprocess, zipfile
HOURS = 11.5        # 캐글 세션 상한 12h. 재개할 때만 줄이기
EPOCHS = 3          # 재개할 때 바꾸지 않기
BATCH_SIZE = 16
LR = 1e-4
IMG = 512
WORKERS = 4
CODE = Path('/kaggle/working/dfine_code')
CKPT = Path('/kaggle/working/ckpt'); CKPT.mkdir(parents=True, exist_ok=True)
MODEL_ID = 'ustc-community/dfine-medium-obj2coco'
AUG = 'dfine'
FILES = {"src/__init__.py": "\"\"\"반려견 피부질환 스크리닝 보조 모델 — 소스 패키지.\n\n⚠️ 이 프로젝트의 산출물은 수의학적 진단이 아닙니다.\n   보호자에게 \"의심 소견이 있으니 병원에 가보세요\" 수준의 안내만 제공합니다.\n   docs/cautions/03_의료AI_안전설계_원칙.md 를 반드시 읽어주세요.\n\"\"\"\n\n__version__ = \"0.1.0\"\n", "src/config.py": "\"\"\"프로젝트 전역 설정 — 모든 \"숫자\"는 여기 한 곳에 모읍니다.\n\n노트북에서 하이퍼파라미터를 직접 고치지 마세요. 여기서 고치고 노트북은 읽기만 하면\n어떤 설정으로 어떤 결과가 나왔는지 나중에 추적할 수 있습니다.\n\n    from src.config import CFG, CLASSES\n    cfg = CFG()                          # 기본값\n    cfg = CFG(img_size=384, epochs=20)   # 일부만 바꾸기\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field, asdict\nfrom pathlib import Path\nfrom typing import Any\n\n# ──────────────────────────────────────────────────────────────\n# 클래스 정의\n#\n# ⚠️ 매우 중요: 이 라벨들은 \"병명\"이 아니라 \"병변의 형태(morphology)\"입니다.\n#   A2 가 나왔다고 \"이 강아지는 지루성 피부염\" 이 아니라\n#   \"비듬·각질 형태의 병변이 보인다\" 까지가 모델이 말할 수 있는 전부입니다.\n#   같은 병변 형태가 여러 질환에서 나오고, 같은 질환이 여러 형태로 나타납니다.\n#   → docs/data/병변_6종_임상_해설.md 참고\n# ──────────────────────────────────────────────────────────────\nCLASSES: list[str] = [\"A1\", \"A2\", \"A3\", \"A4\", \"A5\", \"A6\"]\n\nCLASS_KO: dict[str, str] = {\n    \"A1\": \"구진·플라크\",\n    \"A2\": \"비듬·각질·상피성잔고리\",\n    \"A3\": \"태선화·과다색소침착\",\n    \"A4\": \"농포·여드름\",\n    \"A5\": \"미란·궤양\",\n    \"A6\": \"결절·종괴\",\n}\n\nCLASS_EN: dict[str, str] = {\n    \"A1\": \"Papule / Plaque\",\n    \"A2\": \"Scale / Crust / Epidermal collarette\",\n    \"A3\": \"Lichenification / Hyperpigmentation\",\n    \"A4\": \"Pustule / Acne\",\n    \"A5\": \"Erosion / Ulcer\",\n    \"A6\": \"Nodule / Mass\",\n}\n\n# 1단계(정상/이상) 이진 분류용 라벨.\n# ✅ 실물 확인 결과 무증상 데이터가 존재합니다 — metaData.lesions == \"A7\".\n#    유증상 26,191 / 무증상 28,042 로 거의 반반이라 2단계 모델이 가능합니다.\n#    ⚠️ 무증상 이미지가 'A1_구진_플라크' 같은 폴더 안에 들어 있습니다.\n#       폴더명이 아니라 metaData.lesions 를 봐야 합니다.\nNORMAL_LABEL = \"A7\"\nCLASS_KO[NORMAL_LABEL] = \"무증상(정상)\"\nCLASS_EN[NORMAL_LABEL] = \"Normal / Asymptomatic\"\n\n# 1단계에서 쓰는 클래스 (정상 vs 이상)\nCLASSES_STAGE1 = [NORMAL_LABEL, \"ABNORMAL\"]\n# 2단계에서 쓰는 클래스 (병변 6종) = CLASSES\n\n# 병변별 임상적 긴급도 힌트 — \"의심된다\"의 톤을 조절할 때 씁니다.\n# 진단이 아니라 안내 문구의 강도를 정하는 용도일 뿐입니다.\nURGENCY_HINT: dict[str, str] = {\n    \"A1\": \"관찰\",\n    \"A2\": \"관찰\",\n    \"A3\": \"만성 경과 가능 — 진료 권장\",\n    \"A4\": \"감염 동반 가능 — 진료 권장\",\n    \"A5\": \"피부 장벽 손상 — 조기 진료 권장\",\n    \"A6\": \"종양 감별 필요 — 조기 진료 권장\",\n}\n\n#: 긴급도 등급 (낮을수록 급하지 않음). `URGENCY_HINT` 의 문구를 순서로 옮긴 것.\n#: ⚠️ 이 순서는 `docs/data/병변_6종_임상_해설.md` 요약표의 \"긴급도\" 열이고,\n#:    **이 프로젝트의 어떤 실험보다 먼저** 적혀 있었습니다. 혼동행렬을 보고\n#:    만든 묶음이 아닙니다 — 데이터를 보고 묶으면 뭐든 좋아 보입니다.\nURGENCY_TIER: dict[str, int] = {\n    \"A1\": 0, \"A2\": 0,          # 관찰\n    \"A3\": 1, \"A4\": 1,          # 진료 권장\n    \"A5\": 2, \"A6\": 2,          # 조기 진료 권장\n}\nURGENCY_TIER_NAME: dict[int, str] = {\n    0: \"관찰\", 1: \"진료 권장\", 2: \"조기 진료 권장\",\n}\n\n#: 형태 계열 **3군** — 임상 해설의 **\"🟢 비교적 안전한 혼동\"** 목록\n#: (A1↔A4 · A5↔A6) 과 형태 설명을 그대로 따릅니다.\n#: ⚠️ **화면에 나가는 것은 이게 아니라 `MORPH_GROUP_KEEP_A6`(4군)** 입니다.\n#:    이 3군은 **실험 비교용**이라 과거 결과표와 대조가 되게 **옛 이름을 그대로**\n#:    둡니다 (2026-09-10 에 4군 이름만 보호자 말로 바꿨습니다).\nMORPH_GROUP: dict[str, str] = {\n    \"A1\": \"융기·발진\",   \"A4\": \"융기·발진\",      # 둘 다 작은 융기, 고름 유무로만 갈림\n    \"A2\": \"표면 변화\",   \"A3\": \"표면 변화\",      # 표피 각질 / 색·두께 변화\n    \"A5\": \"손상·덩어리\", \"A6\": \"손상·덩어리\",    # 조기 진료 권장 쪽\n}\n\n#: ★ 같은 묶음인데 **A6(결절·종괴)만 따로 둡니다.** A6 은 종양 감별이 필요한\n#: 유일한 클래스라, 다른 것과 한 이름으로 부르면 그 뜻이 사라집니다.\n#: 실측(STEP 28): 커버리지 62.1% → **61.4%** 로 **0.7%p 밖에 안 잃습니다.**\n#: 전체 데이터 정확도도 0.8365 → 0.8315.\nURGENT_CODES: tuple[str, ...] = (\"A5\", \"A6\")\n\"\"\"★ 계열 4군 중 **급한 쪽의 클래스 코드**. 이름이 아니라 코드로 적습니다.\n\n⚠️ **예전에는 이름 문자열이었습니다** (`(\"미란·궤양\", \"결절·종괴\")`).\n그러면 `MORPH_GROUP_KEEP_A6` 의 이름을 바꿀 때 여기를 같이 안 고치면\n**하향 방지 규칙이 조용히 안 걸립니다** — 에러도 안 나고, A5 하향이\n36.8% 에서 43.8% 로 돌아갑니다. 2026-09-10 에 이름을 바꾸면서 코드로\n옮겼습니다. `URGENT_GROUPS` 는 이제 **아래에서 파생**됩니다.\"\"\"\n\nDOWNGRADE_BLOCK_MIN = 0.25\n\"\"\"★ **하향 방지 규칙**의 문턱 (STEP 35).\n\n    급한 쪽(`URGENT_GROUPS`) 확률의 **합**이 이 값 이상이면,\n    덜 급한 묶음은 **답 후보에서 뺍니다.**\n\n그러면 급한 쪽 이름을 말하거나, 확신이 모자라면 **아무 말도 안 합니다.**\n\n왜 필요한가 — 전체 하향은 3.7% 로 관문(5%)을 통과하는데 **말한 미란·궤양의\n43.8%** 가 하향이었습니다. 표면 변화(전체의 51%)가 평균을 떠받쳐 A5 가 묻힙니다.\n\n실측(holdout): 커버리지 67.9 → **66.5%**(−1.3%p), A5 하향 43.8 → **36.8%**\n(−7.0%p), 전체 하향 3.7 → 2.9%. **추론 0회**이고 릴리스 단독보다 두 축 다 낫습니다.\n\n⚠️ 문턱은 **val 에서 고르고 holdout 은 확인만** 했습니다 (통과한 건 0.25 하나).\n\n★ 이 값은 **판단으로 번역됩니다** — `0.25 ≈ \"하향은 다른 오답의 2배로 나쁘다\"`.\n3배로 보면 커버리지 62.7%, 5배면 55.0%, 8배면 39.8% 입니다. *\"0.25 로 하자\"* 는\n반박 불가지만 *\"하향은 과잉보다 2배 나쁘다\"* 는 **반박 가능**합니다 — 그게 이\n상수가 있는 자리입니다. 재현: `uv run python tools/a5_downgrade.py`\"\"\"\n\nA6_ALERT_MIN = 0.40\n\"\"\"★ **\"덩어리가 의심됩니다\" 경보**의 문턱 (STEP 34, 2026-09-08 켜기로 결정).\n\n⚠️ **`p(이상) × p(A6)` 에 겁니다** — `p(A6)` 단독이 아닙니다.\n`tools/naming_granularity.py` 의 `score = p1 * ens[:, ia6]` 와 같은 값이어야\nSTEP 34 의 표를 그대로 읽을 수 있습니다. `GROUP_CONF_MIN` 과 같은 모양입니다.\n\n실측 (STEP 34 — **같은 문턱**에서 val / holdout):\n\n    문턱    val 재현율   ho 재현율   val 정밀도   ho 정밀도\n    0.30     68.6%      68.2%       71.6%      58.2%\n    0.40     61.1%      59.4%       81.4%      71.3%      <- 채택\n    0.50     55.0%      51.8%       84.0%      77.6%\n    0.70     38.4%      36.9%       90.4%      85.1%\n\n★ **정밀도가 아니라 재현율로 정했습니다.** 정밀도는 모델의 성질이 아니라\n**모델 × 유병률**의 성질이라 문턱을 고정해도 안 고정됩니다 (A6 이 holdout 에서\n절반만큼 드물어 정밀도만 5~13%p 떨어졌습니다). 재현율은 유병률과 무관해\n양쪽에서 유지됩니다 — 0.40 에서 **약 60%**.\n\n⚠️ 이 경보만 **병변 이름을 말합니다.** 예외인 이유:\n  · 임상 해설이 *\"A6 으로 오탐하는 건 상대적으로 안전\"* 이라고 적어뒀습니다\n    (병원에 가서 확인하면 되니까) — 반대 방향(놓침)이 훨씬 나쁩니다\n  · A6 은 **종양 감별**이 필요한 유일한 클래스라 4묶음에서도 혼자 뒀습니다\n끄려면 `DOG_SKIN_SHOW_A6_ALERT=0`.\"\"\"\n\nMORPH_GROUP_KEEP_A6: dict[str, str] = {\n    \"A1\": \"솟아오른 변화\", \"A4\": \"솟아오른 변화\",\n    \"A2\": \"피부 표면·색·두께 변화\", \"A3\": \"피부 표면·색·두께 변화\",\n    \"A5\": \"벗겨지거나 패인 상처\", \"A6\": \"깊거나 단단한 혹\",\n}\n\n#: ★ 급한 쪽 **묶음 이름** — `URGENT_CODES` 에서 **파생**합니다.\n#:   손으로 적지 마세요. 위 표의 이름을 바꾸면 여기가 알아서 따라옵니다.\nURGENT_GROUPS: tuple[str, ...] = tuple(\n    dict.fromkeys(MORPH_GROUP_KEEP_A6[c] for c in URGENT_CODES))\n\n#: ★ **그 묶음이 어떤 라벨을 담고 있나** — 화면에 괄호로 붙습니다 (2026-09-10).\n#:\n#:   `솟아오른 변화` 만 들고 병원에 가면 **수의사가 못 알아듣습니다.** 보호자가\n#:   전달할 수 있는 말이 있어야 합니다. 이건 **데이터 라벨의 이름**입니다.\n#:\n#: ⚠️ **\"1등 병변\" 과 다릅니다.** 그건 *\"이 개는 구진입니다\"* 라고 **단정**하는\n#:    것이고(holdout 46.3% 틀림), 이건 *\"이 묶음은 이런 것들을 담는다\"* 는\n#:    **용어 풀이**입니다. 예측이 아니라 정확도 문제가 안 걸립니다.\n#: ⚠️ **순서는 코드순(A1→A6)으로 고정합니다.** 확률순으로 두면 첫 이름이\n#:    \"1등\" 으로 읽혀서 그때는 진짜로 top1 을 되살리는 셈이 됩니다.\nGROUP_LABELS: dict[str, str] = {\n    g: \"·\".join(CLASS_KO[c] for c in CLASSES\n                if c != NORMAL_LABEL and MORPH_GROUP_KEEP_A6[c] == g)\n    for g in dict.fromkeys(MORPH_GROUP_KEEP_A6.values())\n}\n\n#: ★ **보호자가 사진에서 보는 특징** (2026-09-10, 수의학 근거 대조 후 채택).\n#:   화면에 계열 이름 바로 아래 한 줄로 붙습니다.\n#:   ⚠️ *\"피가 나면 신속 진료\"* 같은 **조건부 긴급도를 넣지 마세요** — STEP 30 에서\n#:      *\"계열 이름과 긴급도를 같이 띄우면 말한 것의 절반이 한 단계 부풀려진다\"* 로\n#:      막아둔 자리입니다. 진료 권고는 이미 마지막 줄에서 한 번 나갑니다.\nGROUP_FEATURE: dict[str, str] = {\n    #   ⚠️ 받은 표는 \"고름이 찬 **병변**\" 이었는데, 보호자 화면에서 `병변` 을 빼기로\n    #      해서(2026-09-10) 여기만 \"자리\" 로 바꿨습니다. 되돌리려면 이 줄만 고치면 됩니다.\n    \"솟아오른 변화\":           \"돌기, 넓게 솟은 부위, 고름이 찬 자리\",\n    \"깊거나 단단한 혹\":        \"피부 안쪽 또는 표면의 덩어리\",\n    \"피부 표면·색·두께 변화\":  \"딱지, 둥근 비늘, 검어진 피부, 두꺼워진 피부\",\n    \"벗겨지거나 패인 상처\":    \"까짐, 진물, 출혈, 깊게 패인 부위\",\n}\n\n#: ★ **수의학적 의미** — \"자세히 보기\" 에만 넣습니다. 본문에 두지 않습니다.\n#:   primary/secondary 는 *진단 순서*의 축이지 *보호자에게 뭐라고 부를지*의\n#:   축이 아닙니다 (STEP 32 에서 그 축으로 2군을 만들었다가 과잉 88.4% 로 기각).\nGROUP_DETAIL: dict[str, str] = {\n    \"솟아오른 변화\":           \"주로 일차 병변. 단단한 병변과 내용물이 찬 병변은 하위 분류로 구분\",\n    \"깊거나 단단한 혹\":        \"염증·낭종·종양 등을 감별해야 하는 일차 병변\",\n    \"피부 표면·색·두께 변화\":  \"병변 진행 후 흔적 또는 만성 염증성 변화\",\n    \"벗겨지거나 패인 상처\":    \"표피 또는 더 깊은 조직이 소실된 이차 병변\",\n}\n\n#: ★ **문헌 근거 (2026-09-08, 원문을 직접 열어 확인)** — 세 축이 각각 표준입니다:\n#:   ① primary / secondary — Merck Vet Manual 이 목록으로 나눕니다\n#:      (secondary: epidermal collarettes · erosions/ulcers · lichenification …)\n#:   ② **1cm 경계** → **A6 을 따로 둔 근거**\n#:      *\"solid elevated lesion **<1cm** diameter\"* (papule) /\n#:      *\"circumscribed solid elevation **>1cm** in diameter that usually\n#:       **extends into deeper layers of skin**\"* (nodule) — Cornell AHDC.\n#:      Veterian Key 도 *\"approximately 1 cm in diameter or smaller\"*.\n#:      ⚠️ **Merck 페이지 자체에는 cm 수치가 없습니다** — 여기 근거로 대지 마세요.\n#:   ③ **표피 소실 여부** → **A5 를 가르는 선**\n#:      Merck: 미란·궤양은 *\"**loss of the epidermis**\"*.\n#:      ⚠️ **\"기저막(basement membrane) 파괴 여부\" 로 말하면 틀립니다.**\n#:         미란(erosion)은 표피 일부만 잃고 **기저막은 온전**하며 흉터 없이\n#:         낫습니다. 기저막까지 가는 것은 궤양(ulcer)뿐입니다. 그 축으로\n#:         가르면 **미란이 A2·A3 쪽에 붙어** 묶음이 무너집니다.\n#:         우리 축은 *표피가 쌓이는가(A2·A3) / 소실되는가(A5)* 입니다.\n#:      장벽이 깨지면 `S. pseudintermedius` 2차 감염 위험 —\n#:      Hillier et al. (2014) *Vet Dermatol* 25(3):163-e43 (ISCAID 지침)\n#:   그리고 A2+A3 는 **Hensel, Santoro, Favrot, Hill, Griffin (2015),\n#:   BMC Vet Res**(ICADA 개 아토피 가이드라인)의 한 문장에 같이 있습니다:\n#:      \"Typical secondary skin lesions are excoriations, alopecia,\n#:       **lichenification, hyperpigmentation, crusting, and seborrhea**.\"\n#: ⚠️ 단 그 목록엔 `excoriations` 도 들어 있는데 우리는 그걸 A5 쪽으로 가릅니다.\n#:    즉 그 문장 하나로 정해지지 않고 **②③ 축과의 조합**이 우리 묶음입니다.\n#:    **이 조합을 쓴 선례는 못 찾았습니다.**\n#:\n#: ⚠️ **정직하게 적어둡니다** — 임상 해설이 \"안전한 혼동\" 으로 **명시한 것은\n#: 두 쌍뿐**입니다 (A1↔A4 · A5↔A6). `MORPH_GROUP` 의 A2+A3 은 그 문서가 인정한\n#: 게 아니라 **남은 것**이고, 이득의 대부분이 거기서 나옵니다:\n#:     6종 33.7% → 문서가 인정한 병합만 42.5% → A2+A3 까지 62.1%\n#: 즉 **\"임상적으로 비슷해서 묶었다\" 는 절반만 맞습니다.**\nDOC_ENDORSED_MERGES: tuple[tuple[str, str], ...] = ((\"A1\", \"A4\"), (\"A5\", \"A6\"))\n\n#: ★ 수의피부과 **표준 축** — primary(병이 직접 만든 것) vs secondary(그 뒤에\n#: 생긴 것). 우리가 만든 묶음이 아니라 교과서 분류입니다:\n#:   primary    macule · papule · plaque · wheal · vesicle · **pustule** · **nodule**\n#:   secondary  **collarette** · scar · excoriation · **erosion** · **ulcer** ·\n#:              fissure · **lichenification** · callus\n#:   둘 다 됨    **scale · crust · 색소 변화** (원인에 따라)\n#: 출처: MSD Veterinary Manual · Clinicians Brief (2026-09-06 확인)\n#:\n#: ⚠️ **A2·A3 는 깨끗이 안 갈립니다** — A2 의 잔고리는 secondary 인데 비듬·가피는\n#:    '둘 다', A3 의 태선화는 secondary 인데 색소침착은 '둘 다' 입니다.\n#:    ★ 그런데 **둘이 똑같이 애매합니다.** 출처가 *\"secondary 는 여러 primary 에서\n#:    나올 수 있어 진단 가치가 낮다\"* 고 적는데, A2+A3 를 한 통에 넣는다는 건\n#:    **덜 진단적인 둘을 묶는 것**이라 사후 정당화가 아닌 근거가 됩니다.\n#: ⚠️ A6 를 따로 둔 근거인 **1cm 는 교과서 경계**입니다 (papule ≤1cm /\n#:    nodule >1cm, 결절은 진피·피하로 더 깊이) — 표준과 맞습니다.\n#:    🔴 **단 \"A1·A4 = ≤1cm\" 로 말하면 틀립니다 (2026-09-10 정정).**\n#:    A1 의 **플라크(plaque)는 보통 >1cm** 입니다. 1cm 는 *구진 vs 결절* 의\n#:    경계이지 *묶음 전체*의 크기가 아닙니다. 융기·발진의 축은 크기가 아니라\n#:    **\"표면보다 솟았는가\"** 이고, A1(단단함)과 A4(고름이 참)는 그 안에서\n#:    내용물로 갈립니다.\n#: 🚫 **그런데 이 축으로 2군을 만들면 기각입니다** (STEP 32): 커버리지는 제일\n#:    높은데(74.6%) primary 에 A6 이 들어 있어 **과잉 분류가 88.4%** 입니다 —\n#:    구진 하나에도 '조기 진료' 가 붙어 그 말이 뜻을 잃습니다.\nLESION_ORIGIN: dict[str, str] = {\n    \"A1\": \"primary\", \"A4\": \"primary\", \"A6\": \"primary\",\n    \"A5\": \"secondary\",\n    \"A2\": \"mixed\",   # 잔고리 secondary / 비듬·가피 '둘 다'\n    \"A3\": \"mixed\",   # 태선화 secondary / 색소침착 '둘 다'\n}\n\n#: 🔴 위험한 혼동 — 임상 해설에 적힌 그대로. **긴급도를 낮춰 말하는 것**입니다.\n#:    (A6→A2 가 최악, A5→A1, A6→A1). 일반화하면 \"실제 등급 > 말한 등급\".\nDANGEROUS_IS_UNDER_TRIAGE = True\n\n# ──────────────────────────────────────────────────────────────\n# 촬영 가이드 밴드 — **이 값들의 출처는 여기 하나뿐입니다**\n# ──────────────────────────────────────────────────────────────\n# `robust.usable_range()` 가 STEP 16 에서 실측한 배율 밴드입니다 (n=2,000).\n# 쓰는 곳:\n#     src/agent.py        화면 점유율로 (÷ 2.5 — 2단계 크롭이 m2.5)\n#     tools/box_error.py  네모 크기 오차로 뒤집어서 (1 / 배율)\n#     demo/index.html     화면\n#     docs/SERVING.md     사람이 읽는 판\n#\n# ⚠️ **왜 `robust.py` 가 아니라 여기인가** — `robust.py` 는 torch 를 import\n#    합니다. 밴드만 필요한 쪽(`box_error.py` 는 `uv run python` 으로 도는\n#    측정 도구)이 torch 를 끌어오게 됩니다. 실제로 그래서 백엔드 사본이\n#    깨졌습니다 (2026-09-06). **상수는 의존성 없는 곳에 둡니다.**\n#\n# ⚠️ STEP 10 은 (0.85, 1.4) / (0.7, 1.7) 이었습니다. **크게 찍는 쪽이\n#    빡빡해졌습니다** — 2.0x 에서 macro-F1 이 0.449 까지 떨어집니다.\n# ⚠️ 데이터가 늘거나 크롭이 바뀌면 `usable_range()` 를 **다시 돌려** 고치세요.\nZOOM_RECOMMEND = (0.7, 1.2)     # 최고점 대비 하락 5% 이내 (STEP 16, n=2,000)\nZOOM_ALLOW = (0.6, 1.4)         # 하락 10% 이내\nZOOM_CENTER_MAX = 0.10          # 병변이 화면 중앙에서 이만큼 이내\n\n# AI Hub 데이터셋 식별자\nAIHUB_DATASET_KEY = \"561\"\nAIHUB_DATASET_NAME = \"반려동물 피부 질환 데이터\"\n\n# 우리가 쓸 데이터 범위 (사용자 결정: 반려견 + 일반카메라만)\nINCLUDE_SPECIES = [\"반려견\"]\nEXCLUDE_SPECIES = [\"반려묘\"]\nINCLUDE_CAMERA = [\"일반카메라\"]\nEXCLUDE_CAMERA = [\"더모스코프\"]  # 보호자가 만들 수 없는 입력이므로 제외\n\n\n# ──────────────────────────────────────────────────────────────\n# 실행 설정\n# ──────────────────────────────────────────────────────────────\n@dataclass\nclass CFG:\n    # --- 재현성 ---\n    seed: int = 42\n    deterministic: bool = False\n\n    # --- 데이터 ---\n    img_size: int = 288\n    crop_margin: float = 1.5          # ROI 크롭 여유 배율. 1.0=박스 딱 맞게, 2.0=주변 2배\n    crop_min_px: int = 64             # 이보다 작은 병변 박스는 버림 (노이즈)\n    # >0 이면 margin 대신 **고정 픽셀 창**으로 자릅니다 (병변 중심, 항상 같은 크기).\n    # margin 크롭은 병변 크기에 따라 확대 배율이 달라져 그 배율이 정답을 흘립니다.\n    # → src/crop.py 의 fixed_box() 설명, docs/cautions/08 참고\n    crop_fixed_px: int = 0\n    save_crop_size: int = 512         # 디스크에 저장할 크롭 해상도 (학습 시 img_size 로 리사이즈)\n    save_crop_quality: int = 92\n\n    # --- 중복 제거 ---\n    phash_size: int = 16              # phash 비트 크기 (16 → 256bit, 기본 8보다 정밀)\n    dedup_hamming: int = 6            # 이 거리 이하면 near-duplicate 로 봄\n\n    # --- 분할 ---\n    n_folds: int = 5\n    use_fold: int = 0                 # 단일 실험에서 검증에 쓸 fold\n    holdout_ratio: float = 0.15       # 최종 1회만 보는 테스트셋 비율 (개체 단위)\n\n    # --- 모델 ---\n    model_name: str = \"tf_efficientnetv2_s.in21k_ft_in1k\"\n    pretrained: bool = True\n    drop_rate: float = 0.2\n    drop_path_rate: float = 0.1\n\n    # --- 학습 ---\n    epochs: int = 15\n    batch_size: int = 0               # 0 이면 env.suggest_batch_size() 로 자동\n    grad_accum: int = 1\n    lr: float = 3e-4\n    backbone_lr_mult: float = 0.1     # 백본은 헤드보다 낮은 lr (파인튜닝 관례)\n    weight_decay: float = 0.05\n    warmup_epochs: int = 2\n    label_smoothing: float = 0.1\n    amp: bool = True\n    ema_decay: float = 0.999          # 0 이면 EMA 끔\n    clip_grad_norm: float = 1.0\n    num_workers: int = -1        # -1 = CPU 코어 수에 맞춰 자동\n    early_stop_patience: int = 5\n    monitor: str = \"macro_f1\"         # ⚠️ accuracy 아님. 불균형 데이터에서 accuracy 는 거짓말을 합니다.\n\n    # --- 증강 ---\n    # ⚠️ 피부 병변은 \"색과 질감\"이 곧 라벨입니다.\n    #    강한 색상 증강은 A3(과다색소침착)을 A1 처럼 만들어 라벨을 파괴합니다.\n    #    아래 값은 일반 이미지 분류 기본값보다 의도적으로 약하게 잡았습니다.\n    rrc_scale: tuple[float, float] = (0.7, 1.0)\n    # ⚠️ RandomResizedCrop 은 **축소를 못 합니다.** 이미지의 일부를 잘라 확대할 뿐이라\n    #    가장 축소된 경우가 \"이미지 전체\"(검증 대비 약 0.88배)이고, rrc_scale 하한을\n    #    낮추면 **확대 쪽만** 넓어집니다. 실측:\n    #        default(0.70,1.0)      → 0.88x ~ 1.05x\n    #        scale_robust(0.35,1.0) → 0.88x ~ 1.49x\n    #    그런데 배율 교란 검사는 0.5x·0.71x 를 묻습니다. 훈련에서 **한 번도 안 본**\n    #    구간이라, rrc_scale 을 아무리 넓혀도 그 하락은 안 줄어듭니다 (실측 확인).\n    #    축소를 배우려면 이미지를 줄여 여백을 채우는 affine 변환이 필요합니다.\n    affine_scale: tuple[float, float] | None = None   # 예: (0.5, 1.3) — 축소 포함\n    hflip: float = 0.5\n    vflip: float = 0.0                # 피부 사진은 위아래 뒤집기가 부자연스러움\n    rotate_deg: int = 15\n    color_jitter: float = 0.1         # brightness/contrast/saturation 공통 강도 (약하게!)\n    hue_jitter: float = 0.02          # 색조는 특히 조심 — 거의 건드리지 않음\n    randaugment_n: int = 0            # 0 이면 끔. 켜려면 2 권장\n    randaugment_m: int = 7\n    mixup_alpha: float = 0.0          # 실험용. 켜면 0.2 권장\n    cutmix_alpha: float = 0.0\n    random_erasing: float = 0.15\n\n    # --- albumentations 전용 노브 ---\n    # torchvision 에는 없거나 느린 것들. albumentations 가 없으면 전부 무시됩니다.\n    # ⚠️ 촬영 조건(흐림·노이즈·조명)을 흉내 내는 쪽입니다. 실측: 정상 사진의\n    #    선명도 중앙값이 39, 병변은 271 — 모델이 화질로 맞힐 여지가 있어서\n    #    흐림을 훈련에 넣어두면 그 지름길을 막는 효과도 기대할 수 있습니다.\n    blur_p: float = 0.0               # 가우시안/모션 블러 확률\n    noise_p: float = 0.0              # 센서 노이즈 확률\n    jpeg_p: float = 0.0               # JPEG 압축 열화 (보호자 사진은 대개 압축됨)\n    clahe_p: float = 0.0              # 국소 대비 보정 — 질감을 살리는 방향\n    shift_limit: float = 0.0          # 평행이동 비율 (위치 교란 20.6% 대응)\n\n    # --- 불균형 대응 ---\n    balance_strategy: str = \"class_weight\"\n    # \"none\" | \"class_weight\" | \"weighted_sampler\" | \"hair_weighted\"\n    #   hair_weighted — 털처럼 가는 선이 많은 **정상** 사진을 더 자주 뽑습니다.\n    #   헛알림 실측(AUROC 0.749)에서 나온 값이고, 클래스 총량은 보존합니다.\n    hair_alpha: float = 1.0   # 0=끔, 1=최상위가 최하위보다 2배 자주\n    focal_gamma: float = 0.0                 # >0 이면 focal loss 사용\n\n    # --- 평가 / 안전장치 ---\n    tta_hflip: bool = True\n    calibrate: bool = True                   # 온도 스케일링\n    target_recall_stage1: float = 0.95       # 1단계 정상/이상: 놓치지 않는 게 우선\n    abstain_threshold: float = 0.45          # 최고확률이 이 미만이면 \"판단 어려움\"\n    topk_report: int = 3                     # 사용자에게 상위 몇 개까지 보여줄지\n\n    # --- 로깅 ---\n    exp_name: str = \"baseline\"\n    log_every: int = 50\n    use_wandb: bool = False\n\n    def resolved_batch_size(self) -> int:\n        if self.batch_size > 0:\n            return self.batch_size\n        from src import env\n\n        scale = _infer_scale(self.model_name)\n        # 백본별 메모리 보정 — timm 이름으로 MODEL_ZOO 를 되짚습니다\n        mf = 1.0\n        for spec in MODEL_ZOO:\n            if spec.timm_name == self.model_name or spec.key == self.model_name:\n                mf = spec.mem_factor\n                break\n        return env.suggest_batch_size(self.img_size, scale, mem_factor=mf)\n\n    def resolved_num_workers(self) -> int:\n        \"\"\"-1 이면 CPU 코어 수에 맞춰 정합니다.\"\"\"\n        if self.num_workers >= 0:\n            return self.num_workers\n        from src import env\n\n        return env.suggest_workers()\n\n    def to_dict(self) -> dict[str, Any]:\n        return asdict(self)\n\n    def save(self, path: str | Path) -> Path:\n        import json\n\n        p = Path(path)\n        p.parent.mkdir(parents=True, exist_ok=True)\n        p.write_text(json.dumps(self.to_dict(), indent=2, ensure_ascii=False), encoding=\"utf-8\")\n        return p\n\n    @classmethod\n    def from_dict(cls, data: dict[str, Any]) -> \"CFG\":\n        \"\"\"알 수 없는 키는 무시하고 만듭니다 (설정 파일이 구버전이어도 동작).\"\"\"\n        data = dict(data)\n        # tuple 필드 복원 (JSON 은 list 로 저장됨)\n        if isinstance(data.get(\"rrc_scale\"), list):\n            data[\"rrc_scale\"] = tuple(data[\"rrc_scale\"])\n        if isinstance(data.get(\"affine_scale\"), list):\n            data[\"affine_scale\"] = tuple(data[\"affine_scale\"])\n        known = set(cls.__dataclass_fields__)\n        return cls(**{k: v for k, v in data.items() if k in known})\n\n    @classmethod\n    def load(cls, path: str | Path) -> \"CFG\":\n        import json\n\n        return cls.from_dict(json.loads(Path(path).read_text(encoding=\"utf-8\")))\n\n\ndef _infer_scale(model_name: str) -> str:\n    n = model_name.lower()\n    for tag in (\"nano\", \"tiny\", \"small\", \"base\", \"large\", \"huge\"):\n        if tag in n:\n            return {\"nano\": \"tiny\", \"huge\": \"large\"}.get(tag, tag)\n    # efficientnet_b0..b7 같은 이름 처리\n    if \"_b0\" in n or \"_b1\" in n or \"_s.\" in n or n.endswith(\"_s\"):\n        return \"small\"\n    if \"_b2\" in n or \"_b3\" in n or \"_m.\" in n:\n        return \"base\"\n    return \"base\"\n\n\n# ──────────────────────────────────────────────────────────────\n# 증강 프리셋\n#\n# 기본값은 의도적으로 약합니다 — 피부 병변은 색과 질감이 곧 라벨이라\n# 강한 색상 증강은 A3(과다색소침착)를 A1 처럼 만들어 버립니다.\n#\n# 그런데 **배율**은 사정이 다릅니다. 크롭이 병변 박스에 맞춰 잘리기 때문에\n# 배율이 클래스마다 다르고(실측 A1 0.47% ~ A6 3.08%, 6.5배), 모델이 그걸\n# 단서로 쓸 수 있습니다. 실사용에서 배율은 무작위이므로 그건 무너집니다.\n#\n# ⚠️ 기본 rrc_scale=(0.7, 1.0) 은 면적 1.43배 범위(선형 1.2배)입니다.\n#    막아야 할 격차가 선형 2.5배인데 이건 아무 효과가 없습니다.\n#\n# ⚠️ 넓은 배율 증강에는 **여유가 있는 크롭**이 필요합니다.\n#    m1.5 처럼 딱 붙은 크롭에 0.15 를 걸면 병변이 화면에서 잘려 나가\n#    라벨이 깨집니다. m2.5 나 f512 같은 넉넉한 크롭을 base 로 쓰세요.\n#    효과 판정은 src/robust.py 의 scale_stress() 로 합니다.\n# ──────────────────────────────────────────────────────────────\nAUG_PRESETS: dict[str, dict] = {\n    # ── 기준 ────────────────────────────────────────────────────\n    \"default\": {},\n\n    # ── ① 폭을 **줄이는** 쪽 (멘토 피드백 1번) ──────────────────\n    # 지금까지 넓히기만 두 번 시도해 둘 다 실패했습니다. 반대 방향은 안 해봤습니다.\n    # 좁히면 과제가 쉬워져 점수가 오를 수 있고, 대신 견고성은 나빠질 수 있습니다.\n    # 좁히기 축에 **점을 두 개** 찍습니다. 하나만 찍으면 \"이겼다\" 는 알아도\n    # \"더 좁혀야 하나\" 를 몰라서 또 한 판 돌려야 합니다.\n    #     0.70(default) → 0.85(narrow) → 0.92(narrower)\n    #   계속 좋아지면 더 좁히고, narrow 가 최고면 거기가 최적점이고,\n    #   셋이 비슷하면 이 축은 상관없다는 뜻입니다. 한 번에 결론이 납니다.\n    \"narrow\": {\"rrc_scale\": (0.85, 1.0), \"rotate_deg\": 10, \"random_erasing\": 0.0},\n    \"narrower\": {\"rrc_scale\": (0.92, 1.0), \"rotate_deg\": 5, \"random_erasing\": 0.0},\n\n    # ── ② 축소를 가르치는 쪽 (2단계 최악 조건이 0.5x) ───────────\n    # affine 이 이미지를 실제로 줄이고 여백을 채웁니다. RRC 는 축소를 못 합니다.\n    \"zoom_both\": {                       # 세게 — 224px 에서는 실패했던 설정\n        \"rrc_scale\": (0.5, 1.0),\n        \"affine_scale\": (0.45, 1.25),\n        \"rotate_deg\": 20,\n        \"random_erasing\": 0.25,\n    },\n    \"zoom_mild\": {                       # 완만하게 — ①과 ②의 절충\n        \"rrc_scale\": (0.75, 1.0),\n        \"affine_scale\": (0.70, 1.15),\n        \"rotate_deg\": 12,\n    },\n\n    # ── ③ 위치 교란 대응 (실측 20.6% 하락) ─────────────────────\n    \"shift\": {\"shift_limit\": 0.15, \"rotate_deg\": 15},\n\n    # ── ④ 촬영 조건 흉내 (멘토 피드백 4번) ─────────────────────\n    # 배율은 기본값 그대로 두고 화질만 흔듭니다.\n    # 실측: 정상 사진 선명도 중앙값 39 vs 병변 271 — 모델이 화질로 맞힐 여지가\n    # 있어서, 흐림을 훈련에 넣으면 그 지름길을 막는 효과도 기대할 수 있습니다.\n    \"photometric\": {\"blur_p\": 0.3, \"noise_p\": 0.25, \"jpeg_p\": 0.3, \"clahe_p\": 0.2},\n\n    # ── ⑤ 조합 ─────────────────────────────────────────────────\n    # ⚠️ 아래 둘은 정의만 남겨둡니다 — 03b 스윕에서는 뺐습니다.\n    #    zoom_shift 는 zoom_mild + shift 결과로 대충 예상되고,\n    #    kitchen_sink 는 \"너무 세면 나빠진다\" 를 확인하는 대조군이라 기대값이 낮습니다.\n    \"zoom_shift\": {                      # 축소 + 이동 (두 교란을 같이)\n        \"rrc_scale\": (0.75, 1.0),\n        \"affine_scale\": (0.70, 1.15),\n        \"shift_limit\": 0.12,\n        \"rotate_deg\": 15,\n    },\n    \"kitchen_sink\": {                    # 전부 — 과한 게 해로운지 확인하는 상한선\n        \"rrc_scale\": (0.6, 1.0),\n        \"affine_scale\": (0.55, 1.25),\n        \"shift_limit\": 0.15,\n        \"rotate_deg\": 20,\n        \"blur_p\": 0.25, \"noise_p\": 0.2, \"jpeg_p\": 0.25,\n        \"random_erasing\": 0.25,\n    },\n\n    # ── 과거 실험용 (결론 남음. 참고로만 둡니다) ────────────────\n    # 확대만 넓힙니다. 2단계 최악 조건이 축소라 여기엔 효과가 없습니다.\n    \"scale_robust\": {\"rrc_scale\": (0.35, 1.0), \"rotate_deg\": 20, \"random_erasing\": 0.25},\n}\n\n\n# 파인튜닝 강도 프리셋\n#\n# ⚠️ 용어 정리 — 자주 헷갈립니다.\n#   \"전이학습(transfer learning)\" 은 우산 개념입니다. 그 안에 두 방식이 있습니다:\n#     · linear probe   백본을 얼리고 헤드만 학습 (freeze_backbone(True))\n#     · fine-tuning    백본까지 같이 학습 ← **우리는 처음부터 이쪽입니다**\n#   `freeze_backbone()` 은 정의만 되어 있고 어디서도 호출하지 않습니다.\n#\n# 그러면 남는 질문은 \"파인튜닝을 할까?\" 가 아니라 \"얼마나 세게 할까?\" 입니다.\n# 그걸 정하는 게 backbone_lr_mult 입니다:\n#\n#     헤드 lr   = cfg.lr                        (랜덤 초기화라 빨리 배워야 함)\n#     백본 lr   = cfg.lr × backbone_lr_mult     (사전학습 지식을 지키려고 낮춤)\n#\n# 기본 0.1 은 \"ImageNet 특징이 이미 쓸만하다\" 는 전제입니다. 그런데 우리 과제는\n# 물체 인식이 아니라 **피부 질감·색의 미세한 구분**이라 도메인 격차가 큽니다.\n# 백본이 3e-5 로 움직이면 12 에폭 동안 거의 제자리입니다.\n#\n# 실측 근거 (VL01 2단계): train 1.352 / val 1.474 — 학습 데이터조차 잘 못 맞춥니다.\n# 과적합이 아니라 **덜 배운** 상태이고, 백본 lr 이 유력한 원인입니다.\n# ──────────────────────────────────────────────────────────────\nFT_PRESETS: dict[str, dict] = {\n    # 지금까지 쓰던 설정 (비교 기준)\n    \"conservative\": {\"backbone_lr_mult\": 0.1},\n\n    # 백본을 3배 더 움직입니다. 도메인 격차가 클 때의 표준적인 선택.\n    \"moderate\": {\"backbone_lr_mult\": 0.3, \"warmup_epochs\": 3},\n\n    # 백본과 헤드를 같은 lr 로. 격차가 아주 클 때 가장 좋을 수 있지만\n    # 사전학습 지식을 잃을 위험(catastrophic forgetting)이 있어 warmup 을 길게 둡니다.\n    \"aggressive\": {\"backbone_lr_mult\": 1.0, \"warmup_epochs\": 4, \"lr\": 1e-4},\n\n    # 백본을 얼리고 헤드만. 우리 데이터(1.5만장)에는 부족하지만,\n    # \"백본 적응이 실제로 기여하는가\" 를 재는 대조군으로 유용합니다.\n    \"linear_probe\": {\"backbone_lr_mult\": 0.0},\n}\n\n\ndef ft_preset(name: str) -> dict:\n    \"\"\"파인튜닝 강도 프리셋 → CFG 오버라이드 사전.\"\"\"\n    if name not in FT_PRESETS:\n        raise KeyError(f\"모르는 프리셋: {name}. 가능: {sorted(FT_PRESETS)}\")\n    return dict(FT_PRESETS[name])\n\n\ndef with_finetune(cfg: \"CFG\", name: str) -> \"CFG\":\n    \"\"\"cfg 에 파인튜닝 강도 프리셋을 얹은 새 CFG.\"\"\"\n    d = {**cfg.to_dict(), **ft_preset(name)}\n    if name != \"conservative\":\n        d[\"exp_name\"] = f\"{cfg.exp_name}_{name}\"\n    return CFG.from_dict(d)\n\n\ndef aug_preset(name: str) -> dict:\n    \"\"\"프리셋 이름 → CFG 오버라이드 사전.\n\n        cfg = CFG(**{**CFG(model_name=\"resnet50\").to_dict(), **aug_preset(\"scale_robust\")})\n    \"\"\"\n    if name not in AUG_PRESETS:\n        raise KeyError(f\"모르는 프리셋: {name}. 가능: {sorted(AUG_PRESETS)}\")\n    return dict(AUG_PRESETS[name])\n\n\ndef with_aug(cfg: \"CFG\", name: str) -> \"CFG\":\n    \"\"\"cfg 에 증강 프리셋을 얹은 새 CFG. 실험 이름에 프리셋을 붙여 둡니다.\"\"\"\n    over = aug_preset(name)\n    d = {**cfg.to_dict(), **over}\n    if name != \"default\":\n        d[\"exp_name\"] = f\"{cfg.exp_name}_{name}\"\n    return CFG.from_dict(d)\n\n\n# ──────────────────────────────────────────────────────────────\n# 노트북 버전\n# ──────────────────────────────────────────────────────────────\n# ⚠️ 노트북 셀은 `git pull` 로 갱신되지 않습니다. Colab/Kaggle 에 올린 .ipynb 는\n#    다시 import 하기 전까지 그대로입니다. src/ 만 매번 최신이 됩니다.\n#    그래서 셀을 고칠 때마다 이 값을 올리고, 노트북 첫 셀이 자기가 들고 있는\n#    값과 비교해 **낡았으면 바로 알립니다.** (몇 시간 뒤에 알게 되면 늦습니다)\nNOTEBOOK_VERSION = \"2026-09-04.5\"\n\n# ★ 채택된 2단계 크롭. STEP 4C(비교) → 4D(재기준선) 에서 확정했습니다.\n#   노트북 05 가 불러온 체크포인트의 크롭이 이것과 다르면 **멈춥니다** —\n#   실제로 예전 실행(m1.5)의 출력을 붙이고 그대로 진행할 뻔했습니다.\n#   그러면 촬영 가이드·보정·임계값이 전부 버린 설정 기준으로 나옵니다.\nADOPTED_STAGE2_CROP = \"m2.5\"\n\n# ──────────────────────────────────────────────────────────────\n# 모델 라인업 — STEP 4 에서 순서대로 돌립니다.\n#\n# timm 모델명은 버전마다 바뀝니다. src/models.py 가 실행 시점에\n# timm.list_models() 로 존재를 검증하고, 없으면 fallback 을 씁니다.\n# ──────────────────────────────────────────────────────────────\n@dataclass\nclass ModelSpec:\n    key: str\n    timm_name: str\n    fallbacks: list[str] = field(default_factory=list)\n    img_size: int = 288\n    scale: str = \"base\"\n    # 배치 추천을 몇 배로 줄일지. env.suggest_batch_size 의 공식이 **ResNet 기준**\n    # 이라 트랜스포머 계열의 활성값 메모리를 모릅니다. 실측으로 정했습니다:\n    #   T4(14.6GB) · 256px · 배치 32 에서\n    #     swinv2_base   → OOM\n    #     siglip2_base  → 14.3GB (98%, EMA 붙으면 터짐)\n    #   같은 조건의 convnextv2_base 는 384px 배치 12 에서 8.9GB 로 멀쩡했습니다.\n    # 그래서 어텐션 계열은 0.4 로 잡습니다 (32 → 12).\n    mem_factor: float = 1.0\n    note: str = \"\"\n\n\nMODEL_ZOO: list[ModelSpec] = [\n    ModelSpec(\n        key=\"resnet50\",\n        timm_name=\"resnet50.a1_in1k\",\n        fallbacks=[\"resnet50\"],\n        img_size=224, scale=\"base\",\n        note=\"기준선. 다른 모든 수치는 이것과 비교해서 읽습니다.\",\n    ),\n    ModelSpec(\n        key=\"effnetv2_s\",\n        timm_name=\"tf_efficientnetv2_s.in21k_ft_in1k\",\n        fallbacks=[\"tf_efficientnetv2_s\", \"efficientnet_b3\"],\n        img_size=300, scale=\"small\",\n        note=\"가볍고 강함. 모바일 배포 1순위 후보.\",\n    ),\n    ModelSpec(\n        key=\"convnextv2_base\",\n        timm_name=\"convnextv2_base.fcmae_ft_in22k_in1k\",\n        fallbacks=[\"convnextv2_tiny.fcmae_ft_in22k_in1k\", \"convnext_base.fb_in22k_ft_in1k\"],\n        img_size=288, scale=\"base\",\n        note=\"현대 CNN 최강급. 질감(texture) 표현이 좋아 피부에 잘 맞을 가능성이 큼.\",\n    ),\n    ModelSpec(\n        key=\"swinv2_base\",\n        timm_name=\"swinv2_base_window12to16_192to256.ms_in22k_ft_in1k\",\n        fallbacks=[\"swinv2_base_window8_256.ms_in1k\", \"swin_base_patch4_window7_224.ms_in22k_ft_in1k\"],\n        img_size=256, scale=\"base\",\n        mem_factor=0.4,\n        note=\"계층적 ViT. 지역 패턴 + 전역 문맥을 같이 봄.\",\n    ),\n    ModelSpec(\n        key=\"eva02_base\",\n        timm_name=\"eva02_base_patch14_448.mim_in22k_ft_in22k_in1k\",\n        fallbacks=[\"eva02_small_patch14_336.mim_in22k_ft_in1k\", \"vit_base_patch16_224.augreg2_in21k_ft_in1k\"],\n        img_size=336, scale=\"base\",\n        mem_factor=0.4,\n        note=\"정확도 상한 확인용. T4 에서는 무거우니 배치 작게 + grad_accum 사용.\",\n    ),\n    ModelSpec(\n        key=\"siglip2_base\",\n        timm_name=\"vit_base_patch16_siglip_256.v2_webli\",\n        fallbacks=[\"vit_base_patch16_siglip_224.webli\", \"vit_base_patch16_clip_224.openai\"],\n        img_size=256, scale=\"base\",\n        mem_factor=0.4,\n        note=\"대규모 이미지-텍스트 사전학습 백본. 소량 데이터에서 특히 강한 편.\",\n    ),\n]\n\nMODEL_BY_KEY = {m.key: m for m in MODEL_ZOO}\n", "src/models.py": "\"\"\"timm 백본 팩토리.\n\ntimm 모델 이름은 버전마다 바뀝니다. \"1년 전 블로그에서 본 이름\"이 지금은\n없어서 노트북이 죽는 일이 흔합니다. 그래서 여기서는 **실행 시점에** 존재를\n확인하고, 없으면 fallback 으로 조용히 갈아탑니다 (경고는 찍습니다).\n\n    from src import models\n    m = models.build(\"convnextv2_base\", n_classes=6)\n    models.available()          # 이 환경에서 실제로 쓸 수 있는 모델 목록\n\"\"\"\n\nfrom __future__ import annotations\n\nimport warnings\nfrom dataclasses import dataclass\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\n\nfrom src.config import MODEL_BY_KEY, MODEL_ZOO, CFG, ModelSpec\n\n\n# ──────────────────────────────────────────────────────────────\n# 이름 해석\n# ──────────────────────────────────────────────────────────────\ndef _exists(name: str) -> bool:\n    import timm\n\n    base = name.split(\".\")[0]\n    try:\n        if name in timm.list_models(pretrained=True):\n            return True\n        # 태그 없이 등록된 경우\n        return base in timm.list_models()\n    except Exception:\n        return False\n\n\ndef resolve(spec: ModelSpec | str, verbose: bool = True) -> str:\n    \"\"\"spec 의 timm_name 이 없으면 fallback 을 순서대로 시도합니다.\"\"\"\n    import timm\n\n    if isinstance(spec, str):\n        spec = MODEL_BY_KEY.get(spec, ModelSpec(key=spec, timm_name=spec))\n\n    for cand in [spec.timm_name, *spec.fallbacks]:\n        if _exists(cand):\n            if cand != spec.timm_name and verbose:\n                print(f\"⚠️ [models] '{spec.timm_name}' 없음 → '{cand}' 로 대체 \"\n                      f\"(timm {timm.__version__})\")\n            return cand\n\n    # 마지막 방어선: 접두어가 같은 아무 모델\n    base = spec.timm_name.split(\".\")[0].split(\"_\")[0]\n    alt = [m for m in timm.list_models(pretrained=True) if m.startswith(base)]\n    if alt:\n        print(f\"⚠️ [models] '{spec.timm_name}' 및 fallback 모두 없음 → '{alt[0]}' 사용\")\n        return alt[0]\n    raise ValueError(\n        f\"'{spec.timm_name}' 을(를) 찾을 수 없습니다. `pip install -U timm` 후 다시 시도하거나 \"\n        f\"timm.list_models('*{base}*') 로 이름을 확인하세요.\"\n    )\n\n\ndef available(verbose: bool = True) -> list[dict]:\n    \"\"\"MODEL_ZOO 중 이 환경에서 실제로 쓸 수 있는 것들.\"\"\"\n    rows = []\n    for s in MODEL_ZOO:\n        try:\n            name = resolve(s, verbose=False)\n            ok = True\n        except Exception:\n            name, ok = \"—\", False\n        rows.append({\"key\": s.key, \"requested\": s.timm_name, \"resolved\": name,\n                     \"ok\": ok, \"img_size\": s.img_size, \"note\": s.note})\n    if verbose:\n        import timm\n\n        print(f\"timm {timm.__version__}\\n\")\n        for r in rows:\n            mark = \"✅\" if r[\"ok\"] else \"❌\"\n            same = \"\" if r[\"resolved\"] == r[\"requested\"] else f\"  →  {r['resolved']}\"\n            print(f\"{mark} {r['key']:<16} {r['requested']}{same}\")\n            print(f\"    {r['note']}\")\n    return rows\n\n\n# ──────────────────────────────────────────────────────────────\n# 생성\n# ──────────────────────────────────────────────────────────────\ndef build(\n    spec: ModelSpec | str,\n    n_classes: int,\n    pretrained: bool = True,\n    drop_rate: float = 0.2,\n    drop_path_rate: float = 0.1,\n    img_size: int | None = None,\n    verbose: bool = True,\n) -> nn.Module:\n    \"\"\"timm 모델을 만들고 분류 헤드를 n_classes 로 갈아 끼웁니다.\"\"\"\n    import timm\n\n    if isinstance(spec, str):\n        spec = MODEL_BY_KEY.get(spec, ModelSpec(key=spec, timm_name=spec))\n    name = resolve(spec, verbose=verbose)\n\n    kwargs: dict = dict(pretrained=pretrained, num_classes=n_classes, drop_rate=drop_rate)\n    # drop_path 를 지원 안 하는 모델이 있어 실패 시 빼고 재시도합니다.\n    try:\n        model = timm.create_model(name, drop_path_rate=drop_path_rate, **kwargs)\n    except TypeError:\n        model = timm.create_model(name, **kwargs)\n    except Exception as exc:\n        if \"img_size\" in str(exc) and img_size:\n            model = timm.create_model(name, img_size=img_size, **kwargs)\n        else:\n            raise\n\n    if verbose:\n        n_par = sum(p.numel() for p in model.parameters()) / 1e6\n        pc = getattr(model, \"pretrained_cfg\", {}) or {}\n        print(f\"[models] {name}  |  {n_par:.1f}M params  |  \"\n              f\"입력 {pc.get('input_size', '?')}  mean={tuple(round(x,3) for x in pc.get('mean', ()))}\")\n    return model\n\n\n# ──────────────────────────────────────────────────────────────\n# 파라미터 그룹 (layer-wise LR)\n# ──────────────────────────────────────────────────────────────\ndef param_groups(model: nn.Module, cfg: CFG) -> list[dict]:\n    \"\"\"백본은 낮은 lr, 새로 만든 헤드는 높은 lr.\n\n    사전학습된 백본은 이미 좋은 특징을 뽑고 있는데 높은 lr 로 흔들면\n    그 지식을 망가뜨립니다(catastrophic forgetting). 헤드는 랜덤 초기화라\n    빠르게 배워야 하고요. 파인튜닝의 기본 관례입니다.\n    (백본을 얼리는 linear probe 에는 해당하지 않습니다 — 거기선 백본 lr 이 아예 없습니다)\n\n    BatchNorm/LayerNorm/bias 에는 weight decay 를 걸지 않습니다 — 통상적인 관례로,\n    이들에 decay 를 걸면 정규화 통계가 왜곡됩니다.\n    \"\"\"\n    head_names = (\"head\", \"fc\", \"classifier\", \"last_linear\")\n    bb_lr = cfg.lr * cfg.backbone_lr_mult\n\n    # backbone_lr_mult == 0 은 \"백본을 얼린다\"(linear probe)는 뜻입니다.\n    # lr 0 으로 두면 결과는 같지만 백본까지 역전파해 계산을 낭비합니다.\n    # requires_grad 를 꺼서 실제로 건너뛰게 합니다.\n    freeze_bb = cfg.backbone_lr_mult == 0\n\n    groups = {\n        \"head_decay\": {\"params\": [], \"lr\": cfg.lr, \"weight_decay\": cfg.weight_decay},\n        \"head_nodecay\": {\"params\": [], \"lr\": cfg.lr, \"weight_decay\": 0.0},\n        \"bb_decay\": {\"params\": [], \"lr\": bb_lr, \"weight_decay\": cfg.weight_decay},\n        \"bb_nodecay\": {\"params\": [], \"lr\": bb_lr, \"weight_decay\": 0.0},\n    }\n    for n, p in model.named_parameters():\n        is_head = any(h in n for h in head_names)\n        if freeze_bb and not is_head:\n            p.requires_grad = False\n            continue\n        if not p.requires_grad:\n            continue\n        no_decay = p.ndim <= 1 or n.endswith(\".bias\")\n        key = (\"head_\" if is_head else \"bb_\") + (\"nodecay\" if no_decay else \"decay\")\n        groups[key][\"params\"].append(p)\n\n    return [g for g in groups.values() if g[\"params\"]]\n\n\ndef freeze_backbone(model: nn.Module, freeze: bool = True) -> None:\n    \"\"\"헤드만 먼저 학습하는 warmup 용. 보통 1~2 에폭만 얼렸다 풉니다.\"\"\"\n    head_names = (\"head\", \"fc\", \"classifier\", \"last_linear\")\n    for n, p in model.named_parameters():\n        if not any(h in n for h in head_names):\n            p.requires_grad = not freeze\n\n\n# ──────────────────────────────────────────────────────────────\n# EMA\n# ──────────────────────────────────────────────────────────────\nclass ModelEMA:\n    \"\"\"가중치의 지수이동평균.\n\n    학습 후반에 가중치가 최적점 주변을 진동할 때, 그 궤적의 평균이\n    어느 한 순간의 가중치보다 대체로 더 좋습니다. 공짜로 0.3~1%p 정도 얻습니다.\n    \"\"\"\n\n    def __init__(self, model: nn.Module, decay: float = 0.999):\n        import copy\n\n        self.ema = copy.deepcopy(model).eval()\n        for p in self.ema.parameters():\n            p.requires_grad_(False)\n        self.decay = decay\n\n    @torch.no_grad()\n    def update(self, model: nn.Module) -> None:\n        d = self.decay\n        for e, m in zip(self.ema.state_dict().values(), model.state_dict().values()):\n            if e.dtype.is_floating_point:\n                e.mul_(d).add_(m.detach(), alpha=1 - d)\n            else:\n                e.copy_(m)\n\n\n# ──────────────────────────────────────────────────────────────\n# 앙상블\n# ──────────────────────────────────────────────────────────────\n@dataclass\nclass Ensemble:\n    \"\"\"여러 모델의 logit 을 평균냅니다.\n\n    서로 다른 구조(CNN + ViT)를 섞을 때 가장 효과가 큽니다 — 틀리는 방식이\n    다르기 때문입니다. 같은 모델을 seed 만 바꿔 섞으면 이득이 작습니다.\n    \"\"\"\n\n    models: list[nn.Module]\n    weights: list[float] | None = None\n\n    @torch.no_grad()\n    def __call__(self, x: torch.Tensor) -> torch.Tensor:\n        w = self.weights or [1.0] * len(self.models)\n        s = sum(w)\n        out = None\n        for m, wi in zip(self.models, w):\n            logit = m(x)\n            out = logit * (wi / s) if out is None else out + logit * (wi / s)\n        return out\n\n    def eval(self):\n        for m in self.models:\n            m.eval()\n        return self\n\n    def to(self, device):\n        for m in self.models:\n            m.to(device)\n        return self\n\n\ndef load_checkpoint(path: str, spec: ModelSpec | str, n_classes: int,\n                    device: str | None = None) -> nn.Module:\n    device = device or (\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    ckpt = torch.load(path, map_location=\"cpu\", weights_only=False)\n    state = ckpt.get(\"ema\") or ckpt.get(\"model\") or ckpt\n    model = build(spec, n_classes, pretrained=False, verbose=False)\n    try:\n        missing, unexpected = model.load_state_dict(state, strict=False)\n    except RuntimeError as exc:\n        # ⚠️ strict=False 는 **키 불일치**만 봐줍니다. 텐서 **모양**이 다르면 터집니다.\n        #    거의 항상 원인은 하나 — 다른 백본의 가중치를 부은 것입니다.\n        #    실제로 effnetv2_s 체크포인트를 resnet50 껍데기에 붓다가 죽었습니다.\n        want = getattr(spec, \"key\", spec)\n        raise RuntimeError(\n            f\"\\n체크포인트를 '{want}' 에 못 넣었습니다 — **백본이 다릅니다.**\\n\"\n            f\"  파일: {path}\\n\\n\"\n            \"  체크포인트 폴더 이름에 어떤 백본으로 학습했는지 적혀 있습니다:\\n\"\n            \"      stage1_effnetv2_s_full_384_moderate  →  effnetv2_s\\n\"\n            \"      stage2_resnet50_m2.5_384_moderate    →  resnet50\\n\"\n            \"  `train.model_key_from_exp(<폴더이름>)` 이 그걸 읽어 줍니다.\\n\\n\"\n            f\"  원본 오류: {str(exc).splitlines()[0]}\"\n        ) from exc\n    if missing or unexpected:\n        warnings.warn(f\"state_dict 불일치 — missing={len(missing)}, unexpected={len(unexpected)}\")\n    return model.to(device).eval()\n\n\n# ──────────────────────────────────────────────────────────────\n# 임베딩 (마지막 직전 특징)\n# ──────────────────────────────────────────────────────────────\n@torch.no_grad()\ndef embed(model: nn.Module, loader, device: str = \"cuda\",\n          verbose: bool = True) -> np.ndarray:\n    \"\"\"분류 헤드 **직전**의 특징 벡터를 뽑습니다. 행 순서는 로더 그대로입니다.\n\n    왜 필요한가\n    -----------\n    STEP 18 에서 \"A4 는 결정 규칙이 아니라 **표현**의 문제\" 까지 왔습니다\n    (로짓 보정으로 prior 를 걷어내도 macro-F1 이 안 오름). 그 다음 질문은\n    **어떻게** 실패하느냐입니다:\n\n        · A4 가 특징 공간에서 **뭉쳐 있는데** 분류기가 못 쓰는 것  → 헤드·손실 문제\n        · A4 가 애초에 **흩어져 있는 것**                        → 특징·데이터 문제\n\n    앞이면 싸게 고쳐지고 뒤면 크롭·데이터를 손봐야 합니다. 로짓만으로는\n    구분이 안 되고, 임베딩을 봐야 갈립니다.\n\n    ⚠️ timm 모델마다 헤드 모양이 달라서 `forward_features` → `forward_head(\n    pre_logits=True)` 순서로 부릅니다. 그게 안 되는 모델은 헤드를 `Identity`\n    로 갈아 끼우고 한 번 통과시킵니다 (원본은 안 건드립니다 — 복원합니다).\n    \"\"\"\n    model = model.to(device).eval()\n    feats: list[np.ndarray] = []\n\n    def _pre_logits(x):\n        f = model.forward_features(x)\n        return model.forward_head(f, pre_logits=True)\n\n    use_head = True\n    try:                                    # 한 배치로 먼저 확인\n        probe = next(iter(loader))[0][:1].to(device)\n        _pre_logits(probe)\n    except Exception:\n        use_head = False\n\n    saved = None\n    if not use_head:\n        # ⚠️ 헤드 이름은 모델마다 다릅니다 — param_groups 와 **같은 목록**을 씁니다.\n        for name in (\"head\", \"fc\", \"classifier\", \"last_linear\"):\n            if hasattr(model, name):\n                saved = (name, getattr(model, name))\n                setattr(model, name, nn.Identity())\n                break\n        if saved is None:\n            raise RuntimeError(\n                \"임베딩을 못 뽑습니다 — forward_head(pre_logits=True) 도 안 되고 \"\n                f\"헤드 속성({('head','fc','classifier','last_linear')})도 없습니다.\")\n\n    try:\n        for batch in loader:\n            x = batch[0].to(device, non_blocking=True)\n            with torch.autocast(device_type=\"cuda\" if device.startswith(\"cuda\") else \"cpu\",\n                                enabled=device.startswith(\"cuda\")):\n                f = _pre_logits(x) if use_head else model(x)\n            if f.ndim > 2:                  # (B,C,H,W) 가 나오면 평균 풀링\n                f = f.mean(dim=tuple(range(2, f.ndim)))\n            feats.append(f.float().cpu().numpy())\n    finally:\n        if saved is not None:\n            setattr(model, saved[0], saved[1])\n\n    out = np.concatenate(feats, 0)\n    if verbose:\n        print(f\"[models] 임베딩 {out.shape}  (헤드 {'통과' if use_head else 'Identity 교체'})\")\n    return out\n", "src/detect.py": "\"\"\"병변 **검출** — 사람에게 안 묻고 모델이 네모를 찾습니다 (STEP 42).\n\n## 왜 이게 남았나\n\n네모를 받는 길이 **아홉 개** 닫혔습니다 (STEP 36~41). 전부 같은 곳에서\n막힙니다 — **사람은 병변이 어디에 얼마나 크게 있는지 못 알려줍니다**\n(크기 상관 −0.05 · 탭이 병변 안에 든 것 47.5%).\n\n그런데 **`bbox` 라벨이 36만 장** 있습니다. 사람에게 묻지 말고 **모델이 그 일을\n직접 배우게** 하는 정공법인데, 지금까지 시도조차 안 했습니다.\n\n⚠️ STEP 37 의 *\"모델이 제안\"* 과 다릅니다. 거기서는 **분류기를 창 탐지기로\n빌려 썼습니다** — 네모를 뽑도록 배운 적이 없는 모델이었고 6.7% 였습니다.\n여기서는 **그 일을 직접 배웁니다.**\n\n## 무엇을 배우나\n\n사진 한 장 → 네모 하나 `(x1, y1, x2, y2)` 0~1 정규화.\n\n⚠️ **한 장에 네모 하나**만 냅니다. 매니페스트의 `n_lesion` 은 라벨 항목 수라\n병변 하나당 2입니다 — **78% 가 병변 1개**이므로 단일 회귀로 시작합니다.\n여러 개인 22% 는 **가장 큰 것**을 정답으로 씁니다 (크롭을 걸 자리를 고르는\n것이 목적이므로).\n\n## 판정\n\n`experiments.DETECT_MIN_USABLE` (밴드 안 비율) 과\n`experiments.DETECT_MIN_COVERAGE_GAIN` (커버리지 이득) — **둘 다** 넘어야\n합니다. 네모가 좋아 보여도 **크롭이 걸리면 중심 오차가 치명적이 될 수**\n있습니다 (STEP 41).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport math\n\nimport torch\nimport torch.nn as nn\n\n\nclass BoxHead(nn.Module):\n    \"\"\"백본 + 네모 회귀 헤드.\n\n    ★ **중심·크기를 따로, 로그 공간에서** 냅니다:\n\n        cx, cy   sigmoid  → 0~1\n        log w, log h      → exp 해서 크기\n\n    왜 로그인가 — 병변 크기가 화면의 **4.6% ~ 62.5%** 로 10배 넘게 흩어져\n    있습니다. 선형으로 회귀하면 **큰 것에 손실이 쏠려** 작은 병변을 통째로\n    놓칩니다 (STEP 17 에서 크기와 놓침의 관계를 이미 봤습니다).\n    \"\"\"\n\n    def __init__(self, backbone: str = \"effnetv2_s\", pretrained: bool = True,\n                 img_size: int = 384):\n        super().__init__()\n        from src import models\n\n        self.body = models.build(backbone, n_classes=0, pretrained=pretrained,\n                                 img_size=img_size, verbose=False)\n        feat = getattr(self.body, \"num_features\", None)\n        if feat is None:                      # timm 이 0-class 를 안 주는 경우\n            with torch.no_grad():\n                feat = self.body(torch.zeros(1, 3, img_size, img_size)).shape[-1]\n        self.head = nn.Linear(int(feat), 4)\n        # 초기값을 **화면 가운데 · 병변 중앙값 크기** 로 둡니다. 안 그러면 첫\n        # 에폭이 그걸 찾는 데 쓰이고, 작은 데이터에서는 못 찾고 끝납니다.\n        nn.init.zeros_(self.head.weight)\n        with torch.no_grad():\n            self.head.bias.copy_(torch.tensor(\n                [0.0, 0.0, math.log(0.215), math.log(0.215)]))\n\n    def forward(self, x):\n        z = self.head(self.body(x))\n        cx, cy = torch.sigmoid(z[:, 0]), torch.sigmoid(z[:, 1])\n        w = torch.exp(z[:, 2].clamp(-4.0, 0.0))     # 1.8% ~ 100%\n        h = torch.exp(z[:, 3].clamp(-4.0, 0.0))\n        return torch.stack([cx, cy, w, h], 1)\n\n\ndef to_xyxy(cwh: torch.Tensor) -> torch.Tensor:\n    \"\"\"(cx, cy, w, h) → (x1, y1, x2, y2). 자르지 않습니다 — 손실은 원본에서.\"\"\"\n    cx, cy, w, h = cwh.unbind(1)\n    return torch.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], 1)\n\n\ndef to_cwh(xyxy: torch.Tensor) -> torch.Tensor:\n    x1, y1, x2, y2 = xyxy.unbind(1)\n    return torch.stack([(x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1], 1)\n\n\ndef box_loss(pred_cwh: torch.Tensor, true_xyxy: torch.Tensor) -> torch.Tensor:\n    \"\"\"중심은 선형, 크기는 **로그**에서 잽니다.\n\n    ⚠️ 크기를 선형으로 재면 큰 병변이 손실을 지배합니다. 우리가 정작 필요한\n    건 **배율이 밴드 안에 드는 것**이고, 그건 **비율**의 문제입니다 —\n    `usable_range()` 밴드가 0.71~1.67**배** 인 것과 같은 이유입니다.\n    \"\"\"\n    t = to_cwh(true_xyxy)\n    ctr = nn.functional.smooth_l1_loss(pred_cwh[:, :2], t[:, :2], beta=0.05)\n    eps = 1e-6\n    size = nn.functional.smooth_l1_loss(\n        torch.log(pred_cwh[:, 2:].clamp_min(eps)),\n        torch.log(t[:, 2:].clamp_min(eps)), beta=0.2)\n    return ctr + size\n\n\ndef band_report(pred_xyxy, true_xyxy) -> dict:\n    \"\"\"제안 네모가 **촬영 밴드** 안에 드는 비율. 사람·창탐지기와 같은 잣대.\n\n    밴드는 `config.ZOOM_ALLOW` / `ZOOM_CENTER_MAX` 에서 끌어 씁니다 — 여기\n    베껴 적으면 출처가 바뀌어도 아무 일이 안 일어납니다 (촬영 밴드에서 당한 것).\n    \"\"\"\n    import numpy as np\n\n    from src.config import ZOOM_ALLOW, ZOOM_CENTER_MAX\n\n    p = np.asarray(pred_xyxy, dtype=float)\n    t = np.asarray(true_xyxy, dtype=float)\n    p_long = np.maximum(p[:, 2] - p[:, 0], p[:, 3] - p[:, 1])\n    t_long = np.maximum(t[:, 2] - t[:, 0], t[:, 3] - t[:, 1])\n    ok = t_long > 0\n    ratio = np.where(ok, p_long / np.maximum(t_long, 1e-9), np.nan)\n    off = np.maximum(\n        np.abs((p[:, 0] + p[:, 2]) / 2 - (t[:, 0] + t[:, 2]) / 2),\n        np.abs((p[:, 1] + p[:, 3]) / 2 - (t[:, 1] + t[:, 3]) / 2))\n    # 네모를 r 배로 그리면 환산 줌은 1/r → 밴드를 뒤집습니다\n    lo, hi = 1 / ZOOM_ALLOW[1], 1 / ZOOM_ALLOW[0]\n    in_size = (ratio >= lo) & (ratio <= hi)\n    in_pos = off <= ZOOM_CENTER_MAX\n    rep = {\"n\": int(ok.sum()),\n           \"ratio_median\": float(np.nanmedian(ratio)),\n           \"off_median\": float(np.median(off)),\n           \"in_size\": float(np.nanmean(in_size)),\n           \"in_pos\": float(np.mean(in_pos)),\n           \"both\": float(np.nanmean(in_size & in_pos)),\n           \"band\": [lo, hi], \"center_max\": float(ZOOM_CENTER_MAX)}\n    rep[\"baseline\"] = _baseline(t, lo, hi)\n    return rep\n\n\ndef _baseline(true_xyxy, lo: float, hi: float) -> dict:\n    \"\"\"★ **아무것도 안 배운 네모** — 화면 한가운데 · 크기는 실측 병변 중앙값.\n\n    왜 이걸 같이 내나 — STEP 42 에서 검출기 **배율 75.0%** 를 보고 \"배율은\n    풀렸다\" 고 읽을 뻔했습니다. 같은 val 에서 이 하한선이 **72.3%** 입니다.\n    번 것은 **+2.7%p** 뿐이고, 검출기가 실제로 배운 것은 **위치**였습니다\n    (10.1% → 38.7%).\n\n    ⚠️ 이건 STEP 29 와 **같은 실수**입니다 — 거기서는 계열 정확도 0.84 를\n    앞세웠는데 최빈 묶음만 말해도 0.671 이었습니다. 작업 규칙 1.\n\n    관문으로 쓰지 않고 **찍기만** 합니다 (`over_triage` 와 같은 처방).\n    \"\"\"\n    import numpy as np\n\n    from src.config import ZOOM_CENTER_MAX\n    from src.experiments import FIXEDSCALE_LESION_FRAC\n\n    t = np.asarray(true_xyxy, dtype=float)\n    t_long = np.maximum(t[:, 2] - t[:, 0], t[:, 3] - t[:, 1])\n    ratio = FIXEDSCALE_LESION_FRAC / np.maximum(t_long, 1e-9)\n    off = np.maximum(np.abs((t[:, 0] + t[:, 2]) / 2 - 0.5),\n                     np.abs((t[:, 1] + t[:, 3]) / 2 - 0.5))\n    in_size = (ratio >= lo) & (ratio <= hi)\n    in_pos = off <= ZOOM_CENTER_MAX\n    return {\"in_size\": float(in_size.mean()), \"in_pos\": float(in_pos.mean()),\n            \"both\": float((in_size & in_pos).mean()),\n            \"off_median\": float(np.median(off)), \"size_frac\": FIXEDSCALE_LESION_FRAC}\n\n\ndef print_report(rep: dict, *, human_both: float = 0.05,\n                 window_both: float = 0.067) -> bool:\n    \"\"\"판정을 찍고 통과 여부를 돌려줍니다. 기준은 `experiments` 에 있습니다.\"\"\"\n    from src.experiments import DETECT_MIN_USABLE\n\n    print(f\"\\n■ 검출기 제안 네모 (n={rep['n']:,})\")\n    print(f\"    크기비 중앙값 {rep['ratio_median']:.2f}배   \"\n          f\"중심 어긋남 중앙값 {rep['off_median']:.3f}\")\n    print(f\"\\n■ 밴드 안 (허용 {rep['band'][0]:.2f}~{rep['band'][1]:.2f}배 · \"\n          f\"중심 {rep['center_max']})\")\n    print(f\"    배율     {rep['in_size']:.1%}\")\n    print(f\"    위치     {rep['in_pos']:.1%}\")\n    print(f\"    둘 다    {rep['both']:.1%}      \"\n          f\"(사람 {human_both:.1%} · 창탐지기 {window_both:.1%})\")\n    b = rep.get(\"baseline\")\n    if b:\n        # ★ 하한선을 **같이** 찍습니다 — 관문으로는 안 씁니다 (작업 규칙 1·2).\n        print()\n        print(f\"■ 하한선 — 화면 한가운데 · 크기 {b['size_frac']:.3f} 고정 \"\n              f\"(아무것도 안 배운 네모)\")\n        print(f\"    배율     {b['in_size']:.1%}    ← 검출기가 번 것 \"\n              f\"{rep['in_size'] - b['in_size']:+.1%}\")\n        print(f\"    위치     {b['in_pos']:.1%}    ← 검출기가 번 것 \"\n              f\"{rep['in_pos'] - b['in_pos']:+.1%}\")\n        print(f\"    둘 다    {b['both']:.1%}    ← 검출기가 번 것 \"\n              f\"{rep['both'] - b['both']:+.1%}\")\n        print(\"    ⚠️ 하한선을 안 보면 '배율은 풀렸다' 로 읽게 됩니다 (STEP 29 와 같은 실수).\")\n    ok = rep[\"both\"] >= DETECT_MIN_USABLE\n    print(f\"\\n■ 사전등록 문턱 {DETECT_MIN_USABLE:.0%}\")\n    print(\"  ⭕ **통과** — 다음은 이 네모로 자른 크롭의 **커버리지**입니다.\"\n          if ok else \"  ❌ 미달\")\n    if ok:\n        print(\"  ⚠️ 밴드만 통과한 것입니다. **커버리지가 안 오르면 채택 안 합니다** \"\n              \"(STEP 41 에서 배운 것).\")\n    return ok\n", "src/experiments.py": "\"\"\"한 번의 실행으로 결론이 나야 하는 비교 실험들.\n\n⚠️ **왜 노트북이 아니라 여기 있나**\n\n노트북 셀은 `git pull` 로 갱신되지 않습니다. 그래서 이 프로젝트는\n\"바뀔 수 있는 로직\" 을 전부 `src/` 에 둡니다 (`src/gates.py` 의 설명 참고).\n\n여기 있는 건 **학습을 여러 번 돌려 비교하는** 코드입니다. 노트북에 복붙하면\n같은 30줄이 해상도마다 반복되고, 고칠 일이 생기면 사용자가 .ipynb 를 다시\n받아야 합니다. 함수로 두면 한 줄로 부르고 `git pull` 로 고쳐집니다.\n\n핵심 원칙 — **판정은 검증 점수가 아니라 견고성으로 합니다.**\n    검증 macro-F1 은 \"정답 박스로 잘라준 사진\" 점수입니다.\n    실제 보호자 사진에는 박스가 없고 배율이 제각각입니다.\n    그래서 배율 교란 하락폭을 같이 재고, 그걸로 채택을 결정합니다.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport gc\nimport time\nfrom typing import Any\n\nimport torch\n\n# 교란 검사(배율/위치/화질) 하락의 잡음 폭.\n#\n# ⚠️ **짝지은 비교 전용입니다** — 두 조건을 *같은 실행·같은 표본*으로 나란히\n#    돌렸을 때만 이 값을 쓰세요 (03c 방식). 그러면 표본 추출 잡음이 상쇄됩니다.\n#\n# **실행이 다르면 ±8%p 로 보세요.** 2단계 설정을 하나도 안 바꾸고 네 번 쟀더니\n# 배율 하락이 이렇게 나왔습니다:\n#     STEP 4D 16.0%  ·  STEP 5 21.6%(n=2000) / 16.8%(n=3000)  ·  STEP 7 23.8%\n# 폭이 7.8%p 입니다. 처음엔 ±3%p, STEP 5 에서 ±5%p 로 올렸는데 그것도 낙관적이었습니다.\nNOISE_PP = 0.03\n\n# 배율 하락이 이 아래면 실사용에 내놓을 만합니다.\nDROP_WANT = 0.15\n\n\ndef train_and_measure(\n    view,\n    *,\n    stage: int,\n    img_size: int,\n    crop_tag: str,\n    device: str,\n    epochs: int,\n    model_name: str = \"resnet50\",\n    finetune: str = \"moderate\",\n    aug: str = \"default\",\n    fold: int = 0,\n    subset_frac: float = 1.0,\n    n_robust: int = 3000,\n    measure_robust: bool = True,\n    measure_blur: bool = False,\n    balance: str | None = None,\n    hair_alpha: float = 1.0,\n    lr: float | None = None,\n    backbone_lr_mult: float | None = None,\n    warmup_epochs: int | None = None,\n    verbose: bool = True,\n) -> dict[str, Any]:\n    \"\"\"한 설정으로 학습하고 **점수와 견고성을 함께** 돌려줍니다.\n\n    stage=1 이면 정상/이상(이진), stage=2 면 병변 6종입니다.\n\n    반환하는 dict 는 `resolution_report()` / `augmentation_report()` 가 그대로 먹습니다.\n    모델은 다 쓰고 나면 버립니다 (해상도를 올리면 VRAM 이 빠듯합니다).\n\n    Args:\n        balance: 불균형 대응 전략을 덮어씁니다. 기본은 1단계 `\"none\"` /\n            2단계 `\"class_weight\"`. `\"hair_weighted\"` 를 주면 **털처럼 가는 선이\n            많은 정상 사진**을 더 자주 뽑습니다 (`data.hair_sampler`).\n        hair_alpha: `balance=\"hair_weighted\"` 일 때의 세기. `0` 이면 균등.\n        lr / backbone_lr_mult / warmup_epochs: 학습률 축을 비교할 때 덮어씁니다.\n            ★ 셋 중 하나라도 주면 **실험 이름에 붙습니다.** 안 붙이면 네 판이\n            같은 폴더를 쓰고, `train.fit` 이 뒤의 세 판을 \"이미 끝난 학습\" 으로\n            조용히 건너뜁니다 — 해상도·샘플러에서 이미 두 번 당한 실패입니다.\n    \"\"\"\n    from src import data, evaluate, models, robust, split, stages, train\n    from src.config import CLASSES, CLASSES_STAGE1, CFG, with_aug, with_finetune\n\n    classes = CLASSES_STAGE1 if stage == 1 else CLASSES\n    cfg = with_aug(\n        with_finetune(\n            CFG(model_name=model_name, img_size=img_size, epochs=epochs,\n                # 1단계는 정상:이상이 5:5 라 가중치가 필요 없습니다.\n                balance_strategy=balance or (\"none\" if stage == 1 else \"class_weight\"),\n                hair_alpha=hair_alpha,\n                monitor=\"macro_f1\",\n                # ⚠️ 해상도를 이름에 넣습니다 — 안 넣으면 224 체크포인트를 384 학습이\n                #    \"이미 끝난 학습\" 으로 착각하고 건너뜁니다.\n                #    (증강 프리셋은 with_aug 가 이름 뒤에 자동으로 붙입니다)\n                # ⚠️ 샘플러가 다르면 **이름도 달라야** 합니다. 안 그러면\n                #    train.fit 이 기준선 체크포인트를 보고 \"이미 끝난 학습\" 으로\n                #    착각해 처치를 통째로 건너뜁니다 — 해상도에서 이미 당한 실패입니다.\n                exp_name=f\"stage{stage}_{model_name}_{crop_tag}_{img_size}\"\n                         + (f\"_hair{hair_alpha:g}\"\n                            if balance == \"hair_weighted\" else \"\")),\n            finetune),\n        aug)\n\n    # ── 학습률 축 (주었을 때만) ──────────────────────────────────\n    # ⚠️ 이름을 먼저 붙이고 값을 넣습니다. 순서가 바뀌어도 결과는 같지만,\n    #    **이름을 안 붙이면** 네 판이 한 폴더를 공유해 뒤의 세 판이 통째로\n    #    건너뛰어집니다. 그래도 표는 그럴듯하게 나오고 아무 에러도 안 납니다.\n    _lr_tag = \"\"\n    if lr is not None:\n        _lr_tag += f\"_lr{lr:g}\"\n    if backbone_lr_mult is not None:\n        _lr_tag += f\"_bb{backbone_lr_mult:g}\"\n    if warmup_epochs is not None:\n        _lr_tag += f\"_wu{warmup_epochs}\"\n    if _lr_tag:\n        over = {\"exp_name\": cfg.exp_name + _lr_tag}\n        if lr is not None:\n            over[\"lr\"] = lr\n        if backbone_lr_mult is not None:\n            over[\"backbone_lr_mult\"] = backbone_lr_mult\n        if warmup_epochs is not None:\n            over[\"warmup_epochs\"] = warmup_epochs\n        cfg = CFG.from_dict({**cfg.to_dict(), **over})\n\n    tr, va = split.get_fold(view, fold)\n\n    # ── 빠른 스윕용 부분 학습 ────────────────────────────────────\n    # 멘토 피드백 6번: \"초반에는 최대한 빠르게 자동으로 시도\"\n    # ⚠️ **학습셋만** 줄입니다. 검증셋을 줄이면 프리셋 간 점수 비교가 흔들립니다.\n    #    클래스별로 같은 비율을 뽑아 불균형 구조를 유지합니다.\n    #    ⚠️ 서브셋 순위가 풀 데이터 순위와 같다는 보장은 없습니다.\n    #       **후보를 줄이는 용도**이고, 확정은 반드시 풀 스케일로 다시 합니다.\n    if subset_frac < 1.0:\n        n_before = len(tr)\n        # groupby(...).sample 은 그룹별 비율 표본을 바로 줍니다.\n        # (groupby.apply 는 pandas 2.2+ 에서 그룹 컬럼 처리로 경고가 납니다)\n        tr = tr.loc[tr.groupby(\"label\").sample(\n            frac=subset_frac, random_state=fold).index].reset_index(drop=True)\n        if verbose:\n            print(f\"\\n  [스윕] 학습셋 {n_before:,} → {len(tr):,}장 \"\n                  f\"({subset_frac:.0%}) · 검증셋 {len(va):,}장은 그대로\")\n\n    if verbose:\n        print(f\"\\n{'━' * 66}\")\n        print(f\"  {stage}단계 @ {img_size}px · 증강 '{aug}'\")\n        print(f\"  크롭 '{crop_tag}' / {epochs}에폭 / 배치 {cfg.resolved_batch_size()}\")\n        print(f\"{'━' * 66}\")\n        print(f\"  train {len(tr):,} / val {len(va):,}\")\n\n    # ⚠️ img_size 를 넘깁니다. CNN 은 해상도가 자유로워 무시되지만, ViT 계열은\n    #    위치 임베딩 크기가 고정이라 안 넘기면 **학습 도중에** shape 오류가 납니다.\n    #    (models.build 는 필요할 때만 timm 에 전달합니다)\n    model = models.build(model_name, n_classes=len(classes),\n                         pretrained=True, drop_rate=cfg.drop_rate,\n                         img_size=img_size)\n    dl_tr, dl_va, ds_tr, _ = data.build_loaders(tr, va, cfg, model=model, classes=classes)\n\n    t0 = time.time()\n    train.print_status(cfg.exp_name)\n    res = train.fit(model, dl_tr, dl_va, cfg, ds_train=ds_tr)\n    minutes = (time.time() - t0) / 60\n\n    logits, y = train.cached_logits(model, dl_va, key=\"val\", exp=cfg.exp_name,\n                                    n_cls=len(classes), device=device,\n                                    tta_hflip=cfg.tta_hflip)\n\n    out: dict[str, Any] = {\n        \"stage\": stage, \"img_size\": img_size, \"crop_tag\": crop_tag, \"aug\": aug,\n        # ★ 샘플러 설정 — 기준선/처치를 표에서 갈라 보려면 이게 있어야 합니다\n        \"balance\": cfg.balance_strategy,\n        \"hair_alpha\": hair_alpha if cfg.balance_strategy == \"hair_weighted\" else 0.0,\n        # ★ 어떤 백본이었는지 남깁니다 — 2×2 비교(stage1_report)가 이걸로 표를 만듭니다\n        \"model_name\": model_name,\n        \"subset_frac\": subset_frac, \"n_train\": len(tr),\n        \"exp_name\": cfg.exp_name, \"epochs\": epochs, \"minutes\": minutes,\n        \"batch_size\": cfg.resolved_batch_size(),\n        # ★ 학습률 축 — 표에서 판을 갈라 보려면 이게 있어야 합니다\n        \"lr\": cfg.lr, \"backbone_lr_mult\": cfg.backbone_lr_mult,\n        \"backbone_lr\": cfg.lr * cfg.backbone_lr_mult,\n        \"warmup_epochs\": cfg.warmup_epochs,\n        \"best_epoch\": res.best_epoch, \"n_epochs\": len(res.history),\n        # 마지막 에폭이 최고면 아직 덜 학습된 것입니다 (에폭을 더 줘야 합니다)\n        \"converged\": res.best_epoch < len(res.history) - 2,\n    }\n\n    if stage == 1:\n        rep = evaluate.binary_report(stages.stage1_scores(logits),\n                                     stages.binary_targets(y),\n                                     target_recall=cfg.target_recall_stage1)\n        out.update(auroc=rep[\"auroc\"], threshold=rep[\"threshold\"],\n                   precision=rep[\"precision_at_target\"], score=rep[\"auroc\"],\n                   score_name=\"AUROC\")\n    else:\n        rep = evaluate.full_report(logits, y, classes, show=False)\n        out.update(macro_f1=rep.metrics[\"macro_f1\"],\n                   a6_recall=rep.metrics[\"per_class\"][\"recall\"][classes.index(\"A6\")],\n                   score=rep.metrics[\"macro_f1\"], score_name=\"macro-F1\")\n        out[\"report\"] = rep\n\n    if measure_robust:\n        r = robust.scale_stress(model, va, cfg, classes, n=n_robust, device=device)\n        s = r.get(\"_summary\", {})\n        # 키 이름은 robust.py 가 정합니다: baseline / worst / rel_drop\n        out.update(scale_drop=s.get(\"rel_drop\"), scale_worst=s.get(\"worst\"),\n                   scale_worst_at=s.get(\"worst_condition\"))\n\n    # ⚠️ 화질 교란은 **1단계 전용**에 가깝습니다. 정상 사진이 계통적으로 흐려서\n    #    (선명도 50 vs 274) 모델이 화질로 맞힐 수 있는데, val 점수만 보면\n    #    그게 안 보입니다. photometric 증강 전후를 이 값으로 비교합니다.\n    if measure_blur:\n        b = robust.blur_stress(model, va, cfg, classes, n=n_robust, device=device)\n        s = b.get(\"_summary\", {})\n        out.update(blur_drop=s.get(\"rel_drop\"), blur_worst=s.get(\"worst\"),\n                   blur_worst_at=s.get(\"worst_condition\"))\n\n    del model, dl_tr, dl_va, ds_tr\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n    return out\n\n\ndef _compare(runs: list[dict], key: str, base_val, title: str,\n             fmt=str) -> dict[str, Any]:\n    \"\"\"`key` 만 다른 실행들을 견고성 기준으로 비교합니다.\n\n    해상도 비교와 증강 비교가 같은 판정 규칙을 쓰도록 한 곳에 모았습니다.\n    \"\"\"\n    runs = [r for r in runs if r]\n    if not runs:\n        return {}\n\n    by_stage: dict[int, list[dict]] = {}\n    for r in runs:\n        by_stage.setdefault(r[\"stage\"], []).append(r)\n\n    verdicts: dict[str, Any] = {}\n    for stage in sorted(by_stage):\n        rs = by_stage[stage]\n        name = rs[0][\"score_name\"]\n        print(f\"\\n{'=' * 68}\\n {stage}단계 — {title}\\n{'=' * 68}\")\n        print(f\"  {'설정':<16}{name:>10}{'배율하락':>10}{'최악조건':>10}{'분':>7}   수렴\")\n        for r in rs:\n            drop = r.get(\"scale_drop\")\n            ds = f\"{drop:>9.1%}\" if drop is not None else f\"{'—':>10}\"\n            print(f\"  {fmt(r[key]):<16}{r['score']:>10.4f}{ds}\"\n                  f\"{str(r.get('scale_worst_at', '?')):>10}{r['minutes']:>7.0f}   \"\n                  f\"{'수렴' if r['converged'] else '더 필요'}\")\n\n        base = next((r for r in rs if r[key] == base_val), rs[0])\n        others = [r for r in rs if r is not base]\n        v: dict[str, Any] = {\"baseline\": base}\n        if not others:\n            print(\"\\n  (비교 대상이 없어 판정을 생략합니다)\")\n            verdicts[f\"stage{stage}\"] = v\n            continue\n\n        # 견고성이 가장 좋은 것 = 하락이 가장 작은 것. 점수가 아니라 이걸로 고릅니다.\n        scored = [r for r in others if r.get(\"scale_drop\") is not None]\n        if not scored or base.get(\"scale_drop\") is None:\n            v[\"verdict\"] = \"unmeasured\"\n            print(\"\\n  ⚠️ 배율 하락을 안 재서 판정할 수 없습니다.\")\n            verdicts[f\"stage{stage}\"] = v\n            continue\n\n        best = min(scored, key=lambda r: r[\"scale_drop\"])\n        bd, hd = base[\"scale_drop\"], best[\"scale_drop\"]\n        d_drop, d_score = hd - bd, best[\"score\"] - base[\"score\"]\n        v.update(best=best, drop_delta=d_drop, score_delta=d_score)\n\n        print(f\"\\n  가장 견고한 설정: {fmt(best[key])}\")\n        print(f\"  {name}  {base['score']:.4f} → {best['score']:.4f}  ({d_score:+.4f})\")\n        print(f\"  배율 하락  {bd:.1%} → {hd:.1%}  ({d_drop:+.1%})\")\n\n        if d_drop < -NOISE_PP and hd <= DROP_WANT:\n            v[\"verdict\"] = \"adopt\"\n            print(f\"\\n  ✅ 채택 — 하락이 잡음(±{NOISE_PP:.0%})보다 크게 줄었고 \"\n                  f\"목표({DROP_WANT:.0%}) 안에 들어왔습니다.\")\n        elif d_drop < -NOISE_PP:\n            v[\"verdict\"] = \"improved\"\n            print(f\"\\n  🤔 개선됐지만 아직 {hd:.0%} 입니다 (목표 {DROP_WANT:.0%}).\")\n            print(\"     방향은 맞습니다 — 더 밀어붙이거나 배포 설계로 보완하세요.\")\n        else:\n            v[\"verdict\"] = \"no_effect\"\n            print(f\"\\n  ❌ 하락폭이 그대로입니다 (잡음 ±{NOISE_PP:.0%} 안).\")\n            print(\"     이 축으로는 안 잡힙니다. 남은 건 배포 설계입니다:\")\n            print('     \"병변이 화면 절반 이상 차지하게 찍어주세요\" 로 입력을 제한하고,')\n            print(\"     full 크롭 점수를 정직한 숫자로 보고하는 쪽.\")\n\n        if not best[\"converged\"]:\n            print(f\"\\n  📈 {fmt(best[key])} 는 마지막 에폭이 최고였습니다 — 덜 학습됐습니다. \"\n                  f\"에폭을 {int(best['epochs'] * 1.6)} 로 올리면 더 오를 수 있습니다.\")\n        verdicts[f\"stage{stage}\"] = v\n\n    return verdicts\n\n\ndef resolution_report(runs: list[dict], *, baseline_size: int = 224) -> dict[str, Any]:\n    \"\"\"해상도 비교. 판정 기준은 `_compare` 에 있습니다.\"\"\"\n    return _compare(runs, \"img_size\", baseline_size, \"해상도 비교\",\n                    fmt=lambda v: f\"{v}px\")\n\n\ndef crop_report(runs: list[dict], *, baseline: str = \"m1.5\") -> dict[str, Any]:\n    \"\"\"크롭 태그 비교 (m1.5 vs m2.5 vs f320 …).\n\n    ⚠️ 크롭은 **다른 축과 성격이 다릅니다.** 증강이나 백본을 바꾸면 모델만\n    바뀌지만, 크롭을 바꾸면 **입력 자체가 바뀝니다.** 그래서 이걸 먼저\n    확정해야 뒤(촬영 가이드·임계값·백본 순위)가 흔들리지 않습니다.\n\n    비교 지점은 \"병변을 크게 보되 흐리게(m1.5)\" vs \"작게 보되 선명하게(m2.5)\"\n    입니다. 크롭 창이 좁으면 384 로 늘릴 때 없는 픽셀을 만들어냅니다 —\n    실측으로 A1 은 m1.5 에서 2.6배 확대, m2.5 에서 1.6배입니다.\n    \"\"\"\n    return _compare(runs, \"crop_tag\", baseline, \"크롭 비교\", fmt=str)\n\n\ndef augmentation_report(runs: list[dict], *, baseline: str = \"default\") -> dict[str, Any]:\n    \"\"\"증강 프리셋 비교.\n\n    ⚠️ **점수가 아니라 배율 하락으로 고릅니다.** 검증 macro-F1 은 \"정답 박스로\n    잘라준 사진\" 점수라서, 증강을 세게 걸면 대개 조금 내려갑니다. 그래도\n    하락폭이 크게 줄면 배포에는 그쪽이 낫습니다.\n    \"\"\"\n    return _compare(runs, \"aug\", baseline, \"증강 비교\", fmt=str)\n\n\n# ──────────────────────────────────────────────────────────────\n# 학습률 축 (STEP 17) — \"best 가 0에폭\" 을 설명할 수 있나\n# ──────────────────────────────────────────────────────────────\n# STEP 16 에서 2단계(convnextv2_base, 89M)의 best 가 **0에폭**이었습니다.\n# 14에폭까지 돌려도 못 넘었고, 그 사이 train loss 가 **올라갔습니다**\n# (1.0292 → 1.2321, warmup 2에폭 구간). 설명이 둘입니다:\n#\n#   ① warmup 이라 정상이다 — 학습률이 꼭대기를 찍을 때 흔들렸다가 회복 중\n#   ② 학습률이 너무 높다   — 백본 9e-5(3e-4 × 0.3) 를 7,569스텝/에폭 먹여\n#                            사전학습 표현이 흐트러졌고 끝내 회복 못 함\n#\n# ②가 맞으면 지금 macro-F1 0.599 는 천장이 아니고, **A4 recall 0.264 와\n# 배율 하락 26.8% 도 \"덜 배운 모델에서 잰 값\"** 이 됩니다. STEP 9→12 에서\n# 이미 같은 일을 당했습니다 (미수렴 기준선이 교란 검사를 왜곡함).\n#\n# ⚠️ **판정 기준을 여기 못 박아 둡니다** (규칙 2). 결과를 보고 기준을 고르면\n#    무슨 숫자가 나와도 성공담이 됩니다.\nLR_MIN_BEST_EPOCH = 3      # ← 1차 기준. 점수보다 이게 먼저입니다\nMACRO_F1_NOISE = 0.02      # 같은 설정 두 실행: 0.4862 vs 0.5024 (STEP 12·13)\n\n\ndef lr_report(runs: list[dict], *, baseline_lr: float = 3e-4,\n              baseline_mult: float = 0.3) -> dict[str, Any]:\n    \"\"\"학습률 비교 — **1차 기준은 점수가 아니라 `best_epoch` 입니다.**\n\n    묻는 것은 \"어느 판이 제일 높나\" 가 아니라 **\"학습이 되긴 하나\"** 입니다.\n    best 가 0에폭이면 그 판도 같은 병에 걸린 것이고, 점수가 조금 높아도\n    설명이 안 됩니다.\n\n    판정 (실험 전 고정):\n      1차 — `best_epoch >= LR_MIN_BEST_EPOCH` (=3)\n      2차 — macro-F1 이 기준선 대비 `+MACRO_F1_NOISE`(0.02) 이상\n      둘 다 만족하는 판이 없으면 → **학습률 축을 닫습니다.**\n      \"0에폭이 진짜 최고\" 도 결론이고, 그러면 원인은 다른 데 있습니다.\n    \"\"\"\n    runs = [r for r in runs if r]\n    if not runs:\n        return {}\n\n    def key(r):\n        return (r.get(\"lr\"), r.get(\"backbone_lr_mult\"), r.get(\"warmup_epochs\"))\n\n    base = next((r for r in runs\n                 if r.get(\"lr\") == baseline_lr\n                 and r.get(\"backbone_lr_mult\") == baseline_mult), runs[0])\n\n    print(f\"\\n{'=' * 74}\\n 2단계 학습률 — 1차 기준은 best_epoch >= \"\n          f\"{LR_MIN_BEST_EPOCH}\\n{'=' * 74}\")\n    print(f\"  {'헤드lr':>8}{'×배수':>7}{'백본lr':>10}{'warmup':>8}\"\n          f\"{'macro-F1':>10}{'best':>6}{'/에폭':>6}{'배율하락':>9}{'분':>6}\")\n    for r in runs:\n        drop = r.get(\"scale_drop\")\n        ds = f\"{drop:>8.1%}\" if drop is not None else f\"{'—':>9}\"\n        mark = \" ★\" if r is base else \"\"\n        print(f\"  {r.get('lr', 0):>8.0e}{r.get('backbone_lr_mult', 0):>7.2f}\"\n              f\"{r.get('backbone_lr', 0):>10.0e}{r.get('warmup_epochs', 0):>8}\"\n              f\"{r['score']:>10.4f}{r.get('best_epoch', -1):>6}\"\n              f\"{r.get('n_epochs', 0):>6}{ds}{r['minutes']:>6.0f}{mark}\")\n\n    # ★ **먼저: 기준선에서 현상이 재현됐는가.**\n    #   \"best 가 0에폭\" 은 한 에폭의 스텝 수가 많아서 생긴 현상입니다.\n    #   서브셋으로 줄이면 스텝/에폭도 줄어 **현상 자체가 안 나타날 수 있습니다.**\n    #   그 상태에서 B·C·D 가 좋아 보이면 \"학습률이 고쳤다\" 가 아니라\n    #   \"원래 문제가 없었다\" 입니다 — 아무것도 못 배우고 시간만 씁니다.\n    if base.get(\"best_epoch\", -1) >= LR_MIN_BEST_EPOCH:\n        print(f\"\\n  🚨 **기준선 A 의 best_epoch 이 이미 \"\n              f\"{base['best_epoch']} 입니다 (≥ {LR_MIN_BEST_EPOCH}).**\")\n        print(\"     STEP 16 에서 보려던 '0에폭 best' 가 이 규모에서는 재현되지\")\n        print(\"     않았습니다. 다른 판이 좋아 보여도 학습률 덕분이라고 말할 수\")\n        print(\"     없습니다 — **이 실험은 무효입니다.**\")\n        print(f\"     스텝/에폭을 STEP 16(7,569)에 가깝게 올려서 다시 하세요\"\n              f\" — 지금 학습 {base.get('n_train', 0):,}장.\")\n        return {\"verdict\": \"무효 — 기준선에서 현상 미재현\",\n                \"baseline\": key(base), \"trained\": [], \"winner\": None,\n                \"min_best_epoch\": LR_MIN_BEST_EPOCH, \"noise\": MACRO_F1_NOISE}\n\n    trained = [r for r in runs if r.get(\"best_epoch\", -1) >= LR_MIN_BEST_EPOCH]\n    better = [r for r in trained if r[\"score\"] >= base[\"score\"] + MACRO_F1_NOISE]\n\n    print(f\"\\n  기준선: macro-F1 {base['score']:.4f} (best epoch \"\n          f\"{base.get('best_epoch', -1)}) — 현상 재현됨 ✅\")\n    print(f\"  1차 통과 (best_epoch >= {LR_MIN_BEST_EPOCH}): \"\n          f\"{len(trained)}/{len(runs)}판\")\n\n    if not trained:\n        verdict = \"축 닫힘\"\n        print(\"\\n  ❌ **어느 판도 0~2에폭을 못 벗어났습니다.**\")\n        print(\"     학습률 축을 닫습니다 — 0에폭 best 는 학습률 탓이 아닙니다.\")\n        print(\"     다음 용의자: 데이터(라벨 노이즈) · 백본 크기 · 증강 강도\")\n    elif not better:\n        verdict = \"학습은 되나 점수는 그대로\"\n        print(f\"\\n  ⚠️ 학습은 되는데 점수가 잡음(±{MACRO_F1_NOISE}) 안입니다.\")\n        print(\"     '0에폭 best' 는 고쳤지만 성능은 안 올랐습니다 —\")\n        print(\"     지금 0.599 가 데이터 천장이라는 쪽으로 기웁니다.\")\n        for r in trained:\n            print(f\"     · lr {r['lr']:.0e} ×{r['backbone_lr_mult']:.2f}: \"\n                  f\"{r['score']:.4f} ({r['score'] - base['score']:+.4f}), \"\n                  f\"best epoch {r['best_epoch']}\")\n    else:\n        win = max(better, key=lambda r: r[\"score\"])\n        verdict = \"채택\"\n        print(f\"\\n  ✅ **채택: 헤드 lr {win['lr']:.0e} × 배수 \"\n              f\"{win['backbone_lr_mult']:.2f} (백본 {win['backbone_lr']:.0e})**\")\n        print(f\"     macro-F1 {base['score']:.4f} → {win['score']:.4f} \"\n              f\"({win['score'] - base['score']:+.4f}, 잡음 ±{MACRO_F1_NOISE} 밖)\")\n        print(f\"     best epoch {base.get('best_epoch', -1)} → {win['best_epoch']}\")\n        print(\"\\n  ⚠️ **서브셋 결과입니다.** 확정하려면 06 을 전체 데이터로 다시\")\n        print(\"     돌려야 합니다 — 서브셋 순위가 풀 순위와 같다는 보장은 없습니다.\")\n\n    return {\"verdict\": verdict, \"baseline\": key(base),\n            \"trained\": [key(r) for r in trained],\n            \"winner\": key(max(better, key=lambda r: r[\"score\"])) if better else None,\n            \"min_best_epoch\": LR_MIN_BEST_EPOCH,\n            \"noise\": MACRO_F1_NOISE}\n\n\n# ──────────────────────────────────────────────────────────────\n# 1단계 2×2 실험 (백본 × 증강)\n# ──────────────────────────────────────────────────────────────\n# STEP 5 에서 1단계가 holdout 에서 무너졌습니다 (AUROC 0.8143 → 0.7412).\n# 용의자가 둘이고, 둘을 한 실행에서 갈라 봅니다:\n#\n#   ① 화질 지름길  — 정상 사진이 계통적으로 흐림 (선명도 50 vs 274)\n#                    → `photometric` 증강으로 막히나?\n#   ② 표현력·사전학습 — resnet50(2015, in1k) 이 약한 건가?\n#                    → in21k 사전학습 백본이면 나은가?\n#\n# ⚠️ **holdout 은 여기서 안 봅니다.** 4개 중에 고르는 데 holdout 을 쓰면\n#    그 순간 holdout 이 오염되어 \"처음 보는 데이터\" 가 아니게 됩니다.\n#    고른 하나를 풀 데이터로 다시 학습한 뒤에만 엽니다.\n\n# 잡음 폭 — 실측 근거를 달아둡니다 (추정치 금지, 규칙 1)\nAUROC_NOISE = 0.01     # 같은 설정 두 실행: 0.8192(08-21) vs 0.8155(08-22)\nBLUR_NOISE_PP = 0.05   # 교란 검사 일반 (STEP 5 에서 ±3%p → ±5%p 상향)\n\n\ndef stage1_report(runs: list[dict], *, base_model: str = \"resnet50\",\n                  base_aug: str = \"default\") -> dict[str, Any]:\n    \"\"\"백본 × 증강 2×2 를 읽고 **미리 정해둔 기준**으로 판정합니다.\n\n    판정 기준 (실험 **전에** 못 박음 — 규칙 2):\n\n      · `photometric` 채택 ← AUROC 가 {AUROC_NOISE} 이상 안 떨어지면서\n                             흐림 하락이 {BLUR_NOISE_PP} 이상 줄어들 때\n      · 백본 교체 채택     ← AUROC 가 {AUROC_NOISE} 이상 오를 때\n      · 둘 다 아니면       → 그 축은 닫고 다음으로\n    \"\"\"\n    runs = [r for r in runs if r]\n    if not runs:\n        return {}\n\n    models_ = sorted({r[\"model_name\"] for r in runs})\n    augs = sorted({r[\"aug\"] for r in runs}, key=lambda a: (a != base_aug, a))\n    get = {(r[\"model_name\"], r[\"aug\"]): r for r in runs}\n\n    def fmt(v, pct=False, nd=4):\n        if v is None:\n            return \"   못 잼\"\n        return f\"{v:>8.1%}\" if pct else f\"{v:>8.{nd}f}\"\n\n    print(\"\\n\" + \"=\" * 74)\n    print(\" 1단계 2×2 — 백본 × 증강\")\n    print(\"=\" * 74)\n    print(f\"  {'백본':<16}{'증강':<14}{'AUROC':>10}{'흐림하락':>10}\"\n          f\"{'precision':>11}{'분':>6}\")\n    for m in models_:\n        for a in augs:\n            r = get.get((m, a))\n            if not r:\n                continue\n            print(f\"  {m:<16}{a:<14}{fmt(r.get('auroc'))}{fmt(r.get('blur_drop'), pct=True)}\"\n                  f\"{fmt(r.get('precision'), nd=3)}{r['minutes']:>6.0f}\")\n\n    base = get.get((base_model, base_aug))\n    if base is None:\n        print(\"\\n  ⚠️ 기준 조합을 못 찾아 판정을 생략합니다.\")\n        return {\"runs\": runs}\n\n    verdict: dict[str, Any] = {\"baseline\": f\"{base_model}/{base_aug}\"}\n    print(f\"\\n  기준: {base_model} / {base_aug}  \"\n          f\"(AUROC {base['auroc']:.4f}, 흐림 하락 {fmt(base.get('blur_drop'), pct=True).strip()})\")\n    print(f\"  잡음 폭: AUROC ±{AUROC_NOISE} · 흐림 하락 ±{BLUR_NOISE_PP:.0%}\")\n\n    # ── 증강 축 ────────────────────────────────────────────────\n    alt_aug = next((a for a in augs if a != base_aug), None)\n    if alt_aug:\n        r = get.get((base_model, alt_aug))\n        if r:\n            d_auroc = r[\"auroc\"] - base[\"auroc\"]\n            d_blur = (r[\"blur_drop\"] - base[\"blur_drop\"]\n                      if r.get(\"blur_drop\") is not None and base.get(\"blur_drop\") is not None\n                      else None)\n            print(f\"\\n  [증강 축] {base_aug} → {alt_aug}\")\n            print(f\"    AUROC     {d_auroc:+.4f}\")\n            print(f\"    흐림 하락  {'못 잼' if d_blur is None else f'{d_blur:+.1%}'}\")\n            if d_auroc > -AUROC_NOISE and d_blur is not None and d_blur <= -BLUR_NOISE_PP:\n                verdict[\"aug\"] = alt_aug\n                print(f\"    ✅ {alt_aug} 채택 — 점수를 안 깎으면서 화질 의존을 줄였습니다.\")\n            elif d_auroc <= -AUROC_NOISE:\n                verdict[\"aug\"] = base_aug\n                print(f\"    ❌ {base_aug} 유지 — {alt_aug} 이 점수를 깎습니다 \"\n                      \"(2단계에서와 같은 이유일 수 있습니다: 신호를 지움).\")\n            else:\n                verdict[\"aug\"] = base_aug\n                print(f\"    ➖ {base_aug} 유지 — 차이가 잡음 안입니다. 이 축은 닫습니다.\")\n\n    # ── 백본 축 ────────────────────────────────────────────────\n    # ⚠️ 전에는 대체 백본을 **하나만** 봤습니다 (2×2 전용). 3종 이상을 비교하면\n    #    나머지가 조용히 무시됩니다. 이제 전부 보고, 그중 가장 많이 오른 것을\n    #    고릅니다. 후보가 하나뿐이면 예전과 똑같이 동작합니다.\n    alts = [m for m in models_ if m != base_model]\n    gains: list[tuple[str, float]] = []\n    for alt_model in alts:\n        r = get.get((alt_model, base_aug))\n        if not r:\n            continue\n        d = r[\"auroc\"] - base[\"auroc\"]\n        gains.append((alt_model, d))\n        print(f\"\\n  [백본 축] {base_model} → {alt_model}\")\n        print(f\"    AUROC     {d:+.4f}\")\n        if d >= AUROC_NOISE:\n            print(f\"    ✅ 기준선을 넘습니다 (+{d:.4f} ≥ 잡음 {AUROC_NOISE})\")\n        elif d <= -AUROC_NOISE:\n            print(f\"    ❌ 더 나쁩니다\")\n        else:\n            print(f\"    ➖ 차이가 잡음({AUROC_NOISE}) 안입니다 — 구분 불가\")\n\n    if gains:\n        best_alt, best_d = max(gains, key=lambda t: t[1])\n        if best_d >= AUROC_NOISE:\n            verdict[\"model\"] = best_alt\n            # 2등과의 차이도 잡음 안이면 \"이겼다\" 고 말하면 안 됩니다.\n            others = [d for m, d in gains if m != best_alt]\n            runner = max(others) if others else None\n            if runner is not None and abs(best_d - runner) < AUROC_NOISE:\n                verdict[\"model_tie\"] = True\n                print(f\"\\n    ⚠️ {best_alt} 이 1등이지만 2등과 차이가 \"\n                      f\"{abs(best_d - runner):.4f} 로 잡음({AUROC_NOISE}) 안입니다 — \"\n                      \"**구분 불가**. 싼 쪽을 고르세요.\")\n            print(f\"\\n    ★ 백본 축: {best_alt} (+{best_d:.4f})\")\n        else:\n            verdict[\"model\"] = base_model\n            print(f\"\\n    ★ 백본 축: {base_model} 유지 — 아무도 잡음을 못 넘었습니다.\")\n\n    # ── 채택 조합 ──────────────────────────────────────────────\n    # ⚠️ **val AUROC 가 가장 높은 조합을 고르면 안 됩니다.** 그게 바로 우리가\n    #    당한 실수입니다 — val 0.8143 로 골랐는데 holdout 에서 0.7412 였습니다.\n    #    val 점수는 지름길을 쓰는 모델도 높게 나옵니다. 그래서 **두 축의 판정을\n    #    그대로 합친 조합**을 채택합니다. 판정에는 흐림 하락이 들어갑니다.\n    pick_m, pick_a = verdict.get(\"model\", base_model), verdict.get(\"aug\", base_aug)\n    picked = get.get((pick_m, pick_a))\n    if picked:\n        verdict[\"best\"] = {\"model\": pick_m, \"aug\": pick_a,\n                           \"auroc\": picked[\"auroc\"], \"blur_drop\": picked.get(\"blur_drop\"),\n                           \"exp_name\": picked[\"exp_name\"]}\n        print(f\"\\n  ★ 채택: {pick_m} / {pick_a}   \"\n              f\"AUROC {picked['auroc']:.4f} · 흐림 하락 \"\n              f\"{fmt(picked.get('blur_drop'), pct=True).strip()}\")\n\n    top = max(runs, key=lambda r: r[\"auroc\"])\n    if picked and (top[\"model_name\"], top[\"aug\"]) != (pick_m, pick_a):\n        print(f\"\\n  ⚠️ val AUROC 가 가장 높은 건 {top['model_name']} / {top['aug']} \"\n              f\"({top['auroc']:.4f}) 이지만 채택하지 않습니다.\")\n        print(f\"     흐림 하락이 {fmt(top.get('blur_drop'), pct=True).strip()} 로 \"\n              f\"지름길에 기대고 있습니다 (채택안 \"\n              f\"{fmt(picked.get('blur_drop'), pct=True).strip()}).\")\n        print(f\"     AUROC 차이 {top['auroc'] - picked['auroc']:+.4f} 는 \"\n              f\"잡음(±{AUROC_NOISE}) 안입니다.\"\n              if abs(top[\"auroc\"] - picked[\"auroc\"]) < AUROC_NOISE else\n              f\"     ⚠️ AUROC 차이 {top['auroc'] - picked['auroc']:+.4f} 는 잡음 밖입니다 \"\n              \"— 어느 쪽을 택할지 사람이 판단하세요.\")\n\n    print(\"\\n  ⚠️ 여기서 고른 조합은 **후보**입니다. 서브셋·짧은 에폭이라\")\n    print(\"     절대값은 풀 학습과 다릅니다 (STEP 4B 에서 확인). 풀 학습으로 확정하세요.\")\n    print(\"  ⚠️ holdout 은 아직 안 봤습니다 — 풀 학습 뒤에 한 번만 엽니다.\")\n    print(\"=\" * 74)\n    return verdict\n\n\n# ──────────────────────────────────────────────────────────────\n# 2단계 백본 비교 (STEP 9)\n# ──────────────────────────────────────────────────────────────\n# 2단계는 지금까지 resnet50 하나로만 돌았습니다. **한 번도 비교한 적이 없습니다.**\n# 1단계는 STEP 6 에서 effnetv2_s 로 바꿨지만, 그 결과를 2단계에 옮겨 적으면\n# 안 됩니다 — 같은 `photometric` 증강이 1단계에는 약이고 2단계에는 독이었습니다.\n\n# macro-F1 잡음 폭 — 실측 근거 (추정치 금지, 규칙 1)\n#   같은 설정 두 실행: m2.5 0.5395 / 0.5313 (Δ0.008), m1.5 0.5697 / 0.5536 (Δ0.016)\n#   부트스트랩 95% CI 반폭: 0.5456 → 0.5243~0.5663 (±0.021)\n# 둘을 합쳐 ±0.02 로 잡습니다.\nF1_NOISE = 0.02\n\n# 배율 하락으로 후보를 떨어뜨릴 때 쓰는 문턱.\n#\n# 원래 8%p 였습니다 (BLUR_NOISE_PP + 0.03). 그런데 우리가 **따로** 잰 교란 검사\n# 잡음이 그것보다 큽니다:\n#   · 설정을 하나도 안 바꾼 2단계를 일곱 번 재서 배율 하락 15.4% ~ 25.0% (폭 9.6%p)\n#   · STEP 10 한 실행 안에서 표본 수만 2,000 → 3,000 으로 바꿨더니\n#     위치 하락이 9.3% → 20.2% (10.9%p)\n#\n# 즉 8%p 짜리 문턱은 **잡음만으로도 걸립니다.** 실제로 STEP 9 에서\n# convnextv2_base 가 +8.1%p 로 탈락했는데, 그건 판정이 아니라 동전 던지기였습니다.\n#\n# ⚠️ 이 값을 STEP 9 결과를 **보고 나서** 올리는 게 아닙니다. 근거는 STEP 10\n#    (다른 실험)에서 나온 잡음 측정이고, 04 를 다시 돌리기 **전에** 못 박습니다\n#    (규칙 2). 이 사이 구간은 탈락도 통과도 아닌 **구분 불가**로 적습니다.\nSCALE_DROP_REJECT_PP = 0.12\n\n\ndef backbone_report(runs: list[dict], *, base_model: str = \"resnet50\") -> dict[str, Any]:\n    \"\"\"2단계 백본 비교를 **미리 정해둔 기준**으로 판정합니다.\n\n    판정 기준 (실험 **전에** 못 박음 — 규칙 2)\n    -------------------------------------------\n    1. macro-F1 이 기준선보다 **+0.02(F1_NOISE) 넘게** 높아야 교체 후보입니다.\n       그 안이면 \"구분 불가\" 이고, 구분 불가일 때는 **기준선을 유지**합니다\n       (바꿀 이유가 없으면 안 바꿉니다 — 바꾸면 비교 이력이 끊깁니다).\n    2. 점수가 올라도 **배율 하락이 12%p(SCALE_DROP_REJECT_PP) 넘게 나빠지면**\n       채택하지 않습니다. STEP 5·6 에서 val 최고점을 골랐다가 holdout 에서\n       무너진 실패를 반복하지 않기 위해서입니다.\n       5~12%p 사이는 **구분 불가**로 적고 판정하지 않습니다 — 우리가 잰\n       교란 검사 잡음이 그만큼 큽니다 (아래 상수 주석 참고).\n    3. 1·2 를 모두 만족하는 후보가 여럿이면 **macro-F1 이 가장 높은 것**.\n\n    ⚠️ 여기서 고른 건 **후보**입니다. 서브셋·짧은 에폭 결과라 절대값이\n       풀 학습과 다릅니다. 확정은 풀 학습으로 다시 합니다.\n    ⚠️ holdout 은 여기서 안 봅니다.\n    \"\"\"\n    runs = [r for r in runs if r]\n    if not runs:\n        print(\"⚠️ 성공한 실행이 없습니다 — 판정할 것이 없습니다.\")\n        return {}\n\n    base = next((r for r in runs if r.get(\"model_name\") == base_model), None)\n    if base is None:\n        print(f\"⚠️ 기준선 '{base_model}' 실행이 없어 상대 비교를 못 합니다.\")\n        base = max(runs, key=lambda r: r[\"score\"])\n        print(f\"   가장 높은 '{base['model_name']}' 를 임시 기준선으로 씁니다.\")\n\n    # ⚠️ 기준선이 수렴하지 않았으면 이 비교 전체가 흔들립니다.\n    #    \"서브셋 하락률을 풀에 대입해도 되나 → 아니오 — 약한 모델은 잃을 것도 적어\n    #    덜 떨어집니다\" (CLAUDE.md) 와 같은 함정입니다. 덜 학습된 기준선은 배운 게\n    #    적어 교란 검사에서 잃을 것도 적고, 그래서 배율 하락이 실제보다 낮게 나와\n    #    다른 백본이 부당하게 나빠 보입니다. 실제로 04 첫 실행에서 resnet50 이\n    #    12에폭에서도 계속 오르는 중이었고, 그 배율 하락(12.2%)이 이 프로젝트의\n    #    다른 풀 학습 실행들(23.8~25.0%)과 안 맞았습니다.\n    if not base.get(\"converged\", True):\n        print(f\"\\n  ⚠️ 기준선 '{base['model_name']}' 이 수렴하지 않았습니다 \"\n              \"(마지막 에폭이 최고였습니다). 덜 학습된 기준선은 교란 검사에서 잃을 게 \"\n              \"적어 배율 하락이 실제보다 낮게 나옵니다. 이 비교의 '탈락' 판정은 \"\n              \"재확인이 필요합니다 — 기준선을 더 학습시키거나 풀 데이터로 다시 재세요.\")\n\n    print(\"\\n\" + \"=\" * 78)\n    print(\" 2단계 백본 비교 — 판정\")\n    print(\"=\" * 78)\n    print(f\"  {'백본':<18}{'해상도':>7}{'macro-F1':>10}{'기준선대비':>11}\"\n          f\"{'배율하락':>10}{'분':>7}   수렴\")\n    for r in sorted(runs, key=lambda r: -r[\"score\"]):\n        d = r[\"score\"] - base[\"score\"]\n        drop = r.get(\"scale_drop\")\n        ds = f\"{drop:>9.1%}\" if drop is not None else f\"{'—':>10}\"\n        mark = \"  ← 기준선\" if r is base else \"\"\n        print(f\"  {r['model_name']:<18}{r['img_size']:>6}p{r['score']:>10.4f}\"\n              f\"{d:>+11.4f}{ds}{r['minutes']:>7.0f}   \"\n              f\"{'수렴' if r['converged'] else '더 필요'}{mark}\")\n\n    base_drop = base.get(\"scale_drop\")\n    ok, rejected, unclear = [], [], []\n    for r in runs:\n        if r is base:\n            continue\n        gain = r[\"score\"] - base[\"score\"]\n        drop = r.get(\"scale_drop\")\n        gap = (drop - base_drop) if (drop is not None and base_drop is not None) else None\n        if gain <= F1_NOISE:\n            rejected.append((r, f\"macro-F1 차이 {gain:+.4f} 가 잡음(±{F1_NOISE}) 안\"))\n        elif gap is not None and gap > SCALE_DROP_REJECT_PP:\n            rejected.append((r, f\"배율 하락이 {gap:+.1%}p 나빠짐 \"\n                                f\"(문턱 {SCALE_DROP_REJECT_PP:.0%}p)\"))\n        else:\n            if gap is not None and gap > BLUR_NOISE_PP:\n                unclear.append((r, gap))\n            ok.append(r)\n\n    print(\"\\n\" + \"-\" * 78)\n    for r, why in rejected:\n        print(f\"  ✗ {r['model_name']:<18} {why}\")\n    # 잡음보다는 크고 탈락 문턱보다는 작은 구간 — 판정하지 말고 그대로 적습니다.\n    for r, gap in unclear:\n        print(f\"  ◐ {r['model_name']:<18} 배율 하락 {gap:+.1%}p — 잡음(±{BLUR_NOISE_PP:.0%}p)\"\n              f\"보다 크지만 탈락 문턱({SCALE_DROP_REJECT_PP:.0%}p)에는 못 미칩니다. \"\n              \"구분 불가로 적고 풀 학습에서 다시 재세요.\")\n\n    verdict: dict[str, Any] = {\"baseline\": base, \"candidates\": ok, \"rejected\": rejected}\n    if ok:\n        best = max(ok, key=lambda r: r[\"score\"])\n        verdict[\"best\"] = best\n        print(f\"\\n  ✅ 채택 후보: **{best['model_name']}** \"\n              f\"(macro-F1 {best['score']:.4f}, 기준선 {base['score']:.4f} 대비 \"\n              f\"{best['score'] - base['score']:+.4f})\")\n    else:\n        verdict[\"best\"] = base\n        print(f\"\\n  ◐ 기준선 **{base_model}** 유지 — 잡음 밖으로 이긴 백본이 없습니다.\")\n        print(\"     바꿀 이유가 없으면 안 바꿉니다 (비교 이력이 끊깁니다).\")\n\n    top = max(runs, key=lambda r: r[\"score\"])\n    if top is not verdict[\"best\"]:\n        print(f\"\\n  ⚠️ macro-F1 이 가장 높은 건 {top['model_name']} ({top['score']:.4f}) 인데\")\n        print(\"     위 기준에서 걸러졌습니다. val 최고점을 그냥 고르지 않습니다\"\n              \" (STEP 5·6 의 실패).\")\n\n    print(\"\\n  ⚠️ 서브셋·짧은 에폭 결과입니다. 순위가 풀 학습과 같다는 보장은 없습니다.\")\n    print(\"  ⚠️ holdout 은 아직 안 봤습니다 — 풀 학습 뒤 05 에서 한 번만 엽니다.\")\n    print(\"=\" * 78)\n    return verdict\n\n\n\n# ── 1단계 고정 창 **크기** 비교 (STEP 20) ────────────────────────────────\n#\n# STEP 18 실측: 1단계는 병변이 **클수록** 점수가 낮습니다 (rho −0.201, 6개 클래스\n# 전부 음수). 2단계(`m2.5`, 병변에 비례하는 창)에서는 그 효과가 없어서, 원인이\n# \"큰 병변이 어렵다\" 가 아니라 **고정 창이 병변으로 꽉 차서 주변 정상 피부가\n# 안 보이는 것** 쪽으로 좁혀졌습니다. 창 안 놓침 1.8% vs 창 밖 4.6% (2.5배).\n#\n# 전체 184,735 병변 중 **23.2%** 가 320px 창을 넘칩니다 (A6 38.7% · A2 27.9%).\n# 창을 키우면 넘침은 줄지만(448 에서 12.4%) **작은 병변이 그만큼 작아집니다.**\n# 맞바꿈이라 실측이 필요합니다.\n#\n# ⚠️ 창을 **비례**로 바꾸면 안 됩니다 — STEP 9-A 에서 ROI 크롭이 정답을 흘려\n#    (정상 bbox 중앙값 0.71% vs 병변 1.25%) AUROC 가 0.8272 → 0.9477 로 갈렸습니다.\n#    **고정이라는 성질을 유지한 채 크기만** 바꿉니다.\n\n#   넘치던 병변의 놓침이 이만큼은 줄어야 \"창 때문\" 이 확인됩니다.\n#   ⚠️ 고정값이 아니라 **그 실행에서 계산한 부트스트랩 CI 반폭**과 비교합니다\n#      (evaluate.bootstrap_ci(metric=\"recall\")). 작은 부분집합에서 잡음에\n#      속지 않으려는 것입니다 — A6 가 val 256장에서 2σ ±0.085 였던 전례.\nWINDOW_MIN_OVERFLOW_GAIN = \"ci\"\n#   헛알림이 이보다 나빠지면 탈락. STEP 15 가 \"2%p 이상 줄어야 채택\" 을 쓴 것과\n#   같은 폭을 반대 방향으로 씁니다.\nWINDOW_FALSE_ALARM_TOL_PP = 0.02\n\n\n\n# ── 2단계 크롭: 비례 창 vs 고정 창 (STEP 22) ─────────────────────────────\n#\n# STEP 21 실측: VL01 안에서 A4 병변이 A1 보다 **1.68배** 큽니다. A1·A4 를 둘 다\n# 가진 개 72마리로 좁혀도 1.88배(Wilcoxon p=8.1e-05)라 \"다른 개를 다르게 찍었다\"\n# 로는 설명이 안 됩니다. 그런데 `m2.5` 는 창 = 병변 × 2.5 라 병변이 언제나\n# 프레임의 40% 를 차지합니다 — **절대 크기가 지워집니다.**\n#\n#     A1 117px → 창 292px → 384 입력에서 154px\n#     A4 196px → 창 490px → 384 입력에서 154px   ← 똑같아짐\n#\n# 고정 창(f320/f448)은 그 크기를 남깁니다. 그럼 A4↔A1 이 덜 헷갈릴까요?\n#\n# ⚠️ **좋아져도 바로 채택할 수 없습니다.** 절대 크기를 쓰면 배율 교란에 약해지고,\n#    배포에서 보호자의 촬영 거리는 통제되지 않습니다. 그래서 판정을 **세 갈래**로\n#    둡니다 — \"정보는 있으나 배율 의존\" 이 나오면 그건 진단이지 채택이 아닙니다.\n\n#   A4 recall 개선은 **그 실행에서 계산한 CI 반폭**과 비교합니다 (고정값 금지).\nS2SIZE_MAIN = \"ci\"\n#   A4→A1 혼동이 이만큼은 줄어야 \"크기 신호가 실제로 쓰였다\" 로 봅니다.\nS2SIZE_A4A1_DROP = 0.03\n\n\ndef stage2_size_report(runs: list[dict], *, base_crop: str = \"m2.5\",\n                       focus: str = \"A4\", other: str = \"A1\") -> dict[str, Any]:\n    \"\"\"2단계 **비례 창 vs 고정 창** 비교를 미리 정해둔 기준으로 판정합니다.\n\n    각 run 은 `train_and_measure` 결과에 아래를 더한 dict 를 기대합니다:\n\n        macro_f1, scale_drop           (train_and_measure 가 이미 넣습니다)\n        focus_recall, focus_recall_ci  관심 클래스 recall 과 CI 반폭\n        f2o_rate                       관심→비교 클래스 혼동률 (A4→A1)\n\n    판정 (실험 **전에** 못 박음 — 규칙 2)\n    -------------------------------------\n    1. **1차**: `focus_recall` 이 CI 반폭보다 크게 오를 것\n    2. `f2o_rate` 가 {S2SIZE_A4A1_DROP} 이상 줄 것\n    3. macro-F1 이 {MACRO_F1_NOISE} 넘게 떨어지지 않을 것\n    4. **배율 하락**이 {SCALE_DROP_REJECT_PP} 넘게 나빠지지 않을 것\n\n    1·2 를 만족하고 4 가 걸리면 **\"정보는 있으나 배율 의존\"** — 진단으로는\n    성공이지만 배포에는 못 씁니다 (보호자 촬영 거리가 통제되지 않으므로).\n    \"\"\"\n    runs = [r for r in runs if r]\n    if not runs:\n        print(\"⚠️ 성공한 실행이 없습니다 — 판정할 것이 없습니다.\")\n        return {}\n    base = next((r for r in runs if r.get(\"crop_tag\") == base_crop), None)\n    if base is None:\n        print(f\"⚠️ 기준 '{base_crop}' 실행이 없어 상대 비교를 못 합니다.\")\n        return {\"runs\": runs}\n\n    def f(v, pct=False, nd=4):\n        if v is None:\n            return \"   못 잼\"\n        return f\"{v:>9.1%}\" if pct else f\"{v:>9.{nd}f}\"\n\n    print(\"\\n\" + \"=\" * 84)\n    print(f\" 2단계 크롭 — 비례 창(크기 지움) vs 고정 창(크기 보존)\")\n    print(\"=\" * 84)\n    print(f\"  {'크롭':<9}{'macro-F1':>10}{focus + ' recall':>12}\"\n          f\"{focus + '→' + other:>10}{'배율하락':>10}{'분':>6}\")\n    for r in sorted(runs, key=lambda r: r[\"crop_tag\"] != base_crop):\n        print(f\"  {r['crop_tag']:<9}{f(r.get('macro_f1'))}{f(r.get('focus_recall'), nd=3)}\"\n              f\"{f(r.get('f2o_rate'), pct=True)}{f(r.get('scale_drop'), pct=True)}\"\n              f\"{r.get('minutes', 0):>6.0f}\")\n\n    verdict: dict[str, Any] = {\"baseline\": base_crop, \"candidates\": []}\n    for r in runs:\n        if r is base:\n            continue\n        d_rec = (r.get(\"focus_recall\") or 0) - (base.get(\"focus_recall\") or 0)\n        half = r.get(\"focus_recall_ci\") or base.get(\"focus_recall_ci\") or 0.0\n        d_f2o = (base.get(\"f2o_rate\") or 0) - (r.get(\"f2o_rate\") or 0)   # +면 개선\n        d_f1 = (r.get(\"macro_f1\") or 0) - (base.get(\"macro_f1\") or 0)\n        d_scale = (r.get(\"scale_drop\") or 0) - (base.get(\"scale_drop\") or 0)\n\n        ok_main = d_rec > half\n        ok_f2o = d_f2o >= S2SIZE_A4A1_DROP\n        ok_f1 = d_f1 > -MACRO_F1_NOISE\n        ok_scale = d_scale < SCALE_DROP_REJECT_PP\n\n        print(f\"\\n  [{base_crop} → {r['crop_tag']}]\")\n        print(f\"    {'[O]' if ok_main else '[X]'} 1차 — {focus} recall {d_rec:+.3f} \"\n              f\"(CI 반폭 ±{half:.3f})\")\n        print(f\"    {'[O]' if ok_f2o else '[X]'} {focus}→{other} 혼동 {-d_f2o:+.1%} \"\n              f\"(문턱 −{S2SIZE_A4A1_DROP:.0%})\")\n        print(f\"    {'[O]' if ok_f1 else '[X]'} macro-F1 {d_f1:+.4f} \"\n              f\"(허용 −{MACRO_F1_NOISE})\")\n        print(f\"    {'[O]' if ok_scale else '[X]'} 배율 하락 {d_scale:+.1%} \"\n              f\"(허용 +{SCALE_DROP_REJECT_PP:.0%})\")\n\n        if ok_main and ok_f2o and ok_f1 and ok_scale:\n            v = \"채택 후보\"\n        elif ok_main and ok_f2o:\n            v = \"정보는 있으나 **배율 의존** — 진단 성공, 배포엔 못 씀\"\n        else:\n            v = \"기각 — 크기 신호가 A4 를 살리지 못합니다\"\n        print(f\"    → {v}\")\n        verdict[\"candidates\"].append({\n            \"crop_tag\": r[\"crop_tag\"], \"verdict\": v, \"d_focus_recall\": d_rec,\n            \"ci_half\": half, \"d_f2o\": d_f2o, \"d_macro_f1\": d_f1,\n            \"d_scale_drop\": d_scale})\n\n    adopted = [c for c in verdict[\"candidates\"] if c[\"verdict\"] == \"채택 후보\"]\n    diag = [c for c in verdict[\"candidates\"] if \"배율 의존\" in c[\"verdict\"]]\n    verdict[\"verdict\"] = (\"채택 후보: \" + adopted[0][\"crop_tag\"]) if adopted else (\n        (\"배율 의존: \" + diag[0][\"crop_tag\"]) if diag else \"채택 없음\")\n    print(f\"\\n  판정: {verdict['verdict']}\")\n    print(\"  ⚠️ VL01 만 · 백본은 빠른 effnetv2_s 입니다. 효과가 크면 확정 백본\"\n          \"(convnextv2_base)으로 다시 확인해야 합니다.\")\n    print(\"  ⚠️ 고정 창은 배포에서 **촬영 거리에 의존**합니다 — 채택 전에\"\n          \" 그 위험을 따로 판단하세요.\")\n    print(\"=\" * 84)\n    return verdict\n\n\ndef stage1_window_report(runs: list[dict], *, base_crop: str = \"f320\") -> dict[str, Any]:\n    \"\"\"1단계 **고정 창 크기** 비교를 미리 정해둔 기준으로 판정합니다.\n\n    각 run 은 `train_and_measure` 결과에 아래를 더한 dict 를 기대합니다:\n\n        auroc                 전체 AUROC\n        blur_drop             흐림 교란 하락 (화질 지름길 감시)\n        false_alarm           recall 0.95 에서의 헛알림률\n        overflow_miss         **기준 창을 넘치던 병변**의 놓침률\n        overflow_miss_ci      그 값의 부트스트랩 CI 반폭\n        within_miss           넘치지 않던 병변의 놓침률\n\n    ⚠️ `overflow` 는 **기준 창(f320)** 기준으로 정의합니다. 창을 키우면 정의가\n       같이 움직이면 안 됩니다 — 같은 사진 집합을 두 모델이 어떻게 다루는지를\n       봐야 하니까요.\n\n    판정 (실험 **전에** 못 박음 — 규칙 2)\n    -------------------------------------\n    1. **1차**: `overflow_miss` 가 CI 반폭보다 크게 줄어야 합니다.\n       이게 가설의 핵심입니다. 점수보다 이게 먼저입니다.\n    2. AUROC 가 {AUROC_NOISE} 넘게 떨어지면 탈락.\n    3. 흐림 하락이 {BLUR_NOISE_PP} 넘게 나빠지면 탈락 (화질 지름길 재개방).\n    4. 헛알림이 {WINDOW_FALSE_ALARM_TOL_PP} 넘게 나빠지면 탈락.\n\n    1번만 만족하고 2~4 중 하나가 걸리면 **맞바꿈**으로 적고 채택하지 않습니다.\n    \"\"\"\n    runs = [r for r in runs if r]\n    if not runs:\n        print(\"⚠️ 성공한 실행이 없습니다 — 판정할 것이 없습니다.\")\n        return {}\n    base = next((r for r in runs if r.get(\"crop_tag\") == base_crop), None)\n    if base is None:\n        print(f\"⚠️ 기준 '{base_crop}' 실행이 없어 상대 비교를 못 합니다.\")\n        return {\"runs\": runs}\n\n    def f(v, pct=False, nd=4):\n        if v is None:\n            return \"   못 잼\"\n        return f\"{v:>9.1%}\" if pct else f\"{v:>9.{nd}f}\"\n\n    print(\"\\n\" + \"=\" * 82)\n    print(\" 1단계 고정 창 **크기** 비교 — 큰 병변이 창을 넘치는 문제\")\n    print(\"=\" * 82)\n    print(f\"  {'크롭':<9}{'AUROC':>9}{'넘친것 놓침':>12}{'안넘친것':>10}\"\n          f\"{'헛알림':>9}{'흐림하락':>10}{'분':>6}\")\n    for r in sorted(runs, key=lambda r: r[\"crop_tag\"] != base_crop):\n        print(f\"  {r['crop_tag']:<9}{f(r.get('auroc'))}{f(r.get('overflow_miss'), pct=True)}\"\n              f\"{f(r.get('within_miss'), pct=True)}{f(r.get('false_alarm'), pct=True)}\"\n              f\"{f(r.get('blur_drop'), pct=True)}{r.get('minutes', 0):>6.0f}\")\n    print(f\"\\n  ⚠️ '넘친것' 은 **{base_crop} 기준**으로 정의합니다 (창을 키워도 같은 사진 집합).\")\n\n    if not base.get(\"converged\", True):\n        print(f\"  ⚠️ 기준 '{base_crop}' 이 수렴하지 않았습니다 — 재확인 필요\"\n              \" (STEP 9 와 같은 함정).\")\n\n    verdict: dict[str, Any] = {\"baseline\": base_crop, \"candidates\": []}\n    for r in runs:\n        if r is base:\n            continue\n        d_of = base.get(\"overflow_miss\", 0) - r.get(\"overflow_miss\", 0)   # +면 개선\n        half = r.get(\"overflow_miss_ci\") or base.get(\"overflow_miss_ci\") or 0.0\n        d_auroc = (r.get(\"auroc\") or 0) - (base.get(\"auroc\") or 0)\n        d_blur = ((r.get(\"blur_drop\") or 0) - (base.get(\"blur_drop\") or 0))\n        d_fa = ((r.get(\"false_alarm\") or 0) - (base.get(\"false_alarm\") or 0))\n\n        ok_main = d_of > half\n        ok_auroc = d_auroc > -AUROC_NOISE\n        ok_blur = d_blur < BLUR_NOISE_PP\n        ok_fa = d_fa < WINDOW_FALSE_ALARM_TOL_PP\n\n        print(f\"\\n  [{base_crop} → {r['crop_tag']}]\")\n        print(f\"    {'[O]' if ok_main else '[X]'} 1차 — 넘친 병변 놓침 {d_of:+.1%} \"\n              f\"(CI 반폭 ±{half:.1%})\")\n        print(f\"    {'[O]' if ok_auroc else '[X]'} AUROC {d_auroc:+.4f} \"\n              f\"(허용 −{AUROC_NOISE})\")\n        print(f\"    {'[O]' if ok_blur else '[X]'} 흐림 하락 {d_blur:+.1%} \"\n              f\"(허용 +{BLUR_NOISE_PP:.0%})\")\n        print(f\"    {'[O]' if ok_fa else '[X]'} 헛알림 {d_fa:+.1%} \"\n              f\"(허용 +{WINDOW_FALSE_ALARM_TOL_PP:.0%})\")\n\n        if ok_main and ok_auroc and ok_blur and ok_fa:\n            v = \"채택\"\n        elif ok_main:\n            v = \"맞바꿈 — 넘친 병변은 좋아지나 다른 데서 잃습니다\"\n        else:\n            v = \"기각 — 창 크기가 원인이 아닙니다\"\n        print(f\"    → {v}\")\n        verdict[\"candidates\"].append({\"crop_tag\": r[\"crop_tag\"], \"verdict\": v,\n                                      \"d_overflow_miss\": d_of, \"ci_half\": half,\n                                      \"d_auroc\": d_auroc, \"d_blur\": d_blur,\n                                      \"d_false_alarm\": d_fa})\n\n    adopted = [c for c in verdict[\"candidates\"] if c[\"verdict\"] == \"채택\"]\n    verdict[\"verdict\"] = (\"채택: \" + adopted[0][\"crop_tag\"]) if adopted else \"채택 없음\"\n    print(f\"\\n  판정: {verdict['verdict']}\")\n    print(\"  ⚠️ VL01 만으로 낸 결과라면 전체 데이터에서 다시 확인해야 합니다\"\n          \" (서브셋 순위가 풀 학습과 같다는 보장은 없습니다).\")\n    print(\"=\" * 82)\n    return verdict\n\n\ndef stage1_crop_report(runs: list[dict], *, base_crop: str = \"full\") -> dict[str, Any]:\n    \"\"\"1단계 입력(크롭 태그) 비교를 **미리 정해둔 기준**으로 판정합니다.\n\n    STEP 8 에서 남은 위험: 1단계가 `full`(강아지 전신)로 학습했는데, 배포에서\n    보호자는 촬영 가이드대로 병변에 다가가서 찍습니다. `f320`(고정 픽셀 창)은\n    창 크기가 병변 크기와 무관해 창 크기 지름길이 없고, 구도도 근접 사진에\n    가깝습니다. 여기서는 **모델·증강은 STEP 6·7 이 정한 대로 고정**하고\n    (`effnetv2_s` + `photometric`) 입력만 바꿔 비교합니다.\n\n    판정 기준 (실험 **전에** 못 박음 — 규칙 2)\n    -------------------------------------------\n    1. AUROC 가 {AUROC_NOISE} 이상 오르거나, 최소한 안 떨어져야 후보입니다.\n       (holdout 변별력 부족이 STEP 8 의 핵심 문제라 AUROC 를 우선합니다)\n    2. 흐림 하락이 {BLUR_NOISE_PP} 넘게 나빠지면 탈락합니다 — f320 이 화질\n       지름길을 다시 열면 안 됩니다.\n    3. 스크리닝 recall 이 얼마나 나오는지는 여기서 안 봅니다. 임계값은\n       val 로 다시 잡아야 하는 값이라, 이 비교 단계에서는 신호가 아닙니다.\n\n    ⚠️ 여기서 고른 건 **후보**입니다. holdout 은 안 봅니다 — 풀 학습 뒤 05 에서\n    한 번만 엽니다.\n    \"\"\"\n    runs = [r for r in runs if r]\n    if not runs:\n        print(\"⚠️ 성공한 실행이 없습니다 — 판정할 것이 없습니다.\")\n        return {}\n\n    base = next((r for r in runs if r.get(\"crop_tag\") == base_crop), None)\n    if base is None:\n        print(f\"⚠️ 기준 '{base_crop}' 실행이 없어 상대 비교를 못 합니다.\")\n        return {\"runs\": runs}\n\n    def fmt(v, pct=False, nd=4):\n        if v is None:\n            return \"   못 잼\"\n        return f\"{v:>8.1%}\" if pct else f\"{v:>8.{nd}f}\"\n\n    print(\"\\n\" + \"=\" * 74)\n    print(\" 1단계 입력 비교 — full vs f320\")\n    print(\"=\" * 74)\n    print(f\"  {'크롭':<12}{'AUROC':>10}{'흐림하락':>10}{'precision':>11}{'분':>6}   수렴\")\n    for r in sorted(runs, key=lambda r: r[\"crop_tag\"] != base_crop):\n        print(f\"  {r['crop_tag']:<12}{fmt(r.get('auroc'))}{fmt(r.get('blur_drop'), pct=True)}\"\n              f\"{fmt(r.get('precision'), nd=3)}{r['minutes']:>6.0f}   \"\n              f\"{'수렴' if r['converged'] else '더 필요'}\")\n\n    if not base.get(\"converged\", True):\n        print(f\"\\n  ⚠️ 기준 '{base_crop}' 이 수렴하지 않았습니다. 이 비교는 재확인이 \"\n              \"필요합니다 (STEP 9 의 2단계 백본 비교와 같은 함정).\")\n\n    verdict: dict[str, Any] = {\"baseline\": base}\n    print(f\"\\n  기준: {base_crop}  (AUROC {base['auroc']:.4f}, \"\n          f\"흐림 하락 {fmt(base.get('blur_drop'), pct=True).strip()})\")\n    print(f\"  잡음 폭: AUROC ±{AUROC_NOISE} · 흐림 하락 ±{BLUR_NOISE_PP:.0%}\")\n\n    candidates = []\n    for r in runs:\n        if r is base:\n            continue\n        d_auroc = r[\"auroc\"] - base[\"auroc\"]\n        d_blur = (r[\"blur_drop\"] - base[\"blur_drop\"]\n                  if r.get(\"blur_drop\") is not None and base.get(\"blur_drop\") is not None\n                  else None)\n        print(f\"\\n  [{base_crop} → {r['crop_tag']}]\")\n        print(f\"    AUROC     {d_auroc:+.4f}\")\n        print(f\"    흐림 하락  {'못 잼' if d_blur is None else f'{d_blur:+.1%}'}\")\n        worse_blur = d_blur is not None and d_blur > BLUR_NOISE_PP\n        if d_auroc >= -AUROC_NOISE and not worse_blur:\n            candidates.append(r)\n            print(f\"    ✅ {r['crop_tag']} 후보 — AUROC 를 깎지 않으면서 화질 의존을 \"\n                  \"늘리지 않았습니다.\")\n        elif d_auroc < -AUROC_NOISE:\n            print(f\"    ❌ 탈락 — AUROC 가 잡음 밖으로 떨어졌습니다.\")\n        else:\n            print(f\"    ❌ 탈락 — 흐림 하락이 {d_blur:+.1%}p 나빠졌습니다.\")\n\n    if candidates:\n        best = max(candidates, key=lambda r: r[\"auroc\"])\n        verdict[\"best\"] = best\n        print(f\"\\n  ✅ 채택 후보: **{best['crop_tag']}** (AUROC {best['auroc']:.4f}, \"\n              f\"기준 {base['auroc']:.4f} 대비 {best['auroc'] - base['auroc']:+.4f})\")\n    else:\n        verdict[\"best\"] = base\n        print(f\"\\n  ◐ 기준 **{base_crop}** 유지 — 후보가 없습니다.\")\n\n    print(\"\\n  ⚠️ 서브셋·짧은 에폭 결과입니다. 순위가 풀 학습과 같다는 보장은 없습니다.\")\n    print(\"  ⚠️ holdout 은 아직 안 봤습니다 — 풀 학습 뒤 05 에서 한 번만 엽니다.\")\n    print(\"=\" * 74)\n    return verdict\n\n\n# ── 1단계 놓침이 병변 탓인가 창 배치 탓인가 (STEP 24) ────────────────────\n#\n# 배경: rho(병변 크기, 1단계 점수) = -0.201 로 **큰 병변을 더 놓칩니다**\n# (STEP 17·18). STEP 20 이 \"창이 작아서\" 를 기각했고(f448 로 넓히니 오히려\n# 악화), 남은 설명이 둘이었습니다:\n#\n#   H-A 병변    큰 병변이 원래 어렵다 — 창이 병변으로만 꽉 차 경계가 안 보인다\n#   H-B' 창배치  네모 중심이 병변을 벗어난다 — 네모는 폴리곤의 **외접사각형**이라\n#                길쭉·굽은 윤곽에서 중심이 빈 곳에 놓인다 (우리 파이프라인 탓)\n#\n# 두 가설은 같은 값(`occ` = 창 안 병변 점유율)에 **정반대**를 예측합니다.\n# 그래서 결과를 보고 기준을 고를 수 없게 여기 먼저 박아 둡니다 (작업 규칙 2).\n#\n# ⚠️ 이 판정은 **관찰**입니다. H-B' 가 지지돼도 \"창을 옮기면 낫다\" 는 아직\n#    아닙니다 — 그건 폴리곤 무게중심으로 다시 잘라 재는 **개입 실험**이 말합니다\n#    (`PLACEMENT_RECOVER_MIN` 아래).\n\nPLACEMENT_BINS = 5           # occ 를 몇 층으로 나눠 볼 것인가\nPLACEMENT_RATIO_HB = 2.0     # 최저층 놓침률 ÷ 최고층 ≥ 이 값이면 H-B'\nPLACEMENT_RATIO_HA = 0.5     # 그 비가 이 값 이하면 H-A (놓침이 꽉 찬 쪽에 몰림)\nPLACEMENT_RHO_RESIDUAL = 0.08   # 층 안에서 크기 상관이 이보다 작으면 \"크기 효과는 배치로 설명됨\"\nPLACEMENT_MIN_MISS = 40      # 놓침이 이보다 적으면 **판정하지 않습니다**\n\n# ⚠️ **이 잠금장치는 제가 실제로 밟은 함정에서 나왔습니다** (2026-09-06).\n#    위 H-A/H-B' 판정을 `occ`(창 안 병변 점유율)로 설계했는데, 막상 재 보니\n#    `rho(occ, 폴리곤 √넓이) = +0.995` 였습니다 — **occ 는 병변 크기의 다른\n#    이름**입니다. 그러면 \"occ 가 높을수록 놓친다\" 는 \"크면 놓친다\" 의 재진술이지\n#    두 가설을 가른 게 아닙니다. 그런데 코드는 아무 말 없이 \"H-A\" 를 찍었습니다.\n#    가르는 변수가 가르려는 원인과 붙어 있으면 **판정 자체가 성립하지 않습니다.**\nPLACEMENT_MAX_COLLINEARITY = 0.80   # |rho(층 변수, 크기)| 가 이 이상이면 판정 거부\n\n# 개입 실험(폴리곤 무게중심으로 창을 옮김) 채택 기준\nPLACEMENT_RECOVER_MIN = 0.30  # 놓쳤던 것 중 이만큼 되살아나야 채택 후보\nPLACEMENT_LOSS_TOL = 0.01     # 이미 맞히던 것을 이보다 많이 잃으면 기각\n\n\ndef stage1_placement_report(rows, *, bins: int = PLACEMENT_BINS,\n                            boot: int = 2000, seed: int = 0) -> dict[str, Any]:\n    \"\"\"`occ`(창 안 병변 점유율) 층별 놓침률로 H-A / H-B' 를 가릅니다.\n\n    `rows` 에 필요한 열: ``occ`` · ``miss``(bool) · ``box_px`` · ``score``.\n\n    돌려주는 값의 ``verdict`` 는 셋 중 하나입니다:\n      \"창 배치(H-B')\" · \"병변 자체(H-A)\" · \"구분 불가\"\n\n    ⚠️ **`occ` 가 크기와 붙어 있으면 판정을 거부합니다** — 그게 실제로 벌어졌고\n       (rho +0.995) 코드가 조용히 \"H-A\" 를 찍었습니다. `PLACEMENT_MAX_COLLINEARITY`.\n\n    ⚠️ 놓침이 `PLACEMENT_MIN_MISS` 미만이면 무조건 \"표본 부족\" 입니다.\n       VL01 val 은 놓침이 77건뿐이라 층마다 15건꼴입니다 — 부트스트랩 CI 를\n       같이 내고, **CI 가 겹치면 비가 아무리 커도 판정하지 않습니다.**\n    \"\"\"\n    import numpy as np\n    import pandas as pd\n\n    d = pd.DataFrame(rows).dropna(subset=[\"occ\", \"miss\"]).reset_index(drop=True)\n    n_miss = int(d[\"miss\"].sum())\n    out: dict[str, Any] = {\"n\": len(d), \"n_miss\": n_miss, \"bins\": [],\n                           \"verdict\": \"표본 부족\", \"why\": \"\"}\n    # 층을 가르는 변수가 크기와 붙어 있으면 이 검사는 아무것도 못 가릅니다\n    if \"box_px\" in d:\n        from scipy.stats import spearmanr\n        col = float(spearmanr(d[\"occ\"], d[\"box_px\"]).statistic)\n        out[\"collinearity\"] = col\n        if abs(col) >= PLACEMENT_MAX_COLLINEARITY:\n            out[\"verdict\"] = \"판정 불가(공선성)\"\n            out[\"why\"] = (\n                f\"층 변수 occ 가 크기와 rho={col:+.3f} 로 붙어 있습니다 \"\n                f\"(>= {PLACEMENT_MAX_COLLINEARITY}). occ 로 나눈 층은 크기로 나눈 \"\n                f\"층과 같아서, 어떤 결과가 나와도 '크면 놓친다' 의 재진술입니다. \"\n                f\"크기와 분리되는 **모양** 변수(slack · center_off · 길쭉함)로 \"\n                f\"크기를 고정한 채 재세요.\")\n            print(f\"[placement] {out['why']}\")\n            return out\n\n    if n_miss < PLACEMENT_MIN_MISS:\n        out[\"why\"] = (f\"놓침 {n_miss}건 < {PLACEMENT_MIN_MISS}건 — 판정하지 않습니다. \"\n                      f\"층마다 {n_miss / bins:.0f}건꼴이라 어떤 비도 잡음입니다.\")\n        print(f\"[placement] {out['why']}\")\n        return out\n\n    d[\"층\"] = pd.qcut(d[\"occ\"], bins, labels=False, duplicates=\"drop\")\n    rng = np.random.default_rng(seed)\n    for b in sorted(d[\"층\"].dropna().unique()):\n        g = d[d[\"층\"] == b]\n        m = g[\"miss\"].to_numpy().astype(float)\n        bs = np.array([rng.choice(m, len(m), replace=True).mean() for _ in range(boot)])\n        out[\"bins\"].append({\n            \"층\": int(b), \"n\": len(g), \"occ_중앙값\": float(g[\"occ\"].median()),\n            \"놓침률\": float(m.mean()),\n            \"ci\": [float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))],\n            \"box_px_중앙값\": float(g[\"box_px\"].median()) if \"box_px\" in g else float(\"nan\"),\n        })\n\n    lo, hi = out[\"bins\"][0], out[\"bins\"][-1]     # occ 최저층 / 최고층\n    ratio = lo[\"놓침률\"] / hi[\"놓침률\"] if hi[\"놓침률\"] > 0 else float(\"inf\")\n    overlap = not (lo[\"ci\"][0] > hi[\"ci\"][1] or hi[\"ci\"][0] > lo[\"ci\"][1])\n    out[\"ratio_lowest_over_highest\"] = ratio\n    out[\"ci_overlap\"] = overlap\n\n    # 크기 효과가 배치로 설명되는가 — 층 **안**에서 상관이 남는지\n    resid = []\n    if \"score\" in d and \"box_px\" in d:\n        from scipy.stats import spearmanr\n        for b in sorted(d[\"층\"].dropna().unique()):\n            g = d[d[\"층\"] == b].dropna(subset=[\"box_px\", \"score\"])\n            if len(g) >= 50:\n                resid.append(float(spearmanr(g[\"box_px\"], g[\"score\"]).statistic))\n    out[\"rho_within_bins\"] = resid\n    out[\"rho_within_max_abs\"] = max((abs(r) for r in resid), default=float(\"nan\"))\n    out[\"size_effect_explained\"] = bool(\n        resid and out[\"rho_within_max_abs\"] < PLACEMENT_RHO_RESIDUAL)\n\n    if overlap:\n        out[\"verdict\"] = \"구분 불가\"\n        out[\"why\"] = (f\"최저층 {lo['놓침률']:.1%} {lo['ci']} vs 최고층 \"\n                      f\"{hi['놓침률']:.1%} {hi['ci']} — CI 가 겹칩니다 (비 {ratio:.2f}).\")\n    elif ratio >= PLACEMENT_RATIO_HB:\n        out[\"verdict\"] = \"창 배치(H-B')\"\n        out[\"why\"] = (f\"병변이 적게 담긴 창에서 {ratio:.2f}배 더 놓칩니다 \"\n                      f\"({lo['놓침률']:.1%} vs {hi['놓침률']:.1%}, CI 안 겹침).\")\n    elif ratio <= PLACEMENT_RATIO_HA:\n        out[\"verdict\"] = \"병변 자체(H-A)\"\n        out[\"why\"] = (f\"창이 병변으로 꽉 찰수록 더 놓칩니다 (비 {ratio:.2f}) — \"\n                      f\"경계가 안 보이는 쪽이 어렵다는 뜻입니다.\")\n    else:\n        out[\"verdict\"] = \"구분 불가\"\n        out[\"why\"] = (f\"비 {ratio:.2f} 가 {PLACEMENT_RATIO_HA}~{PLACEMENT_RATIO_HB} \"\n                      f\"사이입니다 — 어느 쪽도 아닙니다.\")\n\n    print(f\"\\n[placement] n={out['n']:,} · 놓침 {n_miss}건\")\n    print(f\"{'층':>3} {'n':>6} {'occ 중앙':>9} {'네모px':>7} {'놓침률':>8}   95% CI\")\n    for b in out[\"bins\"]:\n        print(f\"{b['층']:>3} {b['n']:>6,} {b['occ_중앙값']:>9.3f} \"\n              f\"{b['box_px_중앙값']:>7.0f} {b['놓침률']:>8.1%}   \"\n              f\"[{b['ci'][0]:.1%}, {b['ci'][1]:.1%}]\")\n    if resid:\n        print(f\"층 안 rho(네모 크기, 점수): \"\n              f\"{', '.join(f'{r:+.3f}' for r in resid)}  \"\n              f\"→ 크기 효과가 배치로 설명되나: \"\n              f\"{'예' if out['size_effect_explained'] else '아니오'}\")\n    print(f\"판정: {out['verdict']} — {out['why']}\")\n    return out\n\n\n# ⚠️ **아래는 사전등록이 아니라 탐색적(exploratory) 분석입니다.** 위 `occ` 판정이\n#    공선성으로 무너진 뒤에 만들었으므로, 문턱을 결과를 보고 고른 것으로\n#    의심해야 합니다. 다만 결론이 **\"신호 없음\"** 쪽이라 문턱을 낮춰 잡을수록\n#    불리해집니다 — 그 방향의 자기기만은 어렵습니다. 그래도 확증으로 쓰지 말고\n#    **H-B'(창 배치)를 기각하는 근거**로만 쓰세요.\nSHAPE_RHO_MIN = 0.10          # 크기 층 안에서 이만큼은 돼야 \"신호\" (크기 자체는 0.20)\nSHAPE_MIN_CONSISTENT = 3      # 4개 층 중 몇 개가 같은 방향이어야 하나\n\n\ndef stage1_shape_report(rows, *, size_col: str = \"box_px\",\n                        shape_cols=(\"slack\", \"center_off\", \"ecc\"),\n                        bins: int = 4) -> dict[str, Any]:\n    \"\"\"**크기를 고정한 채** 모양이 1단계 놓침을 예측하는가.\n\n    `occ` 판정이 못 하는 일을 합니다 — 모양 변수들은 크기와 거의 안 붙어 있어\n    (rho 0.10~0.26) 크기 층 안에서 따로 움직일 수 있습니다.\n\n    ``slack``       1 - 폴리곤/네모 넓이. 네모로 줄이며 생긴 빈 곳\n    ``center_off``  네모 중심 ↔ 폴리곤 무게중심 거리 / 네모 긴 변\n    ``ecc``         네모 긴 변 / 짧은 변 (길쭉함)\n\n    판정: 어느 모양 변수든 4개 크기 층 중 `SHAPE_MIN_CONSISTENT` 개 이상에서\n    같은 방향이고 |rho| >= `SHAPE_RHO_MIN` 이면 \"모양도 관여\".\n    아니면 **\"크기가 전부\"** — 즉 창 배치 가설(H-B') 기각.\n    \"\"\"\n    import pandas as pd\n    from scipy.stats import spearmanr\n\n    d = pd.DataFrame(rows).dropna(subset=[size_col, \"score\"]).reset_index(drop=True)\n    d[\"_q\"] = pd.qcut(d[size_col], bins, labels=False, duplicates=\"drop\")\n    out: dict[str, Any] = {\"n\": len(d), \"size_col\": size_col, \"shape\": {}}\n\n    print(f\"\\n[shape] 크기({size_col}) {bins}층 안에서 모양이 점수를 예측하나  n={len(d):,}\")\n    print(f\"{'변수':<12}\" + \"\".join(f\"{'층'+str(i):>9}\" for i in range(bins))\n          + f\"{'같은방향':>9}{'판정':>10}\")\n    hit = []\n    for c in shape_cols:\n        if c not in d:\n            continue\n        g = d.dropna(subset=[c])\n        rs = [float(spearmanr(x[c], x[\"score\"]).statistic)\n              for _, x in g.groupby(\"_q\", observed=True) if len(x) >= 50]\n        pos, neg = sum(r > 0 for r in rs), sum(r < 0 for r in rs)\n        same = max(pos, neg)\n        strong = sum(abs(r) >= SHAPE_RHO_MIN for r in rs)\n        ok = same >= SHAPE_MIN_CONSISTENT and strong >= SHAPE_MIN_CONSISTENT\n        hit.append(ok)\n        out[\"shape\"][c] = {\"rho_by_size_bin\": rs, \"same_direction\": same,\n                           \"strong\": strong, \"signal\": ok}\n        print(f\"{c:<12}\" + \"\".join(f\"{r:>+9.3f}\" for r in rs)\n              + f\"{same:>9}\" + f\"{'신호' if ok else '없음':>10}\")\n\n    # 대조군 — 크기 자체는 같은 자리에서 얼마나 강한가\n    rs_size = [float(spearmanr(x[size_col], x[\"score\"]).statistic)\n               for _, x in d.groupby(\"_q\", observed=True) if len(x) >= 50]\n    out[\"rho_size_within_own_bins\"] = rs_size\n    print(f\"{'(대조)크기':<12}\" + \"\".join(f\"{r:>+9.3f}\" for r in rs_size)\n          + \"   ← 층 안에서도 크기는 남습니다\" if rs_size else \"\")\n\n    out[\"verdict\"] = \"모양도 관여\" if any(hit) else \"크기가 전부 (창 배치 H-B' 기각)\"\n    out[\"why\"] = (\n        f\"모양 변수 {len(hit)}개 중 신호 {sum(hit)}개. \"\n        f\"{SHAPE_MIN_CONSISTENT}개 층 이상에서 |rho| >= {SHAPE_RHO_MIN} 이고 방향이 \"\n        f\"같아야 신호로 셉니다.\")\n    print(f\"판정: {out['verdict']} — {out['why']}\")\n    return out\n\n\n# ── 2단계 크롭 뷰 앙상블 채택 기준 (STEP 25) ─────────────────────────\n#\n# 같은 사진을 다르게 자른 모델들의 **확률을 평균**합니다. 재학습이 없습니다.\n# STEP 22·23 에서 `m2.5`(비례 창)와 `f320`(고정 창)이 각각 단독으로는 상대를\n# 못 이겼는데, **서로 다른 실수를 한다면** 합이 둘 다를 이깁니다.\n#\n# ⚠️ **STEP 23 에서 제 기준에 구멍이 있었습니다.** \"짝 혼동(A4→A1)이 줄 것\" 은\n#    통과했는데, 줄어든 만큼 정답이 아니라 **나머지 클래스로 흩어졌습니다**\n#    (그 밖 36.8% → 42.1%). 행선지 하나만 보면 악화를 개선으로 읽습니다.\n#    → 그래서 `ENS_NO_SCATTER` 를 넣습니다. 이번엔 안 놓칩니다.\n#\n# ⚠️ 앙상블은 **추론 비용이 팔 수만큼 곱해집니다.** 서빙은 사진 한 장에\n#    CPU 1~3초인데 2팔이면 2~6초입니다. macro-F1 이 올라도 그 값을 치를지는\n#    별도 판단입니다 — 이 함수는 성능만 봅니다.\n\nENS_MIN_GAIN = MACRO_F1_NOISE      # macro-F1 이 이만큼은 올라야 (0.02)\nENS_CI_MUST_EXCLUDE_ZERO = True    # 짝지은 부트스트랩 CI 가 0 을 안 넘을 것\nENS_SCALE_TOL_PP = SCALE_DROP_REJECT_PP   # 배율 하락이 이보다 더 나빠지면 기각\nENS_NO_CLASS_LOSS = 0.03           # 어느 클래스도 recall 이 이보다 더 떨어지면 안 됨\nENS_NO_SCATTER = 0.0               # 주목 클래스의 '그 밖으로' 비율이 늘면 기각\n\n\ndef stage2_ensemble_report(base: dict, cand: dict, *, focus: str = \"A4\",\n                           other: str = \"A1\") -> dict[str, Any]:\n    \"\"\"앙상블을 채택할지 — 네 관문을 **전부** 통과해야 합니다.\n\n    `base` / `cand` 에 필요한 열:\n      ``macro_f1`` · ``d_macro_f1_ci`` (차이의 95% CI, [lo, hi]) ·\n      ``scale_drop`` · ``recall`` (클래스→recall dict) ·\n      ``focus_to_other`` · ``focus_to_rest``\n\n    관문\n      1. macro-F1 이득 >= `ENS_MIN_GAIN` **그리고** 차이 CI 가 0 을 안 넘음\n      2. 배율 하락 악화 <= `ENS_SCALE_TOL_PP`\n      3. 어떤 클래스도 recall 이 `ENS_NO_CLASS_LOSS` 넘게 떨어지지 않음\n      4. 주목 클래스가 **흩어지지 않음** (`focus_to_rest` 가 늘지 않음) ← STEP 23 의 구멍\n    \"\"\"\n    g = {}\n    d = cand[\"macro_f1\"] - base[\"macro_f1\"]\n    lo, hi = cand.get(\"d_macro_f1_ci\", [float(\"nan\")] * 2)\n    g[\"1. macro-F1\"] = (d >= ENS_MIN_GAIN and lo > 0,\n                        f\"{d:+.4f} (문턱 {ENS_MIN_GAIN}) · CI [{lo:+.4f}, {hi:+.4f}]\")\n\n    # ⚠️ **안 잰 것을 '통과' 로 찍지 않습니다.** 이 리포가 반복해 당한 모양입니다\n    #    (export_release 의 temperature.json 이 빠져도 T=1.0 으로 조용히 물러섬,\n    #     unzip -n 이 잘린 파일을 '있으니 건너뜀' 으로 처리, …).\n    #    못 잰 관문이 있으면 판정은 **'미완'** 이지 '채택 후보' 가 아닙니다.\n    if base.get(\"scale_drop\") is None or cand.get(\"scale_drop\") is None:\n        ds = float(\"nan\")\n        g[\"2. 배율 하락\"] = (None, \"**못 쟀습니다** — 재기 전에는 채택 불가\")\n    else:\n        ds = cand[\"scale_drop\"] - base[\"scale_drop\"]\n        g[\"2. 배율 하락\"] = (ds <= ENS_SCALE_TOL_PP,\n                         f\"{base['scale_drop']:.1%} → {cand['scale_drop']:.1%} \"\n                         f\"({ds:+.1%}, 허용 +{ENS_SCALE_TOL_PP:.0%})\")\n\n    worst_c, worst_d = None, 0.0\n    for c, v in base.get(\"recall\", {}).items():\n        dd = cand[\"recall\"].get(c, float(\"nan\")) - v\n        if dd < worst_d:\n            worst_c, worst_d = c, dd\n    g[\"3. 클래스 손실\"] = (worst_d >= -ENS_NO_CLASS_LOSS,\n                      f\"최악 {worst_c or '없음'} {worst_d:+.3f} (허용 −{ENS_NO_CLASS_LOSS})\")\n\n    dr = cand[\"focus_to_rest\"] - base[\"focus_to_rest\"]\n    g[\"4. 흩어짐\"] = (dr <= ENS_NO_SCATTER,\n                   f\"{focus}→그밖 {base['focus_to_rest']:.1%} → \"\n                   f\"{cand['focus_to_rest']:.1%} ({dr:+.1%})\")\n\n    print(\"\\n[ensemble] 채택 관문\")\n    for k, (ok, why) in g.items():\n        print(f\"  {'못 잼' if ok is None else ('통과' if ok else '실패'):5} {k:14} {why}\")\n    failed = any(ok is False for ok, _ in g.values())\n    unknown = any(ok is None for ok, _ in g.values())\n    passed = not failed and not unknown\n    out = {\"gates\": {k: {\"pass\": None if ok is None else bool(ok), \"detail\": w}\n                     for k, (ok, w) in g.items()},\n           \"verdict\": \"기각\" if failed else\n                      (\"미완(못 잰 관문 있음)\" if unknown else \"채택 후보\"),\n           \"d_macro_f1\": d, \"d_scale_drop\": ds}\n    extra = (\"\" if passed else\n             \"  한 관문이라도 실패하면 채택하지 않습니다.\" if failed else\n             \"  못 잰 관문이 있으면 '통과' 가 아닙니다 — 재고 다시 부르세요.\")\n    print(f\"판정: {out['verdict']}{extra}\")\n    if passed:\n        print(\"  ⚠️ **후보**입니다 — VL01 결과라면 전체 val 확인 전에 채택 금지 \"\n              \"(STEP 22 → 23 에서 정확히 이걸로 뒤집혔습니다).\")\n        print(\"  ⚠️ 추론 비용이 팔 수만큼 곱해집니다. 서빙 지연을 따로 재세요.\")\n    return out\n\n\ndef estimate_runtime(model_names: list[str] | list[tuple[str, int]], img_size: int,\n                     n_train: int, epochs: int, n_conditions: int | None = None,\n                     device: str | None = None) -> dict[str, Any]:\n    \"\"\"학습을 시작하기 **전에** 총 예상 시간을 찍습니다.\n\n    \"몇 시간 걸릴지 모르고 돌렸다가 뒤통수\" 를 여러 번 맞아서 넣었습니다.\n    합성 텐서로 GPU 속도만 재므로 백본당 20초 안쪽입니다.\n\n    `model_names` 는 이름 목록이거나 **(이름, 해상도) 목록**입니다. 뒤엣것을 쓰면\n    백본마다 다른 해상도로 잽니다 — ViT 계열은 해상도가 고정이라 CNN 과 같은\n    384 로 재면 안 됩니다 (판 B 가 그 경우입니다).\n\n    ⚠️ 데이터 로딩이 병목이면 실제는 이보다 느립니다. **하한 추정**입니다.\n    \"\"\"\n    from src import bench\n    from src.config import CFG, MODEL_BY_KEY\n\n    pairs = [(m, img_size) if isinstance(m, str) else (m[0], m[1]) for m in model_names]\n    n_conditions = n_conditions or len(pairs)\n    rows, total_min = [], 0.0\n    print(\"\\n\" + \"=\" * 66)\n    print(\" 시작 전 시간 추정 (GPU 속도 실측, 백본당 ~20초)\")\n    print(\"=\" * 66)\n    for key, size in pairs:\n        spec = MODEL_BY_KEY[key]\n        cfg = CFG(model_name=spec.timm_name, img_size=size)\n        # ⚠️ 하나가 터져도 나머지 추정은 보여줘야 합니다. 추정하다 죽으면\n        #    정작 돌 수 있는 백본들의 시간도 못 보고 셀이 멈춥니다.\n        try:\n            g = bench.gpu_speed(cfg, n_classes=2, steps=20)\n        except Exception as exc:                                   # noqa: BLE001\n            print(f\"  {key:<16} 속도 측정 실패 ({type(exc).__name__}: \"\n                  f\"{str(exc).splitlines()[0][:60]}) — 추정 생략\")\n            continue\n        ips = float(g.get(\"img_per_sec\") or 0.0)\n        # GPU 가 없으면 img_per_sec 이 NaN 입니다 (NaN 은 비교가 전부 False 라 따로 봅니다)\n        if not (ips > 0) or ips != ips:\n            print(f\"  {key:<16} 속도를 못 쟀습니다 ({g.get('note', '이유 불명')}) — 추정 생략\")\n            continue\n        epoch_min = n_train / ips / 60\n        run_min = epoch_min * epochs\n        rows.append({\"model\": key, \"img_size\": size, \"img_per_sec\": ips,\n                     \"batch\": g.get(\"batch\"),\n                     \"peak_vram_gb\": g.get(\"peak_vram_gb\"),\n                     \"epoch_min\": epoch_min, \"run_min\": run_min})\n        total_min += run_min\n        print(f\"  {key:<16}{size:>4}px{ips:>7.0f} img/s  배치 {g.get('batch', '?'):>3}  \"\n              f\"VRAM {g.get('peak_vram_gb', 0):>4.1f}GB   \"\n              f\"1에폭 {epoch_min:>5.1f}분   {epochs}에폭 {run_min:>6.0f}분\")\n\n    # 조건 수가 백본 수보다 많으면(2×2 처럼) 백본별로 같은 횟수만큼 돕니다\n    reps = max(1, n_conditions // max(len(rows), 1))\n    total_min *= reps\n    print(\"-\" * 66)\n    print(f\"  조건 {n_conditions}개 → 학습 총 예상 **{total_min / 60:.1f}시간**  \"\n          f\"(+ 교란 검사·크롭 확인 별도)\")\n    print(\"  ⚠️ GPU 속도만 잰 **하한**입니다. 데이터 로딩이 병목이면 더 걸립니다.\")\n    print(\"  → 너무 길면 여기서 멈추고 서브셋을 줄이거나 백본을 바꾸세요.\")\n    print(\"=\" * 66)\n    return {\"rows\": rows, \"total_hours\": total_min / 60, \"n_conditions\": n_conditions}\n\n# ── STEP 27 — \"확신 있을 때만 이름을 말할 것인가\" 판정 기준 ─────────────\n#\n# ⚠️ **결과를 보기 전에 박습니다** (작업 규칙 2). 이 결정은 숫자만으로 끝나지\n#    않지만, 숫자가 어느 쪽이면 **대화 자체가 필요 없는지**는 정할 수 있습니다.\n#\n# 왜 새 기준이 필요한가 — STEP 11 의 \"커버리지 18.2%\" 는 **2단계 val(병변만)**\n# 에서 잰 값입니다. 실제 화면에는 **1단계가 넘긴 사진**이 뜨고, 거기엔 헛알림\n# (멀쩡한 개)이 섞여 있습니다. 그 사진에 병변 이름이 붙으면 **무조건 오답**\n# 입니다. 그래서 분모도 오답 정의도 달라집니다.\n#\n#   분모  = 1단계가 \"이상\" 으로 넘긴 사진 전부 (헛알림 포함)\n#   오답  = 정상인데 이름을 붙였다  OR  병변인데 다른 이름을 붙였다\n#   커버리지 = 그중 실제로 이름을 말한 비율\n\nNAMING_TARGET_ERROR = 0.20\n\"\"\"이름을 말한 것 중 허용할 오답률. STEP 11 이 쓴 값을 그대로 씁니다 —\n기준을 지금 새로 고르면 STEP 11 의 결정과 비교가 안 됩니다.\"\"\"\n\nNAMING_MIN_COVERAGE = 0.50\n\"\"\"이 목표 오답률에서 커버리지가 이보다 높아야 **멘토와 논의할 가치**가 있습니다.\n절반은 말할 수 있어야 화면에 칸을 하나 더 두는 값을 합니다.\"\"\"\n\nNAMING_CLOSE_COVERAGE = 0.30\n\"\"\"이보다 낮으면 **축을 닫습니다.** 셋 중 하나도 말 못 하면서 '가끔 이름을\n말하는' 화면은 보호자에게 일관성 없는 물건이 됩니다.\"\"\"\n\nNAMING_A6_MISS_MAX = 0.30\n\"\"\"★ 안전 관문. 실제 A6(결절·종괴 — 종양 감별이 필요한 병변)인데 **다른 이름을\n말한** 비율. 나머지 오답은 '병원 가세요' 라는 행동을 안 바꾸지만, A6 을 순한\n이름으로 부르면 **미루게 만들 수 있습니다.** 이 관문만 방향이 비대칭입니다.\n⚠️ 문턱 0.30 은 근거가 있는 값이 아니라 **처음 놓는 말뚝**입니다 — STEP 16\nholdout 의 A6 recall 이 0.619 였으니 '말한 것 중에서는 그보다 나아야 한다'\n정도의 뜻입니다. 실측 뒤에 근거를 붙여 다시 놓습니다.\"\"\"\n\n\ndef naming_report(rows: dict, *, target=NAMING_TARGET_ERROR,\n                  a6_index: int | None = None) -> dict:\n    \"\"\"1단계 헛알림까지 포함한 **정직한** 커버리지-오답 곡선.\n\n    `rows` 에 필요한 것 (전부 같은 길이·같은 순서, 1단계가 넘긴 사진만):\n\n        conf   각 사진의 \"이름 확신도\" (내림차순으로 말할 것을 고릅니다)\n        wrong  그 이름이 틀렸는가 (bool) — 정상 사진은 **항상 True**\n        is_a6  실제 라벨이 A6 인가 (bool)\n        said_a6 우리가 A6 이라고 말했는가 (bool)\n\n    돌려주는 것: 목표 오답률에서의 커버리지 · 문턱 · A6 안전 지표 · 판정.\n    \"\"\"\n    import numpy as np\n\n    conf = np.asarray(rows[\"conf\"], dtype=float)\n    wrong = np.asarray(rows[\"wrong\"], dtype=bool)\n    is_a6 = np.asarray(rows[\"is_a6\"], dtype=bool)\n    said_a6 = np.asarray(rows[\"said_a6\"], dtype=bool)\n    n = len(conf)\n    if not (len(wrong) == len(is_a6) == len(said_a6) == n):\n        raise ValueError(\"네 배열의 길이가 다릅니다\")\n\n    order = np.argsort(-conf)\n    err = np.cumsum(wrong[order]) / np.arange(1, n + 1)\n    cov = np.arange(1, n + 1) / n\n\n    ok = np.flatnonzero(err <= target)\n    if len(ok) == 0:\n        k, coverage, thr = 0, 0.0, float(\"inf\")\n    else:\n        k = int(ok[-1]) + 1                 # 목표를 지키는 **가장 넓은** 지점\n        coverage, thr = float(cov[k - 1]), float(conf[order][k - 1])\n\n    spoken = order[:k]\n    a6_true = is_a6[spoken]\n    a6_miss = float((~said_a6[spoken][a6_true]).mean()) if a6_true.any() else float(\"nan\")\n\n    curve = {f\"cov@err{int(t * 100)}\":\n             (float(cov[np.flatnonzero(err <= t)[-1]])\n              if len(np.flatnonzero(err <= t)) else 0.0)\n             for t in (0.10, 0.20, 0.30)}\n\n    if coverage >= NAMING_MIN_COVERAGE:\n        verdict = \"논의할 가치 있음\"\n    elif coverage < NAMING_CLOSE_COVERAGE:\n        verdict = \"축을 닫음\"\n    else:\n        verdict = \"판단 보류 — 멘토 결정\"\n    if a6_true.any() and a6_miss > NAMING_A6_MISS_MAX:\n        verdict = f\"안전 관문 실패 (A6 오명명 {a6_miss:.1%})\"\n\n    out = {\"n_flagged\": n, \"target_error\": target, \"coverage\": coverage,\n           \"threshold\": thr, \"n_spoken\": int(k),\n           \"a6_true_spoken\": int(a6_true.sum()), \"a6_misnamed\": a6_miss,\n           \"curve\": curve, \"verdict\": verdict}\n\n    print(f\"[naming] 1단계가 넘긴 사진 {n:,}장 (헛알림 포함)\")\n    print(f\"  오답률 {target:.0%} 목표에서 **커버리지 {coverage:.1%}** \"\n          f\"({k:,}장, 확신도 문턱 {thr:.3f})\")\n    for t, v in curve.items():\n        print(f\"    {t:12} {v:>7.1%}\")\n    if a6_true.any():\n        mark = \"통과\" if a6_miss <= NAMING_A6_MISS_MAX else \"실패\"\n        print(f\"  안전 관문  말한 것 중 실제 A6 {int(a6_true.sum()):,}장 중 \"\n              f\"**{a6_miss:.1%}** 를 다른 이름으로 (허용 {NAMING_A6_MISS_MAX:.0%}) — {mark}\")\n    else:\n        print(\"  안전 관문  말한 것 중 실제 A6 이 없습니다 — 못 잼\")\n    print(f\"판정: {verdict}\"\n          f\"   (≥{NAMING_MIN_COVERAGE:.0%} 논의 / <{NAMING_CLOSE_COVERAGE:.0%} 닫음)\")\n    return out\n\n# ── STEP 28 — \"무엇을 말할 것인가\" (알갱이 크기) 판정 기준 ──────────────\n#\n# STEP 27 이 \"6종 이름\" 으로는 커버리지 33.7% 라고 했습니다. 그런데 우리가\n# 물은 것이 **\"6종 이름을 말할 것인가\"** 였지 **\"무엇을 말할 것인가\"** 가\n# 아니었습니다. 말할 수 있는 것은 여러 알갱이가 있습니다:\n#\n#   6종 이름      \"농포·여드름으로 보입니다\"\n#   형태 계열     \"융기·발진 계열로 보입니다\"      (A1·A4 / A2·A3 / A5·A6)\n#   긴급도        \"조기 진료를 권합니다\"            (관찰 / 진료 권장 / 조기 진료)\n#   A6 이진       \"덩어리가 의심됩니다\"\n#   두 이름       \"구진 또는 농포로 보입니다\"\n#\n# ⚠️ 묶음은 `config.URGENCY_TIER` / `config.MORPH_GROUP` 에서 가져옵니다 —\n#    **임상 문서에 먼저 있던 것**이고 혼동행렬을 보고 만들지 않았습니다.\n\n#: 1단계가 놓치는 병변의 비율. **이미 제품이 받아들이고 있는 하향**입니다 —\n#: 놓친 병변은 화면에 *\"괜찮아 보여요\"* 로 나가므로 **완전한 하향**입니다.\n#: 실측: holdout recall 0.9430 → 5.70% (STEP 33). val 0.9434 → 5.66%.\nSTAGE1_MISS_RATE = 0.057\n\nUNDER_TRIAGE_MAX = 0.05\n\"\"\"★ 안전 관문. 말한 것 중 **긴급도를 낮춰 말한** 비율의 상한.\n\n임상 해설이 위험한 혼동으로 A6→A2 · A5→A1 · A6→A1 을 꼽는데 셋의 공통점이\n\"급한 걸 안 급하다고 말했다\" 입니다. 반대 방향(안 급한 걸 급하다고)은\n병원에 가게 만들 뿐이라 안전합니다 — 1단계 헛알림도 같은 종류입니다.\n\n★ **닻 (STEP 34)** — 처음엔 근거 없이 박은 말뚝이었습니다. 이제 기준이\n있습니다: **1단계가 이미 받아들이고 있는 하향보다 작아야 합니다.**\n\n    1단계 놓침 5.70%   ← 병변인데 \"괜찮아 보여요\" = **완전한** 하향\n                          (병원에 아예 안 갑니다)\n    2단계 하향  ≤5%    ← 조기진료 → 진료권장 = **부분** 하향\n                          (병원에는 여전히 갑니다)\n\n2단계의 하향은 **더 작고 더 약해야** 합니다. 크기는 이 상수가, 세기는\n'그래도 병원에 간다' 는 사실이 보장합니다. 클래스별로 봐도 같습니다 —\n1단계는 **A6 을 8%** 완전히 놓치는데(STEP 16 holdout), 2단계 계열 4군의\nA6 부분 하향은 3.7% 입니다 (STEP 33).\n\n⚠️ 그래도 **'0 보다 나쁜 것을 받아들인다' 는 판단은 남습니다.** 이름을 아예\n안 말하면 이 값은 정의상 0 이니까요. 닻은 \"얼마나 나쁜 것까지\" 에 답할 뿐\n\"나쁜 걸 받아들일까\" 에는 답하지 않습니다 — 그건 사람 몫입니다.\"\"\"\n\nUNDER_TRIAGE_PER_CLASS_MIN_N = 30\n\"\"\"클래스별 하향을 **찍기만** 하는 최소 표본. 관문이 아닙니다 — 아래 참고.\"\"\"\n\n#: ⚠️ **일부러 문턱을 안 겁니다 (STEP 35).**\n#:\n#: `UNDER_TRIAGE_MAX` 는 전체 평균이라 **한 클래스가 나빠도 안 보입니다.**\n#: 실측(holdout, 계열 4군 3팔): 전체 하향 **3.7%** 로 관문을 통과하는데\n#: **미란·궤양(A5) 하나만 보면 43.8%** 입니다. 관문이 이걸 못 봤습니다.\n#:\n#: 그런데 여기에 숫자를 박을 **닻이 없습니다.** 전체 관문의 5% 는\n#: `STAGE1_MISS_RATE`(1단계가 이미 받아들이는 하향)에서 나왔는데, 클래스별로는\n#: 분모가 다릅니다 — 1단계의 \"A5 를 6% 놓친다\" 는 **A5 전체** 기준이고\n#: 43.8% 는 **말한 A5** 기준입니다. 둘을 비교하면 안 됩니다.\n#:\n#: 결과를 보고 문턱을 고르면 무슨 숫자가 나와도 성공담이 되므로(작업 규칙 2),\n#: **`over_triage` 와 같은 처방**을 씁니다 — 관문으로 쓰지 않고 **찍습니다.**\n#: `tests/test_granularity_gates.py` 가 이 값이 사라지지 않게 지킵니다.\nUNDER_TRIAGE_PER_CLASS_GATE = None\n\nDETECT_MIN_USABLE = 0.50\n\"\"\"★ **검출기가 제안한 네모**가 밴드 안에 들어가야 하는 비율 (STEP 42).\n\n`CAM_BOX_MIN_USABLE` 과 **같은 값을 씁니다** — 같은 질문이기 때문입니다:\n*\"이 제안을 믿고 크롭을 걸 수 있나.\"* 새 숫자를 만들면 갈래마다 다른 잣대를\n대는 셈이 됩니다.\n\n비교 기준선 (같은 밴드 · 같은 자극):\n\n    사람이 그린 네모        배율 안 10.0% · 위치 안 12.5% · **둘 다 5.0%**\n    분류기를 창 탐지기로     배율 안 39.0% · 위치 안 19.0% · **둘 다 6.7%**\n    검출기 (이번)          ?\n\n⚠️ **앞의 둘과 다른 점** — 창 탐지기는 분류기를 **빌려 쓴 것**이라 네모를\n뽑도록 배운 적이 없습니다. 이번엔 **bbox 라벨 36만 장으로 그 일을 직접**\n배웁니다. 지금까지 아홉 갈래가 닫혔는데 **이건 시도조차 안 해본 정공법**입니다.\n\n⚠️ 통과해도 **바로 채택이 아닙니다.** 검출기가 좋아도 그 네모로 자른 크롭에서\n2단계 커버리지가 올라야 의미가 있습니다 — `DETECT_MIN_COVERAGE_GAIN`.\"\"\"\n\nDETECT_MIN_COVERAGE_GAIN = 0.05\n\"\"\"★ 검출기 네모로 잘랐을 때 **계열 커버리지**가 올라야 하는 폭 (STEP 42).\n\n`FIXEDSCALE_MIN_COVERAGE_GAIN` 과 같은 값 — 같은 질문이고 메울 수 있는 폭도\n같습니다 (라벨 네모 50.3% − 사용자 네모 22.4% = **27.9%p**).\n\n⚠️ **위 `DETECT_MIN_USABLE` 만 통과하고 이게 미달이면 채택하지 않습니다.**\nSTEP 41 에서 배운 것 — **네모가 좋아 보여도 크롭이 걸리면 중심 오차가\n치명적이 될 수 있습니다.** 최종 판정은 항상 **제품 지표(커버리지)** 로 합니다.\"\"\"\n\nFIXEDSCALE_LESION_FRAC = 0.215\n\"\"\"★ **네모 크기를 버리고** 이 값으로 대체합니다 (화면 긴 변 대비, STEP 41).\n\n왜 — 사용자 네모는 **크기를 못 믿지만 중심은 어느 정도 맞습니다**\n(크기 상관 −0.05 vs 탭 오차 중앙값 0.111). 그런데 크기를 그대로 쓰면\n`m2.5` 창이 **100% 포화**돼 크롭이 아예 안 걸립니다 (STEP 38·40).\n\n    지금   네모 58.7% × 2.5 = 147%  → 잘려서 사진 전체\n    제안   네모 크기를 **버리고** 21.5% × 2.5 = 54%  → 크롭이 걸림\n\n**0.215 는 실측 병변 긴 변의 중앙값**입니다 (STEP 36, 자극 n=40 의 정답 bbox).\n⚠️ **평가 결과가 아니라 데이터의 성질**에서 왔습니다 — 결과를 보고 고른 값이\n아닙니다. 배포에서도 매니페스트로 알 수 있는 값입니다.\n\n⚠️ `f320` 과 다릅니다: 그건 **320px 고정**이고 그 크롭으로 배운 **약한 모델**\n입니다. 이건 **배율을 `m2.5` 가 기대하는 값으로 맞추고 기존 강한 모델**을 씁니다.\n**재학습 0회 · 추론 그대로 · 서빙 크롭 계산만** 바뀝니다.\"\"\"\n\nFIXEDSCALE_MIN_COVERAGE_GAIN = 0.05\n\"\"\"★ 위 방식을 채택할 문턱 — 계열 4군 커버리지 **+5%p 이상** (절대값).\n\n메울 수 있는 폭이 **27.9%p** 입니다 (라벨 네모 50.3% − 사용자 네모 22.4%).\n그 5분의 1도 못 메우면 서빙 코드를 바꿀 값이 없습니다.\n\n⚠️ **씨앗 둘에서 다 넘어야** 합니다. 커버리지는 작은 하락을 6배로 증폭하는\n문턱 지표라(STEP 39) **한 번 잘 나온 것을 믿으면 안 됩니다** — 오늘만 네 번\n당했습니다.\"\"\"\n\nNOBOX_MIN_GAIN = 0.03\n\"\"\"★ **네모를 아예 안 쓰는 학습**을 채택할 문턱 (STEP 40). 잡음 ±0.016 의 두 배.\n\n왜 이걸 묻나 — STEP 38 에서 **사용자 네모의 100% 가 `m2.5` 창을 포화**시키는\n걸 확인했습니다. 즉 **모델은 어차피 사진 전체를 봅니다.** 그런데 지금 2단계는\n*\"병변에 딱 맞는 크롭\"* 으로 배웠습니다 — **배우는 입력과 받는 입력이 다릅니다.**\n\n그러면 **아예 사진 전체로 배우면** 어떨까요. 이건 앞서 기각된 잡음 증강과\n다릅니다:\n\n    잡음 증강     엉성한 네모로 자른 것을 보여줌 (여전히 크롭)\n    네모 없음     **크롭을 아예 안 함** — 받는 입력 그대로\n\n    판정: **네모 없음으로 배워 네모 없이 평가한 값**이\n          **대조군(라벨 네모로 배워 사용자 네모로 평가)** 보다 +0.03 이상.\n\n⚠️ **대조군을 반드시 같이 돌립니다.** 기준 모델은 원본 1920px 크롭으로 배웠고\n평가는 1080 근접 파이프라인이라, **어떤** 파인튜닝이든 오릅니다 (STEP 38 에서\n분포 적응만으로 +0.109). 그걸 빼야 이 갈래의 순수 몫이 보입니다.\n\n⚠️ 통과해도 **바로 바꾸자가 아닙니다.** 앱이 네모를 계속 보내는 한 서버가\n그걸 무시해야 하고, 1단계는 여전히 중심을 씁니다. 그리고 VL01 기준입니다.\"\"\"\n\nBOXNOISE_MIN_GAIN = 0.03\n\"\"\"★ **네모 잡음 증강**을 채택할 문턱 (STEP 38). 잡음 ±0.016 의 두 배.\n\n왜 이걸 하나 — STEP 36 에서 두 가지가 확인됐습니다:\n\n    ① 사람은 병변의 기하를 못 줍니다 (크기 상관 −0.05 · 탭이 병변 안 47.5%)\n    ② 그 네모로 바꾸면 2단계가 **상대 22~24%** 무너집니다\n\n**사람을 못 고치면 모델을 고칩니다.** 학습 때부터 **사용자처럼 엉성한 네모**로\n잘라 보여주면, 배포에서 만나는 입력과 학습 입력이 같아집니다.\n\n    판정: **사용자 네모**에서 macro-F1 이 기준선 대비 +0.03 이상 오르면 채택.\n\n⚠️ **라벨 네모 성능이 무너지면 안 됩니다** — 같이 찍습니다. 한쪽만 보면\nSTEP 30 에서 배운 \"관문을 한쪽만 세우면 반대로 도망간다\" 를 또 합니다.\n허용: 라벨 네모 −0.03 까지 (`BOXNOISE_LABEL_DROP_MAX`).\n\n⚠️ 이건 **증강**이지 새 구조가 아닙니다. 크롭 함수도 모델도 그대로입니다 —\n학습이 보는 **네모만** 흔듭니다.\"\"\"\n\nBOXNOISE_LABEL_DROP_MAX = 0.03\n\"\"\"네모 잡음 증강이 **라벨 네모** 성능을 이만큼 넘게 깎으면 기각합니다.\"\"\"\n\nCAM_BOX_MIN_USABLE = 0.50\n\"\"\"★ **모델이 네모를 제안**할 때 밴드 안에 들어가야 하는 비율 (STEP 37).\n\n왜 이걸 묻나 — 사람에게 병변의 기하를 물어보는 두 길이 **둘 다 닫혔습니다**:\n\n    네모 크기   병변 크기와 **무관**하게 그림 (상관 −0.05, 정답의 2.80배)\n    점 위치     절반 이상이 병변 **밖** (탭이 병변 안에 든 것 47.5%)\n\n번진 병변(비듬·각질 · 태선화)에는 사람이 답할 수 있는 점도 크기도 **없습니다.**\nUI 를 잘 만들어서 될 문제가 아닙니다.\n\n남은 길은 **모델이 제안하고 사람은 판단만** 하는 것입니다 — 고르는 건 만드는\n것보다 쉽습니다. 다만 제안이 자주 틀리면 *\"아니요\"* 가 반복돼 아무도 안 씁니다.\n\n    판정: 제안 네모가 **배율·위치 밴드에 둘 다** 들어가는 비율이\n          **50% 이상**이어야 \"제안\" 이라 부를 수 있습니다.\n          절반을 넘게 틀리면 그건 제안이 아니라 **방해**입니다.\n\n비교 기준선 (STEP 36 실측, 같은 밴드·같은 자극):\n\n    사람이 그린 네모   배율 안 10.0% · 위치 안 12.5% · **둘 다 5.0%**\n\n⚠️ 밴드는 `config.ZOOM_ALLOW` / `ZOOM_CENTER_MAX` 에서 끌어 씁니다 — 여기 베껴\n적으면 출처가 바뀌어도 아무 일이 안 일어납니다 (촬영 밴드에서 당한 것).\"\"\"\n\nTAP_ERROR_MAX = 0.10\n\"\"\"★ **점 하나만 받는 설계**를 채택할 수 있는 탭 오차 상한 (STEP 36, 화면 대비).\n\n무엇을 묻나 — 사용자가 네모 **크기**를 못 맞춥니다 (상관 −0.05). 그런데\n`f320` 은 **중심만** 씁니다. 크기를 아예 안 물어보면 *\"뭘 기준으로 44%인지\n모르겠다\"* 는 문제가 통째로 사라집니다. 남는 건 **위치를 얼마나 정확히\n찍느냐** 하나입니다.\n\n시뮬레이션(n=1,200/점, VL01)에서 `m2.5` + 사용자 네모 = **0.3609** 기준:\n\n    탭 오차 0.02   f320 0.4521   +0.091   ← 라벨 네모 수준을 유지합니다\n    탭 오차 0.05   f320 0.4318   +0.071\n    탭 오차 0.10   f320 0.3830   +0.022   ← 잡음 ±0.016 보다 큼\n    탭 오차 0.161  f320 0.3363   −0.025   (이건 **네모 중심**이지 탭이 아닙니다)\n\n    판정: 실측 탭 오차 **중앙값 ≤ 0.10** 이면\n          \"위치만 받는 설계가 유리하다\" 로 봅니다.\n\n⚠️ **문턱을 결과 보기 전에 박습니다.** 위 표는 시뮬레이션이고 실제 탭은 아직\n안 쟀습니다 — 재고 나서 문턱을 고르면 무슨 값이 나와도 성공담이 됩니다.\n\n⚠️ 통과해도 **바로 바꾸자가 아닙니다.** 라벨 네모 기준으로는 `m2.5` 가 여전히\n앞섭니다 (0.4913 vs 0.4516) — **완벽한 네모를 받을 수 있으면 `m2.5` 가 낫고,\n못 받으니까 뒤집히는 것**입니다. 3팔 앙상블도 다시 짜야 합니다.\"\"\"\n\nUSER_BBOX_DROP_GAP_MIN = 0.03\n\"\"\"★ **사용자 네모로 바꿨을 때 두 크롭의 하락 차이** 문턱 (STEP 36).\n\n무엇을 묻나 — 지금까지 `m2.5`(크기를 씀) vs `f320`(중심만 씀) 비교는 전부\n**라벨 bbox**(정답에 딱 맞는 네모)로 했습니다 (STEP 22·23). 그런데 실측하니\n사람은 **병변 크기와 무관하게** 화면의 절반쯤으로 그립니다 (상관 −0.05,\n크기 중앙값 정답의 2.80배). 그러면 크기를 쓰는 크롭만 그 오차를 받습니다.\n\n    판정:  (m2.5 의 하락) − (f320 의 하락) ≥ 0.03  이면\n           **\"크기 의존이 실사용에서 실제로 해롭다\"** 로 봅니다.\n\n⚠️ 두 모델을 **같은 백본**(`effnetv2_s`)으로 씁니다. `convnextv2_base` 를 쓰면\n백본과 크롭이 섞여 못 가릅니다 (STEP 12 에서 배운 것).\n\n⚠️ 잡음보다 커야 합니다 — 같은 설정 재실행이 macro-F1 **±0.016** 입니다.\n0.03 은 그 두 배쯤입니다.\n\n⚠️ 이건 **채택 판정이 아닙니다.** 통과해도 \"2단계를 f320 으로 바꾸자\" 가 바로\n나오지 않습니다 — STEP 23 이 라벨 bbox 기준으로 기각한 근거가 그대로 살아\n있고, 실제 사용자 네모 분포는 n=40·한 사람에서 나온 것입니다.\"\"\"\n\nSTAGE1_NOBOX_MAX_RECALL_DROP = 0.01\n\"\"\"★ 가이드 프레임을 **없앱을 때** 1단계 recall 이 잃어도 되는 폭 (STEP 43).\n\n무엇을 묻나 — STEP 40 이 *\"가이드 프레임은 **2단계**에 아무 일도 안 한다\"* 를\n다섯 번 확인했습니다 (사용자 네모 = 네모 없음, 소수점까지 같음). 그럼 틀을\n없애도 되나 — **안 됩니다. 1단계 `f320` 은 중심을 씁니다.** 틀을 없애면\n중심이 화면 한가운데로 가고, 그게 recall 을 얼마나 깎는지를 안 재 봤습니다.\n\n★ **닿** — 이 0.01 은 제가 고른 것이 아니라 `STAGE1_MISS_RATE`(0.057)에서\n나옵니다. 그것은 **제품이 이미 받아들인 놀침**이고, 2단계 관문\n`UNDER_TRIAGE_MAX`(0.05)이 거기서 나왔습니다 (0.05 < 0.057). recall 이 1%p 더\n떨어지면 놀침이 **6.7%** 가 돼 그 닿이 깨집니다.\n\n⚠️ 1단계의 놀침은 *\"괜찮아 보여요\"* 로 나가 **병원에 아예 안 가게** 만듭니다.\n이름이 틀리는 것보다 훨씬 나쁘므로 관문은 recall 하나로 겁니다.\n\n⚠️ **AUROC 와 협알림률은 관문이 아니고 좀지만 합니다** (`over_triage` 와 같은\n처방). 한쪽만 세면 반대쪽으로 도망갑니다.\"\"\"\n\nSTAGE1_NOBOX_MIN_CENTER_OFF = 0.05\n\"\"\"★ **자극 검사** — 시뮬레이션 사진에서 병변이 화면 중앙에서 떨어져 있는 정도.\n\n이 값이 0 에 가까우면 *\"틀 없음(화면 한가운데)\"* 조건이 **공짜로 이깁니다** —\n병변이 원래 가운데 있으니까요. 그러면 무슨 결과가 나와도 재려는 것을 안 재 것입니다.\n\nSTEP 36 에서 똑같은 함정에 30분을 버렸고(`m2.5` 크롭은 정답을 가운데 놓고\n자른 것), STEP 24 의 `occ` 사건도 같은 모양입니다. **표본 수 검사보다 먼저**\n봅니다 — 못 가르는 검사는 표본이 아무리 많아도 못 가릅니다.\"\"\"\n\nSTAGE1_NOBOX_N = 3000\n\"\"\"1차로 돌릴 장수. 방향이 안 보이면 전체를 안 돕니다.\"\"\"\n\nUSER_BBOX_SIM_N = 3000\n\"\"\"1차로 돌릴 장수. 방향이 안 보이면 전체를 안 돕니다 (실측: 3,000장 약 3분).\"\"\"\n\n\ndef user_bbox_report(name: str, label_f1: float, user_f1: float) -> dict:\n    \"\"\"라벨 bbox → 사용자 bbox 로 바꿨을 때의 하락. **판정은 부르는 쪽에서.**\"\"\"\n    drop = float(label_f1) - float(user_f1)\n    print(f\"  {name:22} 라벨 {label_f1:.4f} → 사용자 {user_f1:.4f}\"\n          f\"   하락 {drop:+.4f}\")\n    return {\"crop\": name, \"label_f1\": float(label_f1),\n            \"user_f1\": float(user_f1), \"drop\": drop}\n\n\nARM_SAME_COVERAGE_TOL = 0.02\n\"\"\"★ **같은 커버리지로 맞춰 재봤을 때** 두 설정의 차이를 '같다' 로 볼 폭.\n\nSTEP 34 가 A6 경보를 푼 방법이고 STEP 35 가 A5 에 다시 쓴 방법입니다.\n**비율은 분모가 바뀌면 같이 바뀝니다** — 말을 더 하기로 하면 한계 표본이\n섞여 들어와 비율이 나빠 보입니다. 그걸 모델 탓으로 읽으면 안 됩니다.\n\n실측(STEP 35, holdout): 앙상블 A5 하향이 릴리스 단독 37.4% 대비 43.8% 라\n나빠 보였는데, **같은 장수(20,121)만 말하게 자르니 38.0%** 였습니다.\n차이 +0.6%p 로 이 폭 안 → 모델이 아니라 **선택 효과**입니다.\"\"\"\n\nA5_RULE_UNDER_GAIN_MIN = 0.05\nA5_RULE_COVERAGE_LOSS_MAX = 0.03\n\"\"\"하향 방지 규칙의 채택 기준 (STEP 35, **돌리기 전에** 박음).\n\n    A5 하향이 5%p 이상 줄고  AND  전체 커버리지 손실이 3%p 이하\n\n⚠️ 문턱은 **val 에서 고르고 holdout 은 확인만** 합니다. holdout 에서 고르면\n그건 판정이 아니라 맞춤입니다.\"\"\"\n\nGRANULARITY_MIN_COVERAGE = NAMING_MIN_COVERAGE\n\"\"\"알갱이를 굵게 해서 얻은 커버리지에도 **같은 문턱**을 씁니다 (50%).\n굵게 말한다고 문턱을 낮추면 무슨 묶음이든 통과합니다.\"\"\"\n\n\ndef granularity_report(name: str, rows: dict, *, target=NAMING_TARGET_ERROR) -> dict:\n    \"\"\"알갱이 하나에 대한 커버리지 + **긴급도 하향** 관문.\n\n    `rows` 에 필요한 것 (전부 같은 길이·같은 순서, 1단계가 넘긴 사진만):\n\n        conf        확신도 (내림차순으로 말할 것을 고릅니다)\n        wrong       그 알갱이 기준으로 틀렸는가 (정상 사진은 항상 True)\n        tier_true   실제 긴급도 등급 (정상 사진은 -1 — 하향이 성립 안 함)\n        tier_said   우리가 말한 것의 긴급도 등급\n\n    선택:\n\n        true_class  실제 클래스/묶음 이름 — 주면 **클래스별 하향**을 같이 찍습니다\n                    (STEP 35: 전체 3.7% 가 통과하는데 A5 하나가 43.8% 였습니다)\n    \"\"\"\n    import numpy as np\n\n    conf = np.asarray(rows[\"conf\"], dtype=float)\n    wrong = np.asarray(rows[\"wrong\"], dtype=bool)\n    tt = np.asarray(rows[\"tier_true\"], dtype=int)\n    ts = np.asarray(rows[\"tier_said\"], dtype=int)\n    n = len(conf)\n    if not (len(wrong) == len(tt) == len(ts) == n):\n        raise ValueError(\"네 배열의 길이가 다릅니다\")\n\n    order = np.argsort(-conf)\n    err = np.cumsum(wrong[order]) / np.arange(1, n + 1)\n    ok = np.flatnonzero(err <= target)\n    k = int(ok[-1]) + 1 if len(ok) else 0\n    cov = k / n if k else 0.0\n\n    spoken = order[:k]\n    # 긴급도 하향 = 실제 등급이 말한 등급보다 높음. 정상 사진(-1)은 제외합니다\n    # (거기엔 낮출 긴급도가 없습니다 — 그건 헛알림 문제이고 1단계 몫입니다).\n    real = tt[spoken] >= 0\n    under = float((tt[spoken][real] > ts[spoken][real]).mean()) if real.any() else 0.0\n    # ★ 반대쪽도 **셉니다 — 관문으로 쓰진 않고**. 과잉(안 급한 걸 급하다고)은\n    #   병원에 가게 만들 뿐이라 위험하지 않습니다. 그런데 **공짜도 아닙니다**:\n    #   굵게 묶고 묶음의 긴급도를 높은 쪽으로 잡으면 하향은 0 에 수렴하는 대신\n    #   과잉이 치솟고, 그러면 아무도 그 말을 안 믿게 됩니다.\n    #   ⚠️ 이 값을 안 찍던 동안 권고안(4군)의 과잉이 **49.3%** 인 걸 몰랐습니다.\n    over = float((tt[spoken][real] < ts[spoken][real]).mean()) if real.any() else 0.0\n\n    # ★ 클래스별 하향 (STEP 35). **관문이 아니라 찍기만** 합니다 —\n    #   `UNDER_TRIAGE_PER_CLASS_GATE` 주석에 왜 문턱을 안 거는지 적어뒀습니다.\n    #   ⚠️ 전체 평균은 분모가 커서 한 클래스가 나빠도 묻힙니다.\n    by_cls: dict[str, float] = {}\n    worst: tuple[str, float] | None = None\n    if rows.get(\"true_class\") is not None:\n        tc = np.asarray(rows[\"true_class\"])\n        if len(tc) != n:\n            raise ValueError(\"true_class 길이가 다릅니다\")\n        for c in sorted(set(tc[spoken][real].tolist())):\n            m = spoken[real][tc[spoken][real] == c]\n            sel = (tc[spoken] == c) & real\n            if int(sel.sum()) < UNDER_TRIAGE_PER_CLASS_MIN_N:\n                continue\n            v = float((tt[spoken][sel] > ts[spoken][sel]).mean())\n            by_cls[str(c)] = v\n            if worst is None or v > worst[1]:\n                worst = (str(c), v)\n\n    passed = cov >= GRANULARITY_MIN_COVERAGE and under <= UNDER_TRIAGE_MAX\n    verdict = (\"논의할 가치 있음\" if passed else\n               f\"기각 (커버리지 {cov:.1%}\" +\n               (f\" · 긴급도 하향 {under:.1%}\" if under > UNDER_TRIAGE_MAX else \"\") + \")\")\n\n    tail = f\"   최악 {worst[0]} {worst[1]:.1%}\" if worst else \"\"\n    print(f\"  {name:22} 커버리지 {cov:>6.1%}   하향 {under:>6.1%}\"\n          f\"   과잉 {over:>6.1%}{tail}   {'통과' if passed else '기각'}\")\n    return {\"granularity\": name, \"coverage\": cov, \"n_spoken\": k,\n            \"under_triage\": under, \"over_triage\": over, \"verdict\": verdict,\n            \"under_by_class\": by_cls,\n            \"under_worst_class\": worst[0] if worst else None,\n            \"under_worst\": worst[1] if worst else None,\n            \"threshold\": float(conf[order][k - 1]) if k else float(\"inf\")}\n", "src/env.py": "\"\"\"실행 환경(Colab / Kaggle / 로컬) 자동 감지 + 경로·시크릿 통합.\n\n노트북 첫 셀에서 이것만 부르면 나머지 코드는 환경을 몰라도 됩니다.\n\n    from src import env\n    E = env.describe()          # 환경 요약 출력\n    ROOT = env.data_root()      # 데이터가 놓일 곳\n    KEY  = env.secret(\"AIHUB_API_KEY\")\n\n⚠️ API 키를 코드나 노트북에 하드코딩하지 마세요.\n   Colab  : 왼쪽 사이드바 🔑 Secrets 에 AIHUB_API_KEY 등록\n   Kaggle : Add-ons → Secrets 에 AIHUB_API_KEY 등록\n\"\"\"\n\nfrom __future__ import annotations\n\nimport os\nimport shutil\nimport subprocess\nimport sys\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Literal\n\nEnvName = Literal[\"colab\", \"kaggle\", \"local\"]\n\n\ndef _fix_console_encoding() -> None:\n    \"\"\"Windows 콘솔에서 한글·이모지 출력이 죽지 않게 합니다.\n\n    한국어 Windows 의 cmd 는 기본 코드페이지가 cp949 라서, 이 프로젝트가 쓰는\n    ✅ ⚠️ ★ ─ 같은 문자를 인코딩하지 못하고 UnicodeEncodeError 로 죽습니다.\n    출력 스트림을 UTF-8 로 바꾸고, 그래도 못 찍는 글자는 대체 문자로 넘깁니다\n    (로그 한 줄 때문에 몇십 분짜리 전처리가 죽는 것보다 낫습니다).\n    \"\"\"\n    if os.name != \"nt\":\n        return\n    try:\n        import ctypes\n\n        ctypes.windll.kernel32.SetConsoleOutputCP(65001)   # UTF-8 코드페이지\n    except Exception:\n        pass\n    for stream in (sys.stdout, sys.stderr):\n        try:\n            stream.reconfigure(encoding=\"utf-8\", errors=\"replace\")\n        except Exception:\n            pass\n\n\n_fix_console_encoding()\n\n# GPU 이름 → 대략적인 VRAM(GB). 배치 크기 자동 추천에만 씁니다.\n_VRAM_HINT = {\n    \"T4\": 16, \"P100\": 16, \"V100\": 16, \"L4\": 24, \"A100\": 40,\n    \"A10\": 24, \"RTX 3090\": 24, \"RTX 4090\": 24, \"H100\": 80,\n}\n\n\n# ──────────────────────────────────────────────────────────────\n# 환경 감지\n# ──────────────────────────────────────────────────────────────\ndef detect() -> EnvName:\n    \"\"\"현재 실행 환경을 반환합니다.\n\n    ⚠️ **Kaggle 을 먼저 봅니다.** 예전에는 `\"google.colab\" in sys.modules` 를\n       가장 먼저 봤는데, Kaggle 이미지에도 `google-colab` 패키지와 `/content`\n       가 있어서 **Kaggle 세션이 Colab 으로 오판**됐습니다. 그러면 노트북이\n       `drive.mount()` 를 부르고, Kaggle 에는 실제 Colab VM 이 없으니\n       `NotImplementedError` 로 죽습니다.\n\n       `google.colab` 을 import 할 수 있다는 것과 **Colab VM 위에 있다는 것은\n       다릅니다.** 진짜 Colab VM 의 표식은 `/var/colab/hostname` 입니다 —\n       구글 자신의 `drive.mount()` 도 이걸로 판정합니다.\n    \"\"\"\n    # 1) Kaggle — 가장 확실한 신호부터\n    if os.environ.get(\"KAGGLE_KERNEL_RUN_TYPE\") or os.environ.get(\"KAGGLE_URL_BASE\"):\n        return \"kaggle\"\n    if Path(\"/kaggle/working\").is_dir() or Path(\"/kaggle/input\").is_dir():\n        return \"kaggle\"\n\n    # 2) 진짜 Colab VM\n    if Path(\"/var/colab/hostname\").exists() or os.environ.get(\"COLAB_RELEASE_TAG\"):\n        return \"colab\"\n    # 위 표식이 없는데 google.colab 이 **이미 로드돼 있으면** Colab 계열로 봅니다\n    # (마운트는 못 할 수 있고, mount_drive 가 알아서 넘어갑니다)\n    if \"google.colab\" in sys.modules and Path(\"/content\").is_dir():\n        return \"colab\"\n\n    return \"local\"\n\n\ndef can_mount_drive() -> bool:\n    \"\"\"Drive 를 실제로 마운트할 수 있는 환경인가.\n\n    구글의 `drive.mount()` 가 쓰는 표식과 같은 것을 봅니다. 이게 False 인데\n    마운트를 시도하면 NotImplementedError 가 납니다.\n    \"\"\"\n    return is_colab() and Path(\"/var/colab/hostname\").exists()\n\n\ndef diagnose() -> dict:\n    \"\"\"환경 감지가 이상할 때 근거를 전부 찍습니다.\n\n        env.diagnose()\n\n    \"Kaggle 인데 Colab 이라고 나온다\" 같은 상황에서 뭘 보고 그렇게 판단했는지\n    확인할 수 있습니다.\n    \"\"\"\n    sig = {\n        \"detect()\": detect(),\n        \"KAGGLE_KERNEL_RUN_TYPE\": os.environ.get(\"KAGGLE_KERNEL_RUN_TYPE\"),\n        \"KAGGLE_URL_BASE\": os.environ.get(\"KAGGLE_URL_BASE\"),\n        \"COLAB_RELEASE_TAG\": os.environ.get(\"COLAB_RELEASE_TAG\"),\n        \"/kaggle/working 있음\": Path(\"/kaggle/working\").is_dir(),\n        \"/kaggle/input 있음\": Path(\"/kaggle/input\").is_dir(),\n        \"/content 있음\": Path(\"/content\").is_dir(),\n        \"/var/colab/hostname 있음\": Path(\"/var/colab/hostname\").exists(),\n        \"google.colab 로드됨\": \"google.colab\" in sys.modules,\n        \"can_mount_drive()\": can_mount_drive(),\n        \"workspace()\": str(workspace()),\n        \"work_root()\": str(work_root()),\n        \"persist_root()\": str(persist_root() or \"None\"),\n    }\n    print(\"── 환경 감지 근거 \" + \"─\" * 40)\n    for k, v in sig.items():\n        print(f\"  {k:<26} {v}\")\n    print(\"─\" * 58)\n    return sig\n\n\ndef is_colab() -> bool:\n    return detect() == \"colab\"\n\n\ndef is_kaggle() -> bool:\n    return detect() == \"kaggle\"\n\n\n# ──────────────────────────────────────────────────────────────\n# 경로\n# ──────────────────────────────────────────────────────────────\ndef project_root() -> Path:\n    \"\"\"리포지토리 루트(= 이 파일의 부모의 부모).\"\"\"\n    return Path(__file__).resolve().parent.parent\n\n\ndef workspace() -> Path:\n    \"\"\"쓰기 가능한 작업 루트. 환경마다 다릅니다.\"\"\"\n    e = detect()\n    if e == \"colab\":\n        return Path(\"/content\")\n    if e == \"kaggle\":\n        # /kaggle/input 은 읽기 전용이므로 working 을 씁니다.\n        return Path(\"/kaggle/working\")\n    return project_root()\n\n\ndef persist_root() -> Path | None:\n    \"\"\"**세션이 끊겨도 살아남는** 저장소. 없으면 None.\n\n    ⚠️ Colab 의 `/content` 는 휘발성입니다. 세션이 끊기면 체크포인트까지 전부\n       사라집니다. 90분짜리 학습이 80분에 끊기면 처음부터 다시입니다.\n\n    그래서 체크포인트는 Drive 로 복사해 둡니다. Drive 가 마운트돼 있어야 하므로\n    `env.mount_drive()` 를 먼저 부르세요 (노트북 첫 셀이 합니다).\n\n    환경변수 `DOG_SKIN_PERSIST` 로 직접 지정할 수 있습니다.\n    \"\"\"\n    override = os.environ.get(\"DOG_SKIN_PERSIST\")\n    if override:\n        p = Path(override)\n        p.mkdir(parents=True, exist_ok=True)\n        return p\n\n    if is_colab():\n        drive = Path(\"/content/drive/MyDrive\")\n        if drive.exists():\n            p = drive / \"dogskin_work\"\n            p.mkdir(parents=True, exist_ok=True)\n            return p\n        return None            # Drive 미마운트 — 호출부가 경고합니다\n\n    if detect() == \"kaggle\":\n        # Kaggle 은 /kaggle/working 이 세션 종료 시 출력으로 보존됩니다\n        return workspace()\n\n    # 로컬은 애초에 휘발성이 아닙니다\n    return work_root()\n\n\ndef data_root() -> Path:\n    \"\"\"원본 데이터(압축 해제본)가 놓일 곳. 환경변수 DOG_SKIN_DATA 로 덮어쓸 수 있습니다.\"\"\"\n    override = os.environ.get(\"DOG_SKIN_DATA\")\n    if override:\n        return Path(override)\n    return workspace() / \"data\" / \"raw\"\n\n\ndef work_root() -> Path:\n    \"\"\"전처리 산출물(크롭 이미지, 매니페스트, 체크포인트)이 놓일 곳.\"\"\"\n    override = os.environ.get(\"DOG_SKIN_WORK\")\n    if override:\n        return Path(override)\n    return workspace() / \"data\" / \"work\"\n\n\ndef ensure_dirs() -> dict[str, Path]:\n    \"\"\"필요한 디렉터리를 만들고 경로 사전을 돌려줍니다.\"\"\"\n    paths = {\n        \"data_root\": data_root(),\n        \"work_root\": work_root(),\n        \"manifests\": work_root() / \"manifests\",\n        \"crops\": work_root() / \"crops\",\n        \"checkpoints\": work_root() / \"checkpoints\",\n        \"reports\": work_root() / \"reports\",\n    }\n    for p in paths.values():\n        p.mkdir(parents=True, exist_ok=True)\n    return paths\n\n\ndef mount_drive(mountpoint: str = \"/content/drive\", strict: bool = False) -> Path | None:\n    \"\"\"Colab 에서만 Google Drive 를 마운트합니다. 다른 환경에서는 None.\n\n    ⚠️ **실패해도 예외를 던지지 않습니다.** Drive 는 체크포인트를 세션 밖에\n       남기기 위한 **편의 기능**이지, 학습의 전제가 아닙니다. 여기서 죽으면\n       노트북 전체가 멈추는데, 정작 데이터가 다른 곳에 있으면 그냥 진행하면\n       됩니다. 못 붙었다는 사실은 호출부(`persist_root()` 가 None)가 알립니다.\n\n    실제로 겪은 경우: `/var/colab/hostname` 이 없는 Colab 계열 환경\n    (로컬 런타임 / Colab Enterprise / 일부 프록시 세션)에서\n    `NotImplementedError: Mounting drive is unsupported in this environment`.\n\n    strict=True 로 주면 예외를 그대로 올립니다.\n    \"\"\"\n    if not is_colab():\n        print(\"[env] Colab 이 아니므로 Drive 마운트를 건너뜁니다.\")\n        return None\n    try:\n        from google.colab import drive  # type: ignore[import-not-found]\n\n        drive.mount(mountpoint)\n    except Exception as exc:                                    # noqa: BLE001\n        if strict:\n            raise\n        print(f\"⚠️ [env] Drive 마운트 실패 — {type(exc).__name__}: \"\n              f\"{str(exc).splitlines()[0][:120]}\")\n        print(\"   이 환경에서는 Drive 를 쓸 수 없습니다. **계속 진행할 수 있습니다.**\")\n        print(\"   다만 두 가지가 달라집니다:\")\n        print(\"     · 데이터를 Drive 에서 못 읽습니다 → zip 을 다른 경로에 두거나 Kaggle 사용\")\n        print(\"     · 체크포인트가 세션 밖에 안 남습니다 → 끊기면 학습을 처음부터\")\n        print(\"   체크포인트만 살리려면: os.environ['DOG_SKIN_PERSIST'] = '/어딘가/영구경로'\")\n        return None\n    p = Path(mountpoint) / \"MyDrive\"\n    return p if p.exists() else None\n\n\n# ──────────────────────────────────────────────────────────────\n# 시크릿\n# ──────────────────────────────────────────────────────────────\ndef secret(name: str, required: bool = True) -> str | None:\n    \"\"\"환경에 맞는 시크릿 저장소에서 값을 읽습니다.\n\n    우선순위: 환경변수 → Colab userdata → Kaggle UserSecrets\n    \"\"\"\n    val = os.environ.get(name)\n    if val:\n        return val\n\n    e = detect()\n    if e == \"colab\":\n        try:\n            from google.colab import userdata  # type: ignore[import-not-found]\n\n            val = userdata.get(name)\n        except Exception as exc:  # 미등록 / 접근 거부 모두 여기로\n            val = None\n            _hint = f\"({type(exc).__name__})\"\n        if val:\n            return val\n    elif e == \"kaggle\":\n        try:\n            from kaggle_secrets import UserSecretsClient  # type: ignore[import-not-found]\n\n            val = UserSecretsClient().get_secret(name)\n        except Exception:\n            val = None\n        if val:\n            return val\n\n    if required:\n        raise RuntimeError(\n            f\"시크릿 '{name}' 을(를) 찾을 수 없습니다.\\n\"\n            f\"  현재 환경: {e}\\n\"\n            \"  Colab  → 왼쪽 사이드바 🔑 Secrets 에 추가 후 '노트북 액세스' 토글 ON\\n\"\n            \"  Kaggle → Add-ons → Secrets 에 추가 후 이 노트북에 Attach\\n\"\n            f\"  로컬   → export {name}=...\\n\"\n            \"  ⚠️ 절대 코드나 노트북 셀에 직접 붙여넣지 마세요.\"\n        )\n    return None\n\n\n# ──────────────────────────────────────────────────────────────\n# 하드웨어\n# ──────────────────────────────────────────────────────────────\n@dataclass\nclass Device:\n    kind: str  # \"cuda\" | \"mps\" | \"cpu\"\n    name: str = \"CPU\"\n    vram_gb: float = 0.0\n    bf16: bool = False       # 네이티브 bf16 (Ampere 8.0+). 에뮬레이션은 False 로 둡니다.\n    count: int = 0\n    capability: str = \"\"\n\n    @property\n    def amp_dtype(self) -> str:\n        \"\"\"AMP(혼합정밀) 에 쓸 dtype. bf16 이 되면 bf16 이 안전합니다.\"\"\"\n        if self.kind != \"cuda\":\n            return \"none\"\n        return \"bfloat16\" if self.bf16 else \"float16\"\n\n\ndef device_info() -> Device:\n    try:\n        import torch\n    except ImportError:\n        return Device(kind=\"cpu\")\n\n    if torch.cuda.is_available():\n        props = torch.cuda.get_device_properties(0)\n        # ⚠️ torch.cuda.is_bf16_supported() 는 소프트웨어 에뮬레이션까지 True 로 칩니다.\n        #    T4(Turing, 7.5)에서 True 가 나오는데, 실제로 bf16 을 쓰면 fp16 보다 훨씬 느립니다.\n        #    네이티브 bf16 은 Ampere(8.0) 이상에만 있으므로 compute capability 로 판정합니다.\n        real_bf16 = props.major >= 8\n        return Device(\n            kind=\"cuda\",\n            name=props.name,\n            vram_gb=round(props.total_memory / 1024**3, 1),\n            bf16=real_bf16,\n            count=torch.cuda.device_count(),\n            capability=f\"{props.major}.{props.minor}\",\n        )\n    if getattr(torch.backends, \"mps\", None) and torch.backends.mps.is_available():\n        return Device(kind=\"mps\", name=\"Apple MPS\")\n    return Device(kind=\"cpu\")\n\n\ndef suggest_batch_size(img_size: int, model_scale: str = \"base\",\n                       mem_factor: float = 1.0) -> int:\n    \"\"\"VRAM 과 입력 해상도로 배치 크기를 대충 추천합니다.\n\n    보수적으로 잡습니다 — OOM 으로 3시간짜리 학습이 죽는 것보다\n    배치가 작아 조금 느린 편이 낫습니다. 부족하면 config 에서 직접 올리세요.\n    \"\"\"\n    dev = device_info()\n    if dev.kind != \"cuda\":\n        return 8\n    vram = dev.vram_gb or _VRAM_HINT.get(dev.name.split()[-1], 16)\n\n    # 224px / base 모델 / 16GB 를 기준점 48 로 두고 스케일링\n    # (ResNet50@224 를 AMP 로 돌리면 배치 48 이 8GB 남짓입니다. 여전히 보수적입니다)\n    scale_factor = {\"tiny\": 2.0, \"small\": 1.4, \"base\": 1.0, \"large\": 0.5}.get(model_scale, 1.0)\n    px_factor = (224 / max(img_size, 64)) ** 2\n    # mem_factor: 백본별 보정 (ModelSpec.mem_factor). 이 공식은 ResNet 기준이라\n    # 어텐션 계열의 활성값 메모리를 과소평가합니다 — 실측으로 0.4 를 씁니다.\n    bs = 48 * (vram / 16) * scale_factor * px_factor * mem_factor\n\n    # ⚠️ 2의 거듭제곱으로 내림하면 최대 절반을 버립니다.\n    #    실제로 T4(14.7GB)에서 계산값 44 가 32 도 아닌 **16** 이 됐습니다\n    #    (이전 기준점 32 × 0.92 = 29.4 → 16). 학습이 2~3배 느려집니다.\n    #    중간 단계를 둔 사다리로 내림합니다.\n    ladder = [4, 8, 12, 16, 24, 32, 48, 64, 96, 128, 160, 192, 256]\n    out = ladder[0]\n    for v in ladder:\n        if v <= bs:\n            out = v\n    return out\n\n\ndef require_gpu(hard: bool = True) -> Device:\n    \"\"\"GPU 가 붙어 있는지 확인하고, 없으면 **크게** 알립니다.\n\n    왜 필요한가: GPU 가 없어도 코드는 그냥 돕니다 — 다만 20~30배 느립니다.\n    `DEV = \"cuda\" if torch.cuda.is_available() else \"cpu\"` 는 조용히 CPU 로\n    떨어지고, `suggest_batch_size` 도 8 을 돌려주기 때문에 겉보기엔 정상입니다.\n    실측: 검증 7,751장에 GPU 1~2분 vs CPU **40분**. 학습은 며칠입니다.\n\n    Colab 무료 티어는 GPU 사용량 한도를 넘기면 **말없이 CPU 런타임을 줍니다.**\n    그래서 사람이 알아채기 전에 몇 시간을 버리게 됩니다.\n    \"\"\"\n    d = device_info()\n    if d.kind == \"cuda\":\n        return d\n\n    msg = (\n        \"\\n\" + \"🚨\" * 20 + \"\\n\"\n        \"  GPU 가 없습니다 — 지금 학습/추론하면 20~30배 느립니다.\\n\"\n        + \"🚨\" * 20 + \"\\n\\n\"\n        \"  실측 비교 (검증 7,751장):\\n\"\n        \"     GPU(T4) 1~2분   ↔   CPU 약 40분\\n\"\n        \"     학습은 CPU 로 며칠 걸립니다. 기다리지 마세요.\\n\\n\"\n        \"  · Colab  : 런타임 → 런타임 유형 변경 → T4 GPU → 저장\\n\"\n        \"             이미 GPU 로 되어 있는데 이 메시지가 뜬다면\\n\"\n        \"             **무료 GPU 한도를 다 쓴 것**입니다 (보통 12~24시간 뒤 회복).\\n\"\n        \"             → Kaggle 로 옮기거나(주 30시간 별도) Colab Pro 를 보세요.\\n\"\n        \"  · Kaggle : 우측 Settings → Accelerator → GPU T4 x2\\n\\n\"\n        \"  그래도 CPU 로 진행하려면: env.require_gpu(hard=False)\\n\"\n    )\n    if hard:\n        raise RuntimeError(msg)\n    print(msg)\n    return d\n\n\ndef suggest_workers() -> int:\n    \"\"\"DataLoader 워커 수. CPU 코어에 맞춥니다.\n\n    왜 중요한가: 512px JPEG 을 디코딩+리사이즈하는 건 CPU 일입니다.\n    워커가 부족하면 GPU 가 놀면서 데이터를 기다립니다 — 배치를 키워도 안 빨라집니다.\n    실측: Colab T4 에서 74 img/s 였는데, ResNet50@224 AMP 의 계산 능력은\n          150~200 img/s 입니다. 절반 이상을 데이터 로딩에서 흘리고 있었습니다.\n    \"\"\"\n    import os\n    import sys\n\n    # 환경변수로 강제 (튜닝·디버깅용). 0 이면 메인 프로세스에서 로딩합니다.\n    forced = os.environ.get(\"DOG_SKIN_WORKERS\")\n    if forced is not None and forced.strip().lstrip(\"-\").isdigit():\n        return max(0, int(forced))\n\n    n = os.cpu_count() or 2\n\n    # ⚠️ **윈도우는 다릅니다.** 리눅스(Colab/Kaggle)는 `fork` 라 워커가 부모\n    #    메모리를 그대로 물려받지만, 윈도우는 `spawn` 이라 **워커마다 torch·\n    #    timm·src 를 통째로 다시 import** 합니다. 게다가 이 리포는 train 로더와\n    #    val 로더를 **매 에폭 새로** 만들어서(persistent_workers 안 씀) 에폭마다\n    #    16번을 다시 띄웁니다.\n    #    2026-09-05 실측: RTX 3050 + 워커 8 로 1단계를 돌리다 **교착**했습니다.\n    #    GPU 사용률 97% → 1%, 메모리는 3.6GB 를 잡은 채 CPU 시간이 584.2초에서\n    #    멈췄고(8초 뒤에도 같은 값), 워커 13개가 뜬 상태였습니다.\n    #    → 윈도우에서는 적게 씁니다. 데이터 로딩이 조금 느려도 도는 게 낫습니다.\n    if sys.platform == \"win32\":\n        return min(2, max(n - 1, 0))\n\n    # 메인 프로세스 몫을 남기고, 너무 많으면 오히려 컨텍스트 스위칭 비용이 큽니다\n    return max(2, min(n - 1, 8))\n\n\ndef free_disk_gb(path: Path | None = None) -> float:\n    p = path or workspace()\n    p.mkdir(parents=True, exist_ok=True)\n    return round(shutil.disk_usage(p).free / 1024**3, 1)\n\n\n# ──────────────────────────────────────────────────────────────\n# 요약 출력\n# ──────────────────────────────────────────────────────────────\n@dataclass\nclass EnvSummary:\n    env: EnvName\n    python: str\n    torch: str\n    device: Device\n    paths: dict[str, Path] = field(default_factory=dict)\n    free_disk_gb: float = 0.0\n\n\ndef describe(verbose: bool = True) -> EnvSummary:\n    \"\"\"환경을 감지하고 디렉터리를 만든 뒤 요약을 출력합니다.\"\"\"\n    try:\n        import torch\n\n        tv = torch.__version__\n    except ImportError:\n        tv = \"(미설치)\"\n\n    s = EnvSummary(\n        env=detect(),\n        python=sys.version.split()[0],\n        torch=tv,\n        device=device_info(),\n        paths=ensure_dirs(),\n        free_disk_gb=free_disk_gb(),\n    )\n\n    if verbose:\n        d = s.device\n        print(\"─\" * 58)\n        print(f\" 실행 환경   : {s.env}\")\n        print(f\" Python      : {s.python}   |  torch {s.torch}\")\n        if d.kind == \"cuda\":\n            amp = \"bf16\" if d.bf16 else \"fp16 + GradScaler\"\n            print(f\" GPU         : {d.name}  {d.vram_gb}GB  x{d.count}  \"\n                  f\"(sm_{d.capability}, AMP={amp})\")\n        elif s.env in (\"colab\", \"kaggle\"):\n            print(f\" GPU         : 🚨 없음 ({d.kind})\")\n            print(\" \" * 15 + \"학습·추론이 20~30배 느려집니다. 지금 멈추세요.\")\n            print(\" \" * 15 + \"(런타임 유형이 이미 GPU 인데도 이러면 무료 한도 소진입니다)\")\n        else:\n            # 로컬: 전처리·패키징은 CPU 로 하는 게 맞습니다\n            print(f\" GPU         : 없음 ({d.kind}) — 전처리·패키징에는 필요 없습니다\")\n        print(f\" 여유 디스크 : {s.free_disk_gb} GB\")\n        print(f\" 원본 데이터 : {s.paths['data_root']}\")\n        print(f\" 작업 폴더   : {s.paths['work_root']}\")\n        print(\"─\" * 58)\n        if s.env == \"colab\" and s.free_disk_gb < 60:\n            print(\"⚠️  디스크 여유가 적습니다. STEP 1 에서 부분 다운로드를 꼭 쓰세요.\")\n        # ⚠️ 로컬 PC 는 GPU 가 없는 게 정상입니다 (다운로드·전처리는 CPU 작업).\n        #    거기서까지 경고하면 진짜 경고까지 같이 무시하게 됩니다.\n        if d.kind == \"cpu\" and s.env in (\"colab\", \"kaggle\"):\n            print(\"⚠️  GPU 런타임이 아닙니다 — 학습·추론이 20~30배 느립니다.\")\n            print(\"    Colab : 런타임 → 런타임 유형 변경 → T4 GPU\")\n            print(\"    Kaggle: 우측 Settings → Accelerator → GPU T4 x2\")\n    return s\n\n\ndef _search_roots() -> list[Path]:\n    \"\"\"전처리 결과를 찾아볼 곳.\n\n    ⚠️ `/workspace` 는 **RunPod 등 임대 GPU** 의 표준 마운트입니다. 이게 없어서\n       런팟에서 `load_prepared()` 가 바로 FileNotFoundError 로 죽었습니다.\n       `DOG_SKIN_PREPARED` 로 직접 지정할 수도 있습니다.\n    \"\"\"\n    roots = []\n    override = os.environ.get(\"DOG_SKIN_PREPARED\")\n    if override:\n        roots.append(Path(override))\n    roots += [Path(\"/kaggle/input\"), Path(\"/content/drive/MyDrive\"),\n              Path(\"/content\"), Path(\"/workspace\"), Path.cwd()]\n    seen, out = set(), []\n    for p in roots:\n        if p.exists() and p.resolve() not in seen:\n            seen.add(p.resolve())\n            out.append(p)\n    return out\n\n\n#: 우리가 쓰는 크롭 태그. `crops/` 층 없이 올라온 폴더를 알아볼 때만 씁니다\nCROP_TAGS = (\"m2.5\", \"m1.5\", \"f320\", \"full\")\n\n\ndef _tag_has_jpg(t: Path) -> bool:\n    try:\n        return t.is_dir() and next(t.rglob(\"*.jpg\"), None) is not None\n    except OSError:\n        return False\n\n\ndef _crops_dir(d: Path) -> Path | None:\n    \"\"\"`d` 안에서 **태그 폴더들을 담고 있는 폴더**를 찾습니다. 없으면 None.\n\n    ⚠️ 캐글 데이터셋을 만들 때 `crops/` 층이 사라지는 일이 있습니다.\n       `data/work/crops/m2.5` 를 그대로 올리면 데이터셋 안이\n       `<데이터셋>/m2.5/00/....jpg` 가 됩니다 — `crops/` 가 없습니다.\n       `d/crops` 만 보면 **붙여놓고도** \"전처리 결과를 찾지 못했습니다\" 로\n       죽습니다 (실제로 당했습니다). 그래서 `d` 자신도 봅니다.\n\n       단 `d` 자신을 볼 때는 **이름이 아는 태그인 것만** 인정합니다. 아무\n       폴더나 태그로 받으면 남의 데이터셋이 크롭으로 잡힙니다.\n    \"\"\"\n    c = d / \"crops\"\n    try:\n        if c.is_dir() and any(_tag_has_jpg(t) for t in c.iterdir()):\n            return c\n    except OSError:\n        pass\n    try:\n        if d.is_dir() and any(t.name in CROP_TAGS and _tag_has_jpg(t) for t in d.iterdir()):\n            return d\n    except OSError:\n        pass\n    return None\n\n\ndef _has_crops(d: Path) -> bool:\n    \"\"\"크롭이 **한 장이라도** 들어 있는 태그 폴더가 있는가.\"\"\"\n    return _crops_dir(d) is not None\n\n\ndef _manifest_files(d: Path) -> list[Path]:\n    \"\"\"`d` **바로 아래**의 매니페스트 파일들 (parquet · csv).\"\"\"\n    try:\n        return sorted(p for p in d.iterdir()\n                      if p.is_file() and not p.name.startswith(\".\")\n                      and p.suffix in (\".parquet\", \".csv\")\n                      and \"manifest\" in p.name.lower())\n    except OSError:\n        return []\n\n\ndef _manifests_dir(d: Path, depth: int = 5) -> Path | None:\n    \"\"\"`d` 안에서 **매니페스트 파일이 실제로 들어 있는 폴더**를 찾습니다.\n\n    ⚠️ `d / \"manifests\"` 로 못 박지 마세요. 캐글 데이터셋을 만들면 층이\n       하나 늘거나(`manifests/manifests/*.parquet`,\n       `manifests/data/work/manifests/*.parquet`) 아예 없어져\n       (`<데이터셋>/manifest_final.parquet`) 있는 일이 실제로 있습니다.\n       예전 코드는 `manifests/` **폴더만** 보고 \"매니페스트 복사\" 를 찍은 뒤\n       0개를 복사했습니다 — 8분 뒤 다음 셀에서\n       `FileNotFoundError: manifest_final.parquet` 로 죽었습니다.\n\n    크롭 태그 폴더로는 안 내려갑니다 (36만 장짜리라 훑으면 몇 분 걸립니다).\n    \"\"\"\n    skip = set(CROP_TAGS) | {\"crops\", \"reports\", \"checkpoints\",\n                             \"__pycache__\", \"lost+found\"}\n    frontier = [d / \"manifests\", d]\n    seen: set[Path] = set()\n    for _ in range(depth):\n        nxt: list[Path] = []\n        for p in frontier:\n            if not p.is_dir():\n                continue\n            try:\n                r = p.resolve()\n            except OSError:\n                continue\n            if r in seen:\n                continue\n            seen.add(r)\n            if _manifest_files(p):\n                return p\n            try:\n                nxt += [c for c in sorted(p.iterdir())\n                        if c.is_dir() and c.name not in skip\n                        and not c.name.startswith(\".\")]\n            except OSError:\n                continue\n        if not nxt:\n            break\n        frontier = nxt\n    return None\n\n\ndef _manifest_zip(d: Path, depth: int = 5, max_gb: float = 3.0) -> tuple[Path | None, list[str]]:\n    \"\"\"매니페스트가 들어 있는 **zip** 과 그 안의 항목들. 없으면 `(None, [])`.\n\n    ⚠️ **캐글이 업로드한 zip 을 늘 풀어주는 건 아닙니다.** 실제로 크롭 zip 은\n       풀려서 `m2.5/00/*.jpg` 가 됐는데 `manifests.zip` 은 그대로 남아\n       있었습니다. 폴더만 보면 \"매니페스트 없음\" 으로 죽습니다.\n\n    크롭 zip(12GB 대)은 크기로 걸러냅니다 — 거기엔 매니페스트가 없고,\n    네트워크 마운트에서 목차를 읽으면 느립니다.\n    \"\"\"\n    import zipfile\n\n    skip = set(CROP_TAGS) | {\"crops\", \"reports\", \"checkpoints\",\n                             \"__pycache__\", \"lost+found\"}\n    frontier, seen = [d], set()\n    for _ in range(depth):\n        nxt: list[Path] = []\n        for p in frontier:\n            if not p.is_dir():\n                continue\n            try:\n                r = p.resolve()\n            except OSError:\n                continue\n            if r in seen:\n                continue\n            seen.add(r)\n            try:\n                items = sorted(p.iterdir())\n            except OSError:\n                continue\n            for q in items:\n                if q.is_dir():\n                    if q.name not in skip and not q.name.startswith(\".\"):\n                        nxt.append(q)\n                    continue\n                if q.suffix.lower() != \".zip\":\n                    continue\n                # ⚠️ 통짜 `dogskin*.zip` 은 건드리지 않습니다 — 그건 아래\n                #    \"압축 해제\" 경로가 통째로 풉니다. 여기서 가로채면\n                #    크롭이 안 풀린 채 매니페스트만 옵니다.\n                if q.name.lower().startswith(\"dogskin\"):\n                    continue\n                try:\n                    if q.stat().st_size > max_gb * 1024**3:\n                        continue                    # 크롭 zip — 여기엔 없습니다\n                    with zipfile.ZipFile(q) as f:\n                        names = [n for n in f.namelist()\n                                 if not n.endswith(\"/\")\n                                 and \"manifest\" in n.rsplit(\"/\", 1)[-1].lower()\n                                 and n.lower().endswith((\".parquet\", \".csv\"))]\n                except (OSError, zipfile.BadZipFile):\n                    continue\n                if names:\n                    return q, names\n        frontier = nxt\n        if not frontier:\n            break\n    return None, []\n\n\ndef _bring_manifests(src: Path, man: Path) -> int:\n    \"\"\"`src` 안의 매니페스트를 작업 폴더로 가져옵니다. 가져온 **개수**를 돌려줍니다.\n\n    폴더로 올렸든 zip 으로 올렸든 둘 다 받습니다.\n    \"\"\"\n    import zipfile\n\n    d = _manifests_dir(src)\n    if d is not None:\n        man.mkdir(parents=True, exist_ok=True)\n        n = 0\n        for f in sorted(d.iterdir()):\n            if f.is_file():\n                shutil.copy2(f, man / f.name)\n                n += 1\n        tail = d.name if d != src else \"(최상위)\"\n        print(f\"[env] 매니페스트 {n}개 복사: {src.name}/{tail} → {man}\")\n        return n\n\n    z, members = _manifest_zip(src)\n    if z is not None:\n        man.mkdir(parents=True, exist_ok=True)\n        # ⚠️ 폴더 구조는 버리고 **파일만** 꺼냅니다. zip 안이\n        #    `data/work/manifests/manifest_final.parquet` 이어도\n        #    작업 폴더에는 `manifests/manifest_final.parquet` 이어야 합니다.\n        with zipfile.ZipFile(z) as f:\n            for m in members:\n                with f.open(m) as r, open(man / m.rsplit(\"/\", 1)[-1], \"wb\") as w:\n                    shutil.copyfileobj(r, w)\n        print(f\"[env] 매니페스트 {len(members)}개 풀기: {z.name} → {man}\")\n        return len(members)\n\n    print(f\"[env] {src.name} 안에 매니페스트가 없습니다 (크롭만 있는 입력)\")\n    return 0\n\n\ndef _has_manifest(d: Path) -> bool:\n    return _manifests_dir(d) is not None\n\n\ndef _looks_prepared(d: Path) -> bool:\n    \"\"\"크롭과 매니페스트가 **내용까지** 있는가.\n\n    ⚠️ 폴더 존재만 보면 안 됩니다. `ensure_dirs()` 가 `crops/` 와 `manifests/`\n       를 **빈 폴더로 미리 만듭니다.** 노트북 첫 셀의 `env.describe()` 가 그걸\n       부르므로, 존재만 보면 zip 을 받아놓고도 \"이미 준비됨\" 으로 건너뛰고\n       나중에 `manifest_final.parquet` 을 못 찾아 죽습니다 (런팟에서 당했습니다).\n    \"\"\"\n    return _has_crops(d) and _has_manifest(d)\n\n\ndef _looks_partial(d: Path) -> bool:\n    \"\"\"크롭만 있고 매니페스트는 없는 폴더 — 태그를 나눠 올린 경우.\"\"\"\n    return _has_crops(d) and not _has_manifest(d)\n\n\ndef _looks_manifest_only(d: Path) -> bool:\n    \"\"\"크롭 없이 **매니페스트만** 올린 폴더.\n\n    크롭 데이터셋은 그대로 두고 매니페스트만 따로 올려 붙이는 길을 열어둡니다\n    (크롭은 12GB 라 다시 올리는 데 몇 시간 걸립니다). 폴더든 zip 이든 받습니다.\n    \"\"\"\n    if _has_crops(d):\n        return False\n    return _manifests_dir(d, depth=2) is not None or _manifest_zip(d, depth=2)[0] is not None\n\n\ndef find_prepared(dest: Path | None = None) -> tuple[Path, str]:\n    \"\"\"전처리 결과를 하나 찾습니다. `(경로, \"zip\" | \"dir\")`.\"\"\"\n    src, kind = find_prepared_all(dest)[0]\n    return src, kind\n\n\ndef find_prepared_all(dest: Path | None = None) -> list[tuple[Path, str]]:\n    \"\"\"전처리 결과를 **전부** 찾습니다. `[(경로, \"zip\" | \"dir\"), ...]`.\n\n    ⚠️ **Kaggle 은 업로드한 zip 을 데이터셋에 넣을 때 자동으로 풀어버립니다.**\n       그래서 `/kaggle/input/<데이터셋>/crops/...` 만 있고 zip 은 없습니다.\n       zip 만 찾으면 여기서 막힙니다 — 풀려 있는 폴더도 같이 봅니다.\n\n    ⚠️ 전체 zip 은 5GB 를 넘습니다. 업로드가 자주 끊겨서 **태그별로 나눠**\n       올리는 경우가 있습니다 (`crops/m1.5` 하나, `crops/full` 하나).\n       그래서 하나만 찾고 멈추지 않고 다 모아서 합칩니다.\n    \"\"\"\n    dest = Path(dest) if dest else work_root()\n    zips: list[tuple[Path, str]] = []\n    dirs: list[tuple[Path, str]] = []\n    seen: set[Path] = set()\n\n    for base in _search_roots():\n        for d in _walk(base, depth=5):\n            r = d.resolve()\n            # zip 은 이 폴더 바로 아래만 봅니다 (rglob 은 심볼릭 링크로 안 들어갑니다)\n            for z in sorted(d.glob(\"dogskin*.zip\")):\n                if z.resolve() not in seen:\n                    seen.add(z.resolve())\n                    zips.append((z, \"zip\"))\n            if r in seen or r == dest.resolve():\n                continue\n            if _looks_prepared(d) or _looks_partial(d) or _looks_manifest_only(d):\n                seen.add(r)\n                dirs.append((d, \"dir\"))\n\n    out = dirs or zips              # 풀려 있는 쪽을 우선 (복사 없이 링크만 하면 됨)\n    if not out:\n        raise FileNotFoundError(\n            \"전처리 결과(dogskin_prepared)를 찾지 못했습니다.\\n\\n\"\n            + _what_is_there()\n            + \"\\n찾는 것 (둘 중 하나):\\n\"\n            \"  · dogskin*.zip 파일\\n\"\n            \"  · crops/ 폴더를 가진 폴더 (Kaggle 은 zip 을 자동으로 풀어둡니다)\\n\"\n            \"  · 또는 m2.5/f320/full/m1.5 를 바로 담은 폴더 (crops/ 층 없이 올린 데이터셋)\\n\\n\"\n            \"확인할 것:\\n\"\n            \"  · Kaggle : 우측 패널 [Add Input] 으로 데이터셋을 붙였는지\\n\"\n            \"             (붙였으면 위 목록의 /kaggle/input 아래에 보여야 합니다)\\n\"\n            \"  · Colab  : Drive 를 마운트했는지 → env.mount_drive()\\n\"\n            \"  · 경로를 직접 주려면 env.load_prepared('/kaggle/input/<데이터셋이름>')\"\n        )\n    return out\n\n\n# 데이터가 들어 있을 리 없는데 크고 느린 폴더들 — 여기로 내려가면 탐색이 오래 걸립니다\n_SKIP_DIRS = {\"crops\", \"manifests\", \"reports\", \"checkpoints\", \".git\",\n              \"__pycache__\", \"node_modules\", \"site-packages\", \"lost+found\"}\n\n\ndef _walk(base: Path, depth: int = 5, max_dirs: int = 3000) -> list[Path]:\n    \"\"\"base 아래를 depth 단계까지. **심볼릭 링크 폴더도 들어갑니다.**\n\n    ⚠️ `Path.rglob` 은 심볼릭 링크인 하위 폴더로 들어가지 않습니다.\n       Kaggle 의 `/kaggle/input/<데이터셋>` 이 링크면 rglob 으로는 안 보입니다.\n       `iterdir()` + `is_dir()` 은 링크를 따라가므로 직접 훑습니다.\n\n    ⚠️ Kaggle 의 데이터셋 경로가 생각보다 깊습니다. 실측:\n           /kaggle/input/datasets/<사용자>/<데이터셋>/crops/...\n       `/kaggle/input` 기준으로 **4단계**입니다. 얕게 보면 못 찾습니다.\n\n    크롭 폴더(4만 장) 안으로는 내려가지 않습니다 — 거기엔 찾을 게 없고 느립니다.\n    \"\"\"\n    out = [base]\n    frontier = [base]\n    for _ in range(depth):\n        nxt: list[Path] = []\n        for d in frontier:\n            if len(out) >= max_dirs:\n                return out\n            try:\n                for p in sorted(d.iterdir()):\n                    if not p.is_dir() or p.name in _SKIP_DIRS or p.name.startswith(\".\"):\n                        continue\n                    out.append(p)\n                    # 크롭이 실제로 들어 있으면 더 내려갈 이유가 없습니다.\n                    #\n                    # ⚠️ **`_looks_manifest_only` 로는 멈추면 안 됩니다.** 그 판정은\n                    #    자식 폴더 안까지(depth=2) 훑어서 parquet 을 찾습니다. 그래서\n                    #    데이터셋 **여러 개를 담고 있는 부모 폴더**가 걸립니다 —\n                    #    캐글은 `/kaggle/input/datasets/<계정>/<데이터셋>` 이라\n                    #    `<계정>` 폴더가 매니페스트 데이터셋 때문에 매치됩니다.\n                    #    거기서 멈추면 **형제인 크롭 데이터셋을 아예 안 봅니다.**\n                    #    실제로 당했습니다 (2026-09-04): 크롭을 붙여놨는데\n                    #    `찾은 입력: [('/kaggle/input/datasets/gayoniee', 'dir')]`\n                    #    하나만 나오고 \"사용 가능한 태그: []\" 로 죽었습니다.\n                    if not (_looks_prepared(p) or _looks_partial(p)):\n                        nxt.append(p)\n            except OSError:\n                continue\n        if not nxt:\n            break\n        frontier = nxt\n    return out\n\n\ndef _what_is_there(max_items: int = 12) -> str:\n    \"\"\"검색 경로에 **실제로 뭐가 있는지** 보여줍니다.\n\n    \"못 찾았다\"만 알려주면 사용자가 다음에 뭘 볼지 알 수 없습니다.\n    \"\"\"\n    lines = [\"실제로 있는 것:\"]\n    for base in _search_roots():\n        lines.append(f\"  {base}\")\n        try:\n            items = sorted(base.iterdir())\n        except OSError as exc:\n            lines.append(f\"      (읽을 수 없음: {type(exc).__name__})\")\n            continue\n        if not items:\n            lines.append(\"      (비어 있음)\")\n            continue\n        for p in items[:max_items]:\n            if p.is_dir():\n                try:\n                    inner = sorted(x.name for x in p.iterdir())[:6]\n                except OSError:\n                    inner = [\"(읽을 수 없음)\"]\n                lines.append(f\"      📁 {p.name}/   → {inner}\")\n            else:\n                mb = p.stat().st_size / 1024**2\n                lines.append(f\"      📄 {p.name}  ({mb:,.0f} MB)\")\n        if len(items) > max_items:\n            lines.append(f\"      … 외 {len(items) - max_items}개\")\n    return \"\\n\".join(lines) + \"\\n\"\n\n\ndef _peek(d: Path, depth: int = 3, max_items: int = 10) -> list[str]:\n    \"\"\"`d` 안을 몇 층까지 훑어 사람이 읽을 줄로 만듭니다 (진단용).\n\n    크롭 태그 폴더로는 안 들어갑니다 — 36만 장을 세면 몇 분 걸립니다.\n    \"\"\"\n    skip = set(CROP_TAGS) | {\"crops\"}\n    out: list[str] = []\n\n    def walk(p: Path, level: int, pad: str) -> None:\n        if level > depth:\n            return\n        try:\n            items = sorted(p.iterdir())\n        except OSError as exc:\n            out.append(f\"{pad}(읽을 수 없음: {type(exc).__name__})\")\n            return\n        if not items:\n            out.append(f\"{pad}(비어 있음)\")\n            return\n        for q in items[:max_items]:\n            if q.is_dir():\n                mark = \" …\" if q.name in skip else \"/\"\n                out.append(f\"{pad}📁 {q.name}{mark}\")\n                if q.name not in skip:\n                    walk(q, level + 1, pad + \"   \")\n            else:\n                mb = q.stat().st_size / 1024**2\n                out.append(f\"{pad}📄 {q.name}  ({mb:,.1f} MB)\")\n        if len(items) > max_items:\n            out.append(f\"{pad}… 외 {len(items) - max_items}개\")\n\n    walk(d, 1, \"\")\n    return out\n\n\ndef _unwrap_tag_dir(t: Path, max_depth: int = 3) -> Path:\n    \"\"\"태그 폴더 안이 한 겹 더 싸여 있으면 **진짜 층까지 내려갑니다.**\n\n    크롭의 실제 모양은 `<태그>/<hh>/<이름>.jpg` 입니다 (hh = 해시 앞 두 자리,\n    256개). 그런데 캐글에 올릴 때 폴더를 한 겹 더 감싸면\n    `<태그>/<태그>/<hh>/*.jpg` 가 됩니다 — 실제로 이렇게 올라갔습니다.\n\n    ⚠️ 이걸 안 풀면 **링크는 걸리는데 경로가 한 칸씩 어긋납니다.**\n       `crops/m2.5/ab/x.jpg` 를 찾는데 실제로는 `m2.5/m2.5/ab/x.jpg` 라\n       `switch_tag` 가 \"0/365,428장 존재 (0.0%)\" 로 죽습니다. 링크가 걸렸으니\n       \"붙었다\" 고 보이는데 한 장도 안 읽힙니다 — 제일 나쁜 종류의 실패입니다.\n    \"\"\"\n    cur = t\n    for _ in range(max_depth):\n        if _has_shard_jpg(cur):\n            return cur\n        try:\n            subs = [q for q in cur.iterdir() if q.is_dir()]\n        except OSError:\n            break\n        if len(subs) != 1:            # 갈래가 여럿이면 여기가 맞는 층입니다\n            break\n        cur = subs[0]\n    return cur if _has_shard_jpg(cur) else t\n\n\ndef _has_shard_jpg(d: Path, probe: int = 8) -> bool:\n    \"\"\"`d` 바로 아래 폴더들(`<hh>`) 안에 jpg 가 있는가.\n\n    256개를 다 뒤지지 않습니다 — 네트워크 마운트에서는 왕복이 비쌉니다.\n    \"\"\"\n    try:\n        n = 0\n        for sub in d.iterdir():\n            if not sub.is_dir():\n                continue\n            if next(sub.glob(\"*.jpg\"), None) is not None:\n                return True\n            n += 1\n            if n >= probe:\n                break\n    except OSError:\n        pass\n    return False\n\n\ndef _link_tags(src_crops: Path, dst_crops: Path) -> dict[str, str]:\n    \"\"\"크롭을 **태그 단위로** 연결합니다. 여러 입력을 합칠 수 있습니다.\n\n    Kaggle 의 `/kaggle/input` 은 읽기 전용이고, 크롭 45,885장을\n    `/kaggle/working`(20GB 제한) 으로 복사하는 건 시간도 용량도 낭비입니다.\n    폴더 통째로가 아니라 태그별로 링크해야 `crops/m1.5` 와 `crops/full` 을\n    **서로 다른 데이터셋에서** 가져와 합칠 수 있습니다.\n    \"\"\"\n    dst_crops.mkdir(parents=True, exist_ok=True)\n    out: dict[str, str] = {}\n    # ⚠️ `crops/` 층 없이 올라온 폴더(`_crops_dir` 이 폴더 자신을 돌려준 경우)에는\n    #    `manifests/` 같은 형제 폴더가 같이 있습니다. 이름으로 걸러내지 않으면\n    #    매니페스트가 **크롭 태그로 링크**됩니다.\n    for tag_dir in sorted(p for p in src_crops.iterdir()\n                          if p.is_dir() and p.name not in _SKIP_DIRS):\n        # ⚠️ **빈 태그 폴더는 건너뜁니다.** 노트북 출력을 데이터셋으로 만들면\n        #    work/crops/ 의 심볼릭 링크가 빈 폴더로 남는 일이 있습니다.\n        #    먼저 연결된 쪽이 이기므로, 그 빈 폴더가 진짜 크롭 데이터셋을 가로막고\n        #    \"크롭이 0% 밖에 없습니다\" 로 죽습니다. 이름 순서 운에 맡길 일이 아닙니다.\n        if next(tag_dir.iterdir(), None) is None:\n            out[tag_dir.name] = \"비어 있어 건너뜀\"\n            continue\n        # ★ `<태그>/<태그>/<hh>/*.jpg` 처럼 한 겹 더 싸여 있으면 풀어서 연결합니다\n        inner = _unwrap_tag_dir(tag_dir)\n        name = tag_dir.name\n        dst = dst_crops / name\n        tag_dir = inner\n        if dst.is_symlink():\n            out[name] = (\"이미 연결됨\" if dst.resolve() == tag_dir.resolve()\n                         else \"다른 곳에 연결됨(유지)\")\n            continue\n        if dst.exists():\n            out[name] = \"이미 있음\"\n            continue\n        try:\n            dst.symlink_to(tag_dir.resolve(), target_is_directory=True)\n            out[name] = \"링크\"\n        except OSError:\n            shutil.copytree(tag_dir, dst)\n            out[name] = \"복사\"\n    return out\n\n\ndef load_prepared(\n    zip_path: str | Path | None = None,\n    dest: Path | None = None,\n    force: bool = False,\n) -> Path:\n    \"\"\"로컬에서 만든 전처리 결과를 클라우드 작업 폴더에 붙입니다.\n\n    한국 PC 에서 `prepare_local.py` 로 전처리한 결과를 Drive/Kaggle 에 올린 뒤,\n    학습 노트북 첫 부분에서 이걸 부르면 됩니다. 두 가지 형태를 다 받습니다:\n\n      · `dogskin_prepared.zip`  → 로컬 디스크로 풉니다 (Colab + Drive)\n      · 이미 풀린 `crops/ manifests/` 폴더 → 링크만 겁니다 (Kaggle)\n\n    ⚠️ zip 은 반드시 **로컬 디스크**로 풉니다. Drive 에 마운트된 채로 이미지를\n       읽으면 네트워크 왕복 때문에 학습이 10배 가까이 느려집니다.\n    \"\"\"\n    import zipfile\n\n    dest = Path(dest) if dest else work_root()\n    dest.mkdir(parents=True, exist_ok=True)\n\n    # ★ 이미 work_root 에 크롭+매니페스트가 있으면 **할 일이 없습니다.**\n    #    ⚠️ 이 검사는 find_prepared_all() **앞에** 있어야 합니다. 그 함수는\n    #       dest 자신을 후보에서 빼기 때문에(아래 `r == dest.resolve()`),\n    #       런팟처럼 데이터를 work_root 에 직접 넣은 경우 \"못 찾았습니다\" 로\n    #       **먼저 죽습니다.** 뒤에 두면 이 줄에 영영 도달하지 못합니다.\n    if zip_path is None and _looks_prepared(dest):\n        n = sum(1 for _ in (dest / \"crops\").iterdir() if _.is_dir())\n        print(f\"[env] 이미 준비돼 있습니다: {dest}  (크롭 태그 {n}종) — 그대로 씁니다.\")\n        return dest\n\n    if zip_path is None:\n        sources = find_prepared_all(dest)\n    elif isinstance(zip_path, (list, tuple)):\n        sources = [(Path(p), \"zip\" if Path(p).suffix == \".zip\" else \"dir\") for p in zip_path]\n    else:\n        p = Path(zip_path)\n        sources = [(p, \"zip\" if p.suffix == \".zip\" else \"dir\")]\n\n    if len(sources) > 1:\n        print(f\"[env] 입력 {len(sources)}개를 합칩니다: {[s.name for s, _ in sources]}\")\n\n    marker = dest / \".prepared_from\"\n    seen_before = set(marker.read_text().splitlines()) if marker.exists() else set()\n    now: list[str] = []\n\n    for src, kind in sources:\n        # 마커에 **여러 줄**로 적습니다. zip 을 두 개 올린 경우\n        # 한 줄만 쓰면 재실행 때 앞의 zip 을 또 풉니다.\n        already = str(src) in seen_before and not force and kind == \"zip\"\n\n        if kind == \"dir\":\n            # 읽기 전용일 수 있으므로 크롭은 태그별 링크, 매니페스트는 복사\n            # ⚠️ `src / \"crops\"` 로 못 박지 마세요 — 캐글 데이터셋은 `crops/` 층이\n            #    빠진 채 올라올 수 있습니다 (`_crops_dir` 주석 참고).\n            src_crops = _crops_dir(src)\n            if src_crops is not None:\n                where = src.name if src_crops == src else f\"{src.name}/crops\"\n                how = _link_tags(src_crops, dest / \"crops\")\n                for tag, act in how.items():\n                    print(f\"[env] 크롭 {act}: {where}/{tag}\")\n            # ⚠️ `ensure_dirs()` 가 work_root()/manifests 를 **빈 폴더로 미리 만듭니다.**\n            #    \"없으면 복사\" 로 조건을 걸면 그 빈 폴더 때문에 영원히 복사가 안 되고,\n            #    나중에 manifest_final.parquet 을 못 찾아 죽습니다. 비어 있으면 채웁니다.\n            man = dest / \"manifests\"\n            if force and man.exists() and not man.is_symlink():\n                shutil.rmtree(man)\n            # ⚠️ `ensure_dirs()` 가 work_root()/manifests 를 **빈 폴더로 미리\n            #    만듭니다.** \"폴더가 없으면\" 으로 조건을 걸면 영영 안 가져옵니다.\n            #    `manifest_final` 이 이미 있을 때만 건너뜁니다.\n            have = sorted(man.glob(\"*.parquet\")) if man.exists() else []\n            if not any(\"final\" in p.name for p in have):\n                _bring_manifests(src, man)\n        elif already:\n            print(f\"[env] 이미 풀려 있습니다: {dest}  (다시 풀려면 force=True)\")\n        else:\n            size = src.stat().st_size / 1024**3\n            print(f\"[env] 압축 해제 {src.name} ({size:.2f}GB) → {dest}\")\n            print(\"      (Drive 에서 직접 읽지 않고 로컬 디스크로 풉니다 — 학습 속도 때문)\")\n            with zipfile.ZipFile(src) as z:\n                z.extractall(dest)\n        now.append(str(src))\n\n    marker.write_text(\"\\n\".join(dict.fromkeys(now)))\n\n    crops = dest / \"crops\"\n    mans = sorted((dest / \"manifests\").glob(\"*.parquet\")) if (dest / \"manifests\").exists() else []\n\n    # ★ 매니페스트가 없으면 **여기서 멈춥니다.** 크롭 세기 전에요.\n    #\n    # ⚠️ 예전에는 경고만 찍고 진행했습니다. 크롭 36만 장을 세는 데 13분이\n    #    걸리고, 그 뒤 다음 셀이 `FileNotFoundError: manifest_final.parquet` 로\n    #    죽었습니다 — 20분을 버리고, 정작 원인을 알려주는 줄은 스크롤 위로\n    #    밀려 올라가 보이지도 않았습니다. 실제로 두 번 당했습니다.\n    #    **예외 메시지에 데이터셋 안을 같이 넣습니다** — 사람들이 복사해 오는\n    #    건 로그 꼬리(=트레이스백)뿐이라, 거기 없으면 전달되지 않습니다.\n    if not any(\"final\" in m.name for m in mans):\n        found = [m.name for m in mans] or [\"(하나도 없음)\"]\n        lines = [\n            \"manifest_final.parquet 이 없습니다. (크롭은 붙었는데 라벨이 없습니다)\",\n            \"\",\n            f\"작업 폴더 {dest / 'manifests'} 에 있는 것: {found}\",\n            \"\",\n            \"붙인 입력 안을 열어봤습니다 ↓ — 여기에 parquet 이 안 보이면\",\n            \"데이터셋에 매니페스트가 **안 들어간 것**입니다:\",\n        ]\n        for s, kind in sources:\n            lines.append(f\"  ── {s}  ({kind}) ──\")\n            if kind == \"dir\":\n                lines += [f\"     {t}\" for t in _peek(s)]\n            else:\n                lines.append(\"     (zip)\")\n        lines += [\n            \"\",\n            \"할 일 — 둘 중 하나:\",\n            \"  · manifest_final.parquet 을 Private 데이터셋으로 따로 올려\",\n            \"    [Add Input] 으로 붙이세요. 크롭 데이터셋은 그대로 두면 됩니다\",\n            \"    (매니페스트는 100MB 안팎이라 몇 분이면 올라갑니다)\",\n            \"  · 이미 붙였는데 위 목록에 안 보이면 업로드가 덜 끝난 것입니다\",\n        ]\n        raise FileNotFoundError(\"\\n\".join(lines))\n    # ⚠️ `crops.rglob()` 은 **심볼릭 링크 하위 폴더로 들어가지 않습니다.**\n    #    Kaggle 에서는 태그마다 링크를 걸므로 여기서 0장이 나옵니다.\n    #    태그 폴더를 하나씩(= 링크 자체를 시작점으로) 훑어야 합니다.\n    tags = sorted(p.name for p in crops.iterdir() if p.is_dir()) if crops.exists() else []\n\n    # ⚠️ 태그별로 세어 보여줍니다. 하나만 덜 올라간 경우가 실제로 있었습니다\n    #    (full 30% / m1.5 100%) — 합계만 보면 안 보입니다.\n    #\n    # ⚠️ **한 번만 훑습니다.** 예전에는 합계용으로 한 번, 태그별로 또 한 번 훑어서\n    #    같은 일을 두 번 했습니다. Kaggle 입력은 네트워크 마운트라 9만 장을 세는 데\n    #    8분 30초가 걸렸고, 그동안 아무 출력이 없어 멈춘 것처럼 보였습니다.\n    per_tag: dict[str, int] = {}\n    for t in tags:\n        print(f\"[env] 크롭 세는 중 … {t}\", flush=True)\n        per_tag[t] = sum(1 for _ in (crops / t).rglob(\"*.jpg\"))\n        print(f\"[env]   {t}: {per_tag[t]:,}장\", flush=True)\n    n_crop = sum(per_tag.values())\n    print(f\"[env] 크롭 {n_crop:,}장, 태그별 {per_tag}\")\n    if len(per_tag) > 1:\n        hi = max(per_tag.values())\n        short = {t: n for t, n in per_tag.items() if n < hi * 0.95}\n        if short:\n            print(f\"🚨 태그마다 장수가 다릅니다 — {short} (가장 많은 태그: {hi:,}장)\")\n            print(\"   업로드가 덜 끝났을 가능성이 큽니다. 이대로 쓰면 그 태그를 쓰는\")\n            print(\"   단계만 **부분 데이터**로 학습되고, 숫자를 다른 실행과 비교할 수 없습니다.\")\n    print(f\"[env] 매니페스트 {[m.name for m in mans]}\")\n    if n_crop == 0:\n        print(\"⚠️ 크롭이 하나도 없습니다. 데이터셋 내용을 확인하세요.\")\n    return dest\n\n\ndef set_seed(seed: int = 42, deterministic: bool = False) -> None:\n    \"\"\"재현성을 위한 시드 고정.\n\n    deterministic=True 면 cuDNN 을 결정론 모드로 두는데, 느려집니다.\n    최종 결과 재현이 필요할 때만 켜세요.\n    \"\"\"\n    import random\n\n    random.seed(seed)\n    os.environ[\"PYTHONHASHSEED\"] = str(seed)\n    try:\n        import numpy as np\n\n        np.random.seed(seed)\n    except ImportError:\n        pass\n    try:\n        import torch\n\n        torch.manual_seed(seed)\n        torch.cuda.manual_seed_all(seed)\n        if deterministic:\n            torch.backends.cudnn.deterministic = True\n            torch.backends.cudnn.benchmark = False\n        else:\n            torch.backends.cudnn.benchmark = True\n    except ImportError:\n        pass\n\n\ndef pip_install(packages: str = \"timm imagehash pyarrow grad-cam\", quiet: bool = True) -> None:\n    \"\"\"노트북에서 필요한 패키지를 설치합니다 (이미 있으면 빠르게 통과).\"\"\"\n    flags = [\"-q\"] if quiet else []\n    subprocess.run(\n        [sys.executable, \"-m\", \"pip\", \"install\", *flags, *packages.split()],\n        check=False,\n    )\n"}

for name, source in FILES.items():
    path = CODE / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source)
sys.path.insert(0, str(CODE))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers>=4.52'], check=True)
import torch
assert torch.cuda.is_available(), 'GPU T4 를 고르세요'
print(torch.__version__, torch.cuda.get_device_name(0))
T0 = time.monotonic()


In [ ]:
import pandas as pd, json as _json
IN = Path('/kaggle/input')
labels = next(IN.rglob('boxes_multi.parquet'))
les = pd.read_parquet(labels); les['boxes'] = les.boxes.map(_json.loads)
normals = pd.concat([pd.read_parquet(p) for p in IN.rglob('boxes.parquet') if 'normals' in str(p)]).drop_duplicates('image')
normals = normals[normals.label == 'A7'].copy(); normals['boxes'] = [[] for _ in range(len(normals))]
index = {p.name: p for p in IN.rglob('*.jpg')}
df = pd.concat([les[['image', 'boxes', 'label', 'group', 'fold']], normals[['image', 'boxes', 'label', 'group', 'fold']]], ignore_index=True)
df = df[df.image.isin(index)].reset_index(drop=True)
print(f'병변 {int((df.label != "A7").sum()):,} · 정상 {int((df.label == "A7").sum()):,} · 사진 파일 {len(index):,}')
assert (df.label != 'A7').sum() > 140000 and (df.label == 'A7').sum() > 140000, '조각이 빠졌습니다 — Dataset 6개를 다 붙이세요'
dtr, dva = df[df.fold != 0].reset_index(drop=True), df[df.fold == 0].reset_index(drop=True)
assert not (set(dtr.group) & set(dva.group)), '개체가 겹칩니다'
# 검증 표본: 병변 3,000 + 정상 3,000 고정 (seed 7) — 매 epoch 같은 사진
dva_s = pd.concat([dva[dva.label != 'A7'].sample(3000, random_state=7), dva[dva.label == 'A7'].sample(3000, random_state=7)]).reset_index(drop=True)
print(f'학습 {len(dtr):,} / 검증 {len(dva):,} (표본 {len(dva_s):,})')
# 재개: 이전 출력 ZIP 또는 캐글이 풀어 올린 폴더
resumed = None
for z in IN.rglob('dfine_stage1_resume*.zip'):
    with zipfile.ZipFile(z) as zf:
        for n in zf.namelist():
            assert not Path(n).is_absolute() and '..' not in Path(n).parts
        zf.extractall(CKPT); resumed = z; break
if resumed is None:
    for p in IN.rglob('dfine_last.pt'):
        shutil.copytree(p.parent, CKPT, dirs_exist_ok=True); resumed = p.parent; break
print('재개 입력:', resumed, '→', sorted(x.name for x in CKPT.iterdir()) if resumed else '⚠️ 재개 없음 — 처음부터')


In [ ]:
import torch, random, math, json, time
import numpy as np
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageEnhance
from transformers import AutoImageProcessor, DFineForObjectDetection
from src.detect import band_report
from sklearn.metrics import roc_auc_score

proc = AutoImageProcessor.from_pretrained(MODEL_ID)
proc.size = {'height': IMG, 'width': IMG}
MEAN = tuple(int(255*m) for m in proc.image_mean)

# ── 증강: D-FINE 공식 레시피(RandomPhotometricDistort p.5 · RandomZoomOut · RandomIoUCrop p.8 · hflip · 다중 스케일 ·
#    마지막 구간 강한 증강 끄기)를 우리 근거로 거른 것. 상세·근거는 STEP51 사전등록 표.
AUGS = {
    # 공식 레시피 정렬판: 밝기/대비/채도(색조 제외 — 붉은기가 병변 단서, STEP 30) · 줌 0.5~2.0(앱 병변 5.6~59.6% 를 덮음) · 좌우반전 · 다중 스케일
    'dfine':       dict(zoom=(0.5, 2.0), color=0.3, hue=0.0, vflip=0.0, rot90=0.0, jpeg=0.0, blur=0.0),
    # + 기하: 상하반전·90° 회전 — 피부 사진엔 위아래가 없다는 가설 (프로젝트 config 는 vflip 0, 검출에선 한 번 시험)
    'dfine+geo':   dict(zoom=(0.5, 2.0), color=0.3, hue=0.0, vflip=0.5, rot90=0.5, jpeg=0.0, blur=0.0),
    # + 화질: 약한 JPEG·블러 — 휴대폰 사진 (STEP 6 에서 1단계엔 화질 증강이 도움, 2단계엔 해로움 — 검출에선 미측정)
    'dfine+photo': dict(zoom=(0.5, 2.0), color=0.3, hue=0.0, vflip=0.0, rot90=0.0, jpeg=0.5, blur=0.3),
}
SCALES = [416, 448, 480, 512, 544, 576, 608]   # 다중 스케일 학습 (D-FINE collate base_size ± ) · 마지막 epoch 는 640 고정

def _clip_boxes(boxes, fn):
    '''네모를 fn 으로 옮기고 화면 밖은 자릅니다. 50% 미만 남으면 버립니다. 하나도 안 남으면 None.'''
    kept = []
    for b in boxes:
        x1, y1, x2, y2 = fn(b)
        cx1, cy1, cx2, cy2 = max(0, x1), max(0, y1), min(1, x2), min(1, y2)
        if cx2 > cx1 and cy2 > cy1 and (cx2-cx1)*(cy2-cy1) >= 0.5*(x2-x1)*(y2-y1):
            kept.append([cx1, cy1, cx2, cy2])
    return kept or None

def augment(im, boxes, cfg, rng):
    '''반환 (im, boxes). 네모가 다 잘려 나가면 원본을 돌려줍니다 (버리지 않음).'''
    import io as _io
    from PIL import ImageFilter
    W = im.size[0]
    orig = (im, [list(b) for b in boxes])
    boxes = [list(b) for b in boxes]
    if rng.random() < 0.5:
        im = im.transpose(Image.FLIP_LEFT_RIGHT); boxes = [[1-b[2], b[1], 1-b[0], b[3]] for b in boxes]
    if rng.random() < cfg['vflip']:
        im = im.transpose(Image.FLIP_TOP_BOTTOM); boxes = [[b[0], 1-b[3], b[2], 1-b[1]] for b in boxes]
    if rng.random() < cfg['rot90']:                       # 시계 반대 90° : (x, y) → (y, 1-x)
        im = im.transpose(Image.ROTATE_90); boxes = [[b[1], 1-b[2], b[3], 1-b[0]] for b in boxes]
    z = rng.uniform(*cfg['zoom'])
    if z < 1.0:                                           # 확대 = RandomIoUCrop 역할: 창 안을 잘라냄
        s = z; ox, oy = rng.uniform(0, 1-s), rng.uniform(0, 1-s)
        kept = _clip_boxes(boxes, lambda b: ((b[0]-ox)/s, (b[1]-oy)/s, (b[2]-ox)/s, (b[3]-oy)/s))
        if kept is None:
            return orig
        im = im.crop((int(ox*W), int(oy*W), int((ox+s)*W), int((oy+s)*W))).resize((W, W)); boxes = kept
    elif z > 1.0:                                         # 축소 = RandomZoomOut: 여백을 평균색으로
        canvas = Image.new('RGB', (int(W*z), int(W*z)), MEAN)
        ox, oy = rng.randint(0, canvas.size[0]-W), rng.randint(0, canvas.size[1]-W)
        canvas.paste(im, (ox, oy)); im = canvas.resize((W, W))
        boxes = [[(b[0]*W+ox)/(W*z), (b[1]*W+oy)/(W*z), (b[2]*W+ox)/(W*z), (b[3]*W+oy)/(W*z)] for b in boxes]
    if cfg['color'] and rng.random() < 0.5:               # RandomPhotometricDistort p=0.5 (색조는 제외)
        c = cfg['color']
        im = ImageEnhance.Brightness(im).enhance(rng.uniform(1-c, 1+c))
        im = ImageEnhance.Contrast(im).enhance(rng.uniform(1-c, 1+c))
        im = ImageEnhance.Color(im).enhance(rng.uniform(1-c, 1+c))
    if cfg['blur'] and rng.random() < cfg['blur']:
        im = im.filter(ImageFilter.GaussianBlur(rng.uniform(0.3, 1.0)))
    if cfg['jpeg'] and rng.random() < cfg['jpeg']:
        buf = _io.BytesIO(); im.save(buf, format='JPEG', quality=rng.randint(50, 95)); buf.seek(0)
        im = Image.open(buf).convert('RGB')
    return im, boxes

class DS(Dataset):
    def __init__(self, d, train, cfg=None, seed=0):
        self.d, self.train, self.cfg, self.seed = d.reset_index(drop=True), train, cfg, seed
    def __len__(self): return len(self.d)
    def __getitem__(self, i):
        r = self.d.iloc[i]
        with Image.open(index[r.image]) as im:
            im = im.convert('RGB')
        boxes = list(r.boxes)
        if self.train and self.cfg is not None and boxes:
            im, boxes = augment(im, boxes, self.cfg, random.Random(self.seed*1000003 + i))
        elif self.train:
            rr = random.Random(self.seed*1000003 + i)
            if rr.random() < 0.5:
                im = im.transpose(Image.FLIP_LEFT_RIGHT); boxes = [[1-b[2], b[1], 1-b[0], b[3]] for b in boxes]
            if self.cfg is not None and rr.random() < 0.5:                 # 정상 사진: 색·줌만 (네모가 없어 잘라낼 제약이 없음)
                im, _ = augment(im, [[0.45, 0.45, 0.55, 0.55]], self.cfg, rr)
        px = proc(images=im, return_tensors='pt')['pixel_values'][0]
        if boxes:
            b = torch.tensor(boxes, dtype=torch.float32)
            cxcywh = torch.stack([(b[:,0]+b[:,2])/2, (b[:,1]+b[:,3])/2, b[:,2]-b[:,0], b[:,3]-b[:,1]], 1)
            largest = max(boxes, key=lambda x: (x[2]-x[0])*(x[3]-x[1]))
        else:
            cxcywh, largest = torch.zeros(0, 4), [-1.0, -1.0, -1.0, -1.0]
        return px, {'class_labels': torch.zeros(len(boxes), dtype=torch.long), 'boxes': cxcywh}, torch.tensor(largest), int(r.label != 'A7')

def collate(batch):
    return torch.stack([b[0] for b in batch]), [b[1] for b in batch], torch.stack([b[2] for b in batch]), torch.tensor([b[3] for b in batch])

def multiscale(px, rng):
    size = rng.choice(SCALES)
    return px if size == IMG else torch.nn.functional.interpolate(px, size=(size, size), mode='bilinear', align_corners=False)

dev = 'cuda'
def make_model():
    return DFineForObjectDetection.from_pretrained(MODEL_ID, num_labels=1, ignore_mismatched_sizes=True).to(dev)

def make_opt(model):
    bb = [p for n, p in model.named_parameters() if 'backbone' in n and p.requires_grad]
    rest = [p for n, p in model.named_parameters() if 'backbone' not in n and p.requires_grad]
    return torch.optim.AdamW([{'params': bb, 'lr': LR*0.05}, {'params': rest, 'lr': LR}], weight_decay=1.25e-4)

@torch.no_grad()
def evaluate(model, d):
    '''1단계 후보로서: 최고 쿼리 점수 → AUROC · recall@헛알림 34.3% · 헛알림@recall 72.9% (STEP 52 관문 자리) + 병변 밴드.'''
    model.eval()
    dl = DataLoader(DS(d, False), batch_size=BATCH_SIZE*2, shuffle=False, num_workers=WORKERS, collate_fn=collate)
    S, Y, P, T = [], [], [], []
    for px, _, largest, y in dl:
        with torch.autocast('cuda', dtype=torch.float16):
            out = model(pixel_values=px.to(dev))
        S.extend(out.logits.float().sigmoid().amax(dim=(1, 2)).cpu().tolist()); Y.extend(y.tolist())
        res = proc.post_process_object_detection(out, threshold=0.0, target_sizes=[(1, 1)]*len(px))
        for r, t in zip(res, largest.numpy()):
            if t[0] < 0:
                continue
            k = int(r['scores'].argmax()) if len(r['scores']) else None
            P.append(r['boxes'][k].float().cpu().numpy() if k is not None else np.array([0.4, 0.4, 0.6, 0.6])); T.append(t)
    S, Y = np.asarray(S), np.asarray(Y)
    order = np.argsort(-S); s_sorted, y_sorted = S[order], Y[order]
    tp = np.cumsum(y_sorted); fp = np.cumsum(1-y_sorted)
    recall, fpr = tp/max(Y.sum(), 1), fp/max((1-Y).sum(), 1)
    rep = band_report(np.stack(P), np.stack(T))
    return {'auroc': float(roc_auc_score(Y, S)),
            'recall_at_fa343': float(recall[np.searchsorted(fpr, 0.343, side='right')-1]) if (fpr <= 0.343).any() else 0.0,
            'fa_at_recall729': float(fpr[np.searchsorted(recall, 0.729)]) if (recall >= 0.729).any() else 1.0,
            'both': rep['both'], 'in_pos': rep['in_pos'], 'off_median': rep['off_median'], 'ratio_median': rep['ratio_median']}

def export():
    z = Path('/kaggle/working/dfine_stage1_resume.zip'); tmp = z.with_suffix('.part')
    with zipfile.ZipFile(tmp, 'w', zipfile.ZIP_STORED) as zf:
        for p in sorted(CKPT.rglob('*')):
            if p.is_file():
                zf.write(p, p.relative_to(CKPT))
    tmp.replace(z); print('Resume bundle:', z, flush=True)

from torch.optim.swa_utils import AveragedModel
model = make_model(); opt = make_opt(model)
steps = EPOCHS * (len(dtr)//BATCH_SIZE)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 0.1 + 0.9*(1+math.cos(math.pi*min(s, steps)/steps))/2)
scaler = torch.amp.GradScaler('cuda')
ema_step = [0]
def ema_avg(avg, new, num_averaged):
    ema_step[0] += 1
    d = 0.9999 * (1 - math.exp(-ema_step[0] / 2000))
    return avg * d + new * (1 - d)
ema = AveragedModel(model, avg_fn=ema_avg, use_buffers=True)
last, history, start, best = CKPT / 'dfine_last.pt', [], 0, None
if last.exists():
    s = torch.load(last, map_location=dev, weights_only=False)
    model.load_state_dict(s['model']); opt.load_state_dict(s['opt']); sched.load_state_dict(s['sched']); scaler.load_state_dict(s['scaler'])
    ema.load_state_dict(s['ema']); ema_step[0] = s['ema_step']; history, start, best = s['history'], s['epoch'], s['best']
    assert s['epochs'] == EPOCHS and s['rows'] == len(dtr), '재개 설정이 다릅니다'
    print(f'재개: epoch {start} 부터', [(h['epoch'], round(h['auroc'], 4)) for h in history], flush=True)
else:
    print('⚠️ 재개 없음 — 처음부터', flush=True)

deadline = T0 + HOURS*3600 - 900          # ZIP 내보내기용 15분 예비
reason = 'completed'
try:
    for ep in range(start, EPOCHS):
        done = [h['minutes'] for h in history]
        per_epoch = max(done[-2:]) if done else 0
        if history and time.monotonic() + per_epoch*60*1.15 > deadline:
            reason = 'time_budget_before_epoch'; print('⏱ 예산 부족 — 다음 세션에서 재개', flush=True); break
        model.train(); tot = k = 0; t_ep = time.monotonic()
        last_epoch = ep == EPOCHS-1; rng = random.Random(1000+ep)
        dl = DataLoader(DS(dtr, True, None if last_epoch else AUGS[AUG], seed=ep), batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=WORKERS, pin_memory=True, drop_last=True, collate_fn=collate, persistent_workers=True)
        for px, labels, _, _ in dl:
            if time.monotonic() > deadline:
                reason = 'time_budget_during_epoch'; break
            if not last_epoch:
                px = multiscale(px, rng)
            labels = [{kk: v.to(dev) for kk, v in l.items()} for l in labels]
            with torch.autocast('cuda', dtype=torch.float16):
                loss = model(pixel_values=px.to(dev, non_blocking=True), labels=labels).loss
            if not torch.isfinite(loss):
                opt.zero_grad(set_to_none=True); continue            # fp16 오버플로 배치는 건너뜀
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
            scaler.step(opt); scaler.update(); sched.step(); ema.update_parameters(model)
            tot += float(loss.detach()); k += 1
            if k % 500 == 0:
                print(f'  ep{ep+1} {k}/{len(dl)} loss {tot/k:.3f} {(time.monotonic()-T0)/60:.1f}분', flush=True)
        if reason == 'time_budget_during_epoch':
            print('⏱ epoch 중간에 예산 소진 — 이 epoch 은 다음 세션에서 다시 돕니다', flush=True); break
        rep = evaluate(ema.module, dva_s)
        rec = {'epoch': ep+1, 'loss': tot/max(k, 1), **rep, 'minutes': (time.monotonic()-t_ep)/60}
        history.append(rec); print(json.dumps(rec), flush=True)
        if best is None or rep['auroc'] > best:                        # 1단계 후보 — best 는 AUROC
            best = rep['auroc']; ema.module.save_pretrained(CKPT / 'dfine_best'); proc.save_pretrained(CKPT / 'dfine_best')
        torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'sched': sched.state_dict(), 'scaler': scaler.state_dict(),
                    'ema': ema.state_dict(), 'ema_step': ema_step[0], 'history': history, 'epoch': ep+1, 'epochs': EPOCHS,
                    'rows': len(dtr), 'best': best}, last)
        (CKPT / 'dfine_history.json').write_text(json.dumps(history, indent=1))
finally:
    (CKPT / 'session.json').write_text(json.dumps({'stop_reason': reason, 'elapsed_min': (time.monotonic()-T0)/60, 'model': MODEL_ID,
                                                   'img': IMG, 'epochs': EPOCHS, 'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__}, indent=1))
    export()
print('끝:', reason, '· epoch 별', [(h['epoch'], round(h['auroc'], 4), round(h['fa_at_recall729'], 3)) for h in history])


In [ ]:
print((CKPT / 'dfine_history.json').read_text() if (CKPT / 'dfine_history.json').exists() else '(epoch 이 하나도 안 끝남)')
print('다운로드: /kaggle/working/dfine_stage1_resume.zip — 다음 세션 재개용이자 로컬 판정용 (dfine_best 폴더 포함)')
print('로컬 판정: uv run --extra train python tools/detect_coverage.py --detector-kind dfine --detector <dfine_best> --detector-stage1 --release <3팔>')
